# AI Hub 라벨 정제 기록

`data/label_fix/label_fixes.csv`의 aihub 줄을 어떻게 만들었는지 적은 작업 기록이다. AI Hub 데이터의 EDA는 [`../01_aihub_eda.ipynb`](../01_aihub_eda.ipynb)에 있고, 이 노트북은 그 2절(라벨 품질과 정제)에서 정제 도구와 근거 그림을 옮긴 것이다.

## 이 노트북이 하는 일

- 사진에서 알약을 OpenCV로 찾는 방법을 만들고 검증한다. (2-2-2, 2-2-3)
- 라벨 오류(누락, 좌표 오타, 알약을 감싸지 않는 box, 클래스 바뀜)를 규칙으로 고치고, 갤러리로 눈으로 확인한다. (2-1-3, 2-2-4~2-2-9, 2-3)
- 처리 결과를 정제 목록 한 줄씩 모으고(2-4), 적용해 정제 라벨을 만들고(2-5), 파일명과 대회 train 형식을 확인하고(2-6, 2-7), 1·2절의 검사를 다시 한다(2-8).
- 마지막에 메모리의 정제 목록이 레포의 `label_fixes.csv` aihub 줄과 같은지 확인한다. (2-9)

## 읽는 법

- 절 번호는 EDA 노트북 초판의 번호(2-1-3, 2-2-2, …)를 그대로 둔다. 그림 파일 이름, 캐시 이름, `tests/helpers.py`의 주석이 이 번호를 쓴다.
    - 2-1-1(라벨 누락 표), 2-1-2(알약 정보), 2-2-1(좌표 오류 표)은 EDA 노트북 2-1, 2-2에 있다. 뒤 절이 쓰는 변수를 만들기 위해 코드만 0절에 다시 둔다.
- 0절은 EDA 노트북의 0절, 1-1-4, 1-2, 1-3과 같은 코드다. 두 노트북은 서로 독립적으로 실행한다.
- 처음 실행은 OpenCV로 알약을 찾느라 오래 걸린다. 결과를 `runs/eda_raewoo/cache/`에 저장해서 다시 실행하면 빨라진다. 갤러리 시트는 `runs/eda_raewoo/figures_01/` 아래에 저장한다.
- 학습 데이터는 이 노트북이 아니라 `scripts/prepare_train_data.py`가 만든다. 정제 목록 CSV도 이 노트북이 쓰지 않고 읽어서 비교만 한다.

## 용어

| 용어 | 뜻 |
|---|---|
| 조합 | 함께 놓고 찍은 알약 묶음. 파일명의 첫 부분(`K-001900-016548-019607-029451`)으로, 알약의 K-코드를 이은 것이다. 조합마다 70°, 75°, 90° 사진이 있다 |
| K-코드 | 알약 코드(`K-001900`). 숫자 부분이 클래스 번호(category_id)다 |
| 촬영 PNG | 이미지 zip에 든 실제 사진 (`<조합>_0_2_0_2_<각도>_000_200.png`) |
| copy 파일 | 이름에 ` copy`, ` copy 2`가 붙은 AI Hub JSON (2-1-3) |
| `-오류` 폴더 | 이름이 `K-<코드>-오류`인 AI Hub 알약 폴더 (2-2-6) |
| IoU | 두 box가 겹친 넓이 ÷ 합친 넓이. 1이면 같은 box, 0이면 겹치지 않는다. 대회 지표는 IoU 0.75~0.95에서 계산한다 |
| box 맞음 | 라벨 box가 알약을 감싸는 정도. OpenCV로 잰 알약 범위에 여백을 더한 box와 라벨 box의 IoU다 (2-2-3) |
| 정제 목록 | 고칠 annotation과 고칠 값을 한 줄씩 적은 표 (2-4) |
| 정제 라벨 | 정제 목록을 적용한 AI Hub 라벨 (2-5) |


## 0. 준비

EDA 노트북 0절(설정, 데이터 읽기), 1-1-4(1차 정제), 1-2(키 구조), 1-3(Kaggle 형식 변환), 2-1-1(라벨 누락 표)과 같은 코드다. 설명은 EDA 노트북을 본다.

### 0-1. 라이브러리

In [ ]:
import collections
import concurrent.futures
import contextlib
import csv
import hashlib
import io
import json
import os
import sys
import unicodedata
from pathlib import Path

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize
from IPython.display import Markdown, display
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 금지 파일 검사, 파일명 해석, 정제 목록 읽기는 공용 모듈의 함수를 쓴다. 모두 파일을 읽기만 한다
from pill_detection.data import check_aihub_file, combo_name, pill_codes, read_label_fixes

# 실행 환경의 버전 확인 (팀원 간 환경 차이로 오류가 생겼을 때 원인 파악용)
print(f"Python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")
print(f"OpenCV       : {cv2.__version__}")

### 0-2. 경로

In [ ]:
# 노트북을 어디서 실행해도 레포 최상위(pyproject.toml이 있는 폴더)를 기준으로 경로를 잡는다
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TRAIN_IMG_DIR = RAW_DIR / 'train_images'
TRAIN_ANN_DIR = RAW_DIR / 'train_annotations'
TEST_IMG_DIR = RAW_DIR / 'test_images'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
AIHUB_DIR = EXTERNAL_DIR / 'aihub'  # scripts/prepare_train_data.py가 AI Hub를 받아 푸는 폴더
AIHUB_RAW_ANN_DIR = AIHUB_DIR / 'raw' / 'train_annotations'  # 라벨 zip을 그대로 푼 원본 JSON
AIHUB_IMG_DIR = AIHUB_DIR / 'raw' / 'train_images'  # 이미지 zip의 PNG
AIHUB_SOURCES_CSV = AIHUB_DIR / 'sources.csv'  # 파일마다 나온 zip (zip, path)
LABEL_FIXES_CSV = PROJECT_ROOT / 'data' / 'label_fix' / 'label_fixes.csv'  # 학습 데이터의 정제 목록. 대회 train 줄만 읽는다
EDA_DIR = PROJECT_ROOT / 'notebooks' / 'eda'
# 그림, 갤러리, 계산 결과(캐시)를 저장하는 곳. 이 노트북은 runs/ 밖에는 아무것도 쓰지 않는다 (runs/는 git에서 제외됨)
FIG_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'figures_01'
CACHE_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'cache'
FIG_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 대회와 AI Hub 이미지 크기 (너비, 높이)
IMG_W, IMG_H = 976, 1280

for folder in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR, AIHUB_RAW_ANN_DIR, AIHUB_IMG_DIR]:
    assert folder.is_dir(), (f'폴더가 없다: {folder.relative_to(PROJECT_ROOT)}\n'
                             '대회 데이터는 data/raw에 풀고, AI Hub는 uv run scripts/prepare_train_data.py로 받는다 (README의 학습 데이터 준비)')
print(f"레포 최상위  : {PROJECT_ROOT.name}")
print(f"그림 저장    : {FIG_DIR.relative_to(PROJECT_ROOT)}")
print(f"캐시 저장    : {CACHE_DIR.relative_to(PROJECT_ROOT)}")

### 0-3. 한글 글꼴

In [ ]:
# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트 사용 (macOS, Windows, Linux 순)
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ['AppleGothic', 'Malgun Gothic', 'NanumGothic']:
    if font_name in installed_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False  # 한글 폰트에서 마이너스 기호가 깨지는 문제 방지
print(f"그래프 폰트  : {plt.rcParams['font.family'][0]}")

### 0-4. seed

In [ ]:
# 랜덤 샘플 시각화 등의 재현성을 위한 seed 고정
SEED = 42
np.random.seed(SEED)
print(f'SEED : {SEED}')

### 0-5. 도우미 함수

In [ ]:
def save_fig(fig, name):
    """그림을 FIG_DIR에 png 파일로 저장한다."""
    fig.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')


def show_table(header, rows):
    """열 이름 리스트와 행 리스트를 마크다운 표로 출력한다."""
    lines = ['| ' + ' | '.join(header) + ' |', '|' + ' --- |' * len(header)]
    # 값 속 |(예: 분할선 각인 K|50)가 칸 구분자로 읽히지 않게 \|로 바꾼다
    lines += ['| ' + ' | '.join(str(value).replace('|', r'\|') for value in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


def box_iou(box_a, box_b):
    """두 xywh box의 IoU(겹친 넓이 ÷ 합친 넓이, 1이면 완전히 같음)를 계산한다."""
    inter_w = max(0, min(box_a[0] + box_a[2], box_b[0] + box_b[2]) - max(box_a[0], box_b[0]))
    inter_h = max(0, min(box_a[1] + box_a[3], box_b[1] + box_b[3]) - max(box_a[1], box_b[1]))
    inter = inter_w * inter_h
    return inter / (box_a[2] * box_a[3] + box_b[2] * box_b[3] - inter)


def crop_square(image, box, pad=0.1, size=224):
    """box 중심에서 긴 변보다 pad만큼 큰 정사각형을 잘라 size×size로 줄인다. 알약 모양이 늘어나지 않는다."""
    x, y, w, h = box
    side = max(w, h) * (1 + pad)
    cx, cy = x + w / 2, y + h / 2
    return image.crop((int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2))).resize((size, size))


def pill_label(ann):
    """box 위에 적을 K-코드와 각인(메타데이터 print_front/print_back)을 만든다."""
    marks = [ann['image'][key] for key in ('print_front', 'print_back') if ann['image'].get(key)]
    return f"K-{ann['category_id']:06d} ({'/'.join(marks) or '각인 없음'})"

def load_image_records(ann_dir):
    """annotation 폴더의 알약별 JSON을 이미지 단위로 묶는다.

    Args:
        ann_dir (Path): annotation 폴더. 하위 폴더의 JSON을 경로 순서로 모두 읽는다.

    Returns:
        dict: 이미지 파일명 → {'image': 그 이미지의 첫 JSON의 images[0], 'annotations': 알약마다 dict 하나}.
            annotation dict는 annotations[0]에 그 JSON의 images[0]('image')와 JSON 경로('json_path')를 더한 것이다.
    """
    records = {}
    for path in sorted(ann_dir.rglob('*.json')):
        data = json.loads(path.read_text(encoding='utf-8'))
        image = data['images'][0]
        record = records.setdefault(image['file_name'], {'image': image, 'annotations': []})
        for ann in data['annotations']:
            record['annotations'].append({**ann, 'image': image, 'json_path': str(path)})
    return records

### 0-6. 데이터 읽기

In [ ]:
train_records = load_image_records(TRAIN_ANN_DIR)
train_files = sorted(os.listdir(TRAIN_IMG_DIR))
test_files = sorted(os.listdir(TEST_IMG_DIR), key=lambda name: int(name.split('.')[0]))
train_classes = sorted({a['category_id'] for r in train_records.values() for a in r['annotations']})
train_combos = sorted({combo_name(fn) for fn in train_files})
assert set(train_records) == set(train_files), '라벨이 있는 이미지와 이미지 파일 목록이 다르다'

# 대회 train 정제: 학습 데이터의 정제 목록에서 대회 train 줄(source=kaggle)만 읽어 원본 라벨에 적용한다
IMAGE_FIELDS = ['back_color', 'camera_la', 'camera_lo', 'drug_S', 'drug_dir', 'file_name', 'height', 'id', 'imgfile',
                'light_color', 'size', 'width']  # 메타데이터 중 같은 이미지의 알약끼리 값이 같은 필드 (나머지는 알약 필드)
train_fixes = read_label_fixes(LABEL_FIXES_CSV, source='kaggle')  # add: 빠진 알약의 box를 넣음, modify: box를 고침
class_example = {}  # K-코드 → 대회 train에서 그 클래스의 annotation 하나 (그려 넣은 box의 메타데이터용)
for fn in train_files:
    for ann in train_records[fn]['annotations']:
        class_example.setdefault(ann['category_id'], ann)
clean_records = {}
for fn, record in sorted(train_records.items()):
    by_code = {a['category_id']: a for a in record['annotations']}
    anns = [dict(a) for a in record['annotations']]
    for fix in (row for row in train_fixes if row['file_name'] == fn):
        code, box = fix['category_id'], fix['bbox_after']
        if fix['action'] == 'add':  # 그려 넣은 box
            example = class_example[code]
            anns.append({**example, 'id': None, 'image_id': record['image']['id'], 'json_path': None, 'bbox': box,
                         'image': {**example['image'], **{key: record['image'][key] for key in IMAGE_FIELDS}}})
            continue
        # 원본 annotation 순서로 찾는다 (라벨을 서로 바꾸면 category_id가 바뀌므로 바뀐 값으로 찾지 않는다)
        ann = anns[next(i for i, a in enumerate(record['annotations']) if a['category_id'] == code)]
        ann['bbox'] = box
        if fix['category_id_after']:  # 라벨이 서로 바뀐 경우: 클래스와 알약 메타데이터를 상대 라벨의 것으로 바꾼다
            other = by_code[fix['category_id_after']]
            ann['category_id'], ann['image'] = other['category_id'], other['image']
    for ann in anns:
        ann['area'] = ann['bbox'][2] * ann['bbox'][3]
    clean_records[fn] = {**record, 'annotations': anns}
clean_object_count = collections.Counter(a['category_id'] for r in clean_records.values() for a in r['annotations'])
assert (len(clean_records), sum(clean_object_count.values()), len(clean_object_count)) == (232, 771, 56), \
    (f'대회 train 정제 결과가 이미지 232장, 객체 771개, 클래스 56개가 아니다.\n'
     f'{LABEL_FIXES_CSV.relative_to(PROJECT_ROOT)}의 kaggle 줄이 바뀌었는지 확인한다: git log -p -- data/label_fix/label_fixes.csv')

with open(EDA_DIR / '00_raewoo_test_review.csv', encoding='utf-8') as f:
    review_rows = list(csv.DictReader(f))

# AI Hub: zip은 풀고 지웠으므로, 파일마다 나온 zip은 sources.csv(zip, path)로 본다. path는 raw/ 기준이다
PREPARE_HINT = 'AI Hub 받는 명령: uv run scripts/prepare_train_data.py (README의 학습 데이터 준비)'
assert AIHUB_SOURCES_CSV.is_file(), f'{AIHUB_SOURCES_CSV.relative_to(PROJECT_ROOT)}이 없다.\n{PREPARE_HINT}'
with open(AIHUB_SOURCES_CSV, encoding='utf-8') as f:
    sources = [(unicodedata.normalize('NFC', row['zip']), unicodedata.normalize('NFC', row['path'])) for row in csv.DictReader(f)]
for zip_name in sorted({z for z, _ in sources}):
    check_aihub_file(zip_name)  # 금지 파일이면 여기서 오류가 난다
label_zips = sorted({z for z, path in sources if path.startswith('train_annotations/')})
assert len(label_zips) == 8, f'허용 조합 라벨 zip 8개를 모두 받아야 한다. 지금 {len(label_zips)}개\n{PREPARE_HINT}'
label_zip_of_combo = {}  # 조합 폴더 이름 → 라벨 zip
zip_json_paths = set()  # sources.csv에 적힌 JSON 경로 (train_annotations/ 기준)
for zip_name, path in sources:
    if path.startswith('train_annotations/') and path.endswith('.json'):
        json_rel = path.removeprefix('train_annotations/')
        combo_folder = json_rel.split('/')[0]
        assert label_zip_of_combo.setdefault(combo_folder, zip_name) == zip_name, f'{combo_folder} 폴더가 두 zip에 있다'
        zip_json_paths.add(json_rel)
raw_paths = sorted(AIHUB_RAW_ANN_DIR.rglob('*.json'))
assert {unicodedata.normalize('NFC', p.relative_to(AIHUB_RAW_ANN_DIR).as_posix()) for p in raw_paths} == zip_json_paths, \
    f'{AIHUB_RAW_ANN_DIR.relative_to(PROJECT_ROOT)}의 JSON이 sources.csv와 다르다. 받다가 멈췄으면 다시 실행한다.\n{PREPARE_HINT}'
aihub_jsons = []  # 원본 JSON 파일 하나가 한 줄. 아무것도 거르지 않고 읽은 그대로 둔다
for path in raw_paths:
    combo_folder, pill_folder = (unicodedata.normalize('NFC', p.name) for p in (path.parent.parent, path.parent))
    try:
        data, error = json.loads(path.read_text(encoding='utf-8')), None
    except json.JSONDecodeError as e:
        data, error = None, str(e)
    aihub_jsons.append({
        'path': path, 'zip': label_zip_of_combo[combo_folder], 'combo_folder': combo_folder, 'pill_folder': pill_folder,
        # JSON 쪽 이미지 파일명: JSON 안의 file_name이 아니라 JSON 파일 이름에서 ' copy'를 떼고 만든다
        'file_name': unicodedata.normalize('NFC', path.name)[:-len('.json')].split(' copy')[0] + '.png',
        'data': data, 'error': error,
    })
aihub_png = sorted(p.name for p in AIHUB_IMG_DIR.glob('*.png'))
aihub_index_files = [fn for fn in aihub_png if fn.endswith('_index.png')]  # 90° 사진에 K-코드와 제품명을 적은 참고용 사진 (1-1)
aihub_files = [fn for fn in aihub_png if not fn.endswith('_index.png')]  # 촬영 사진

print(f'대회 train {len(train_files)}장 (정제 라벨 객체 {sum(clean_object_count.values())}개), test {len(test_files)}장, '
      f'test 기록표 {len(review_rows)}줄')
print(f"AI Hub 라벨 zip {len(label_zips)}개: 원본 JSON {len(aihub_jsons):,}개 (읽지 못한 JSON {sum(j['error'] is not None for j in aihub_jsons)}개), "
      f'받은 PNG {len(aihub_png):,}장 (그중 _index.png {len(aihub_index_files)}장)')

### 0-7. 이미지 값 미리 계산 (캐시)

In [ ]:
THUMB_W, THUMB_H = 32, 40  # 축소 그림 크기 (원본 976×1280과 같은 비율)


def image_stats(path):
    """이미지 하나의 MD5, 크기, 채널 평균 밝기, 회색 축소 그림을 계산한다. 읽지 못하면 크기를 (0, 0)으로 둔다."""
    data = path.read_bytes()
    bgr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        return hashlib.md5(data).hexdigest(), (0, 0), np.zeros(3), np.zeros((THUMB_H, THUMB_W), np.uint8)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    thumb = cv2.resize(gray, (THUMB_W, THUMB_H), interpolation=cv2.INTER_AREA)
    return hashlib.md5(data).hexdigest(), (bgr.shape[1], bgr.shape[0]), bgr.reshape(-1, 3).mean(axis=0)[::-1], thumb


stat_paths = ([('train', TRAIN_IMG_DIR / fn) for fn in train_files] + [('test', TEST_IMG_DIR / fn) for fn in test_files]
              + [('aihub', AIHUB_IMG_DIR / fn) for fn in aihub_png])
stat_keys = np.array([f'{group}/{path.name}' for group, path in stat_paths])
cache_path = CACHE_DIR / 'image_stats.npz'
cache = np.load(cache_path) if cache_path.exists() else None
if cache is not None and np.array_equal(cache['keys'], stat_keys):
    stats = {name: cache[name] for name in cache.files}
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:  # OpenCV는 이미지를 읽는 동안 다른 스레드를 막지 않는다
        results = list(pool.map(image_stats, [path for _, path in stat_paths]))
    stats = {'keys': stat_keys, 'md5': np.array([r[0] for r in results]), 'size': np.array([r[1] for r in results]),
             'mean_rgb': np.array([r[2] for r in results]), 'thumb': np.array([r[3] for r in results])}
    np.savez(cache_path, **stats)
stat_index = {key: i for i, key in enumerate(stats['keys'])}  # 'aihub/<파일명>' → 행 번호
print(f"이미지 {len(stat_keys):,}장의 값을 계산했다. (대회 train {len(train_files)}, test {len(test_files)}, AI Hub {len(aihub_png):,})")

### 0-8. 1차 정제 (EDA 1-1-4)

In [ ]:
aihub_jsons_1st = [j for j in aihub_jsons if j['file_name'] in set(aihub_files)]  # 1차 정제 후 JSON: 촬영 PNG가 있는 JSON만 남긴다
aihub_photos_1st = aihub_files  # 1차 정제 후 PNG: 촬영 PNG만 남긴다 (_index.png를 뺀다)

CLEAN_ITEMS = {  # 항목 → (정의, 값을 적는 형식)
    'JSON 파일': ('라벨 zip에서 나온 파일. 알약 하나의 라벨', '{}'),
    'JSON 쪽 이미지 파일명': ('JSON 파일 이름에서 ` copy`를 떼고 .png로 바꾼 이름의 종류 수', '{}'),
    'PNG 파일 (촬영 PNG / _index.png)': ('이미지 zip에서 나온 파일', '{} ({} / {})'),
    'JSON과 촬영 PNG가 둘 다 있는 이미지 파일명': ('JSON 쪽과 PNG 쪽에 모두 있는 이름', '{}'),
    'JSON만 있는 이미지 파일명': ('JSON은 있는데 그 이름의 촬영 PNG가 없는 이름', '{} (JSON {}개)'),
    '촬영 PNG만 있는 이미지 파일명': ('촬영 PNG는 있는데 그 이름을 가리키는 JSON이 없는 이름', '{}'),
    '조합 수 (JSON 쪽 / PNG 쪽 / 둘 다)': ('이미지 파일명의 `<조합>` 부분의 종류 수', '{} / {} / {}'),
    '클래스 수': ('JSON이 든 K-코드 폴더 이름(앞 8자)의 종류 수. `-오류` 폴더도 앞 8자가 K-코드이다', '{}'),
}


def clean_counts(jsons, pngs):
    """JSON 목록과 PNG 파일 이름 목록으로 CLEAN_ITEMS의 값을 센다. 항목마다 형식의 {} 수만큼 숫자가 있다."""
    json_files = {j['file_name'] for j in jsons}
    photos = {fn for fn in pngs if not fn.endswith('_index.png')}
    json_only = json_files - photos
    json_combos, photo_combos = {combo_name(fn) for fn in json_files}, {combo_name(fn) for fn in photos}
    return {
        'JSON 파일': (len(jsons),),
        'JSON 쪽 이미지 파일명': (len(json_files),),
        'PNG 파일 (촬영 PNG / _index.png)': (len(pngs), len(photos), len(pngs) - len(photos)),
        'JSON과 촬영 PNG가 둘 다 있는 이미지 파일명': (len(json_files & photos),),
        'JSON만 있는 이미지 파일명': (len(json_only), sum(j['file_name'] in json_only for j in jsons)),
        '촬영 PNG만 있는 이미지 파일명': (len(photos - json_files),),
        '조합 수 (JSON 쪽 / PNG 쪽 / 둘 다)': (len(json_combos), len(photo_combos), len(json_combos & photo_combos)),
        '클래스 수': (len({j['pill_folder'][:8] for j in jsons}),),
    }


def signed(n):
    """변화량을 부호와 함께 적는다. 0은 부호 없이 적는다."""
    return f'{n:+,}' if n else '0'


before, after = clean_counts(aihub_jsons, aihub_png), clean_counts(aihub_jsons_1st, aihub_photos_1st)
class_before = collections.Counter(j['pill_folder'][:8] for j in aihub_jsons)
class_after = collections.Counter(j['pill_folder'][:8] for j in aihub_jsons_1st)
reduced = [f'{code} {class_before[code]:,}→{class_after[code]:,}' for code in sorted(class_before) if class_after[code] != class_before[code]]
rows = [[item, definition, form.format(*(f'{n:,}' for n in before[item])), form.format(*(f'{n:,}' for n in after[item])),
         form.format(*(signed(a - b) for a, b in zip(after[item], before[item])))]
        for item, (definition, form) in CLEAN_ITEMS.items()]
rows.append(['JSON이 줄어든 클래스', '1차 정제로 JSON 수가 바뀐 K-코드 (전→후)', '-', f'{len(reduced)}개: ' + ', '.join(reduced), '-'])
show_table(['항목', '정의', '1차 정제 전', '1차 정제 후', '변화 (후 - 전)'], rows)

### 0-9. 키 구조 비교 (EDA 1-2)

In [ ]:
def key_structure(data):
    """JSON의 최상위, images, annotations, categories에 있는 키 이름을 순서대로 모은다."""
    return tuple(tuple(part) for part in (data, data['images'][0], data['annotations'][0], data['categories'][0]))


def json_name(j):
    """표에 적을 JSON 경로(`<조합 폴더>/<K-코드 폴더>/<파일 이름>`)를 만든다."""
    return f"`{j['combo_folder']}/{j['pill_folder']}/{unicodedata.normalize('NFC', j['path'].name)}`"


def broken_line(path):
    """읽지 못한 JSON에서 오류가 난 줄 번호와 그 줄의 내용을 찾는다."""
    text = path.read_text(encoding='utf-8')
    try:
        json.loads(text)
    except json.JSONDecodeError as e:
        return e.lineno, text.splitlines()[e.lineno - 1].strip()


def key_difference(data):
    """JSON 키 구성에서 Kaggle과 다른 곳을 문장으로 적는다."""
    notes = []
    for part, keys, kaggle_keys in zip(['최상위', 'images', 'annotations', 'categories'], key_structure(data), kaggle_structure):
        missing, extra = [k for k in kaggle_keys if k not in keys], [k for k in keys if k not in kaggle_keys]
        if missing:
            notes.append(f"{part}에 {', '.join(missing)} 키가 없다")
        if extra:
            notes.append(f"{part}에 {', '.join(extra)} 키가 더 있다")
        if not missing and not extra and keys != kaggle_keys:
            notes.append(f'{part}의 키 순서가 다르다')
    return ', '.join(notes)


def top_values(counter):
    """가장 많은 값 두 개를 '값 (개수)' 문자열로 만든다."""
    return ', '.join(f'{v} ({n})' for v, n in counter.most_common(2))


kaggle_json_paths = sorted(TRAIN_ANN_DIR.rglob('*.json'))
kaggle_structures = collections.Counter(key_structure(json.loads(p.read_text(encoding='utf-8'))) for p in kaggle_json_paths)
kaggle_structure = kaggle_structures.most_common(1)[0][0]
kaggle_anns = [ann for r in train_records.values() for ann in r['annotations']]

# 1차 정제 후 JSON(1-1)을 센다. JSON 내용으로 세는 항목은 읽을 수 있는 JSON만 센다
broken = [j for j in aihub_jsons_1st if j['error']]
valid = [j for j in aihub_jsons_1st if not j['error']]
images = [j['data']['images'][0] for j in valid]
outside_error = [j for j in valid if not j['pill_folder'].endswith('-오류')]
key_mismatch = [j for j in valid if key_structure(j['data']) != kaggle_structure]
aihub_ids = {
    'image': collections.Counter(image['id'] for image in images),
    'category': collections.Counter(a.get('category_id') for j in valid for a in j['data']['annotations']),
    'annotation': collections.Counter(a.get('id') for j in valid for a in j['data']['annotations']),
    'category_name': collections.Counter(j['data']['categories'][0]['name'] for j in valid),
}
dl_idx_other = collections.defaultdict(list)  # drug_N → dl_idx가 K-코드 - 1이 아닌 JSON의 images
for image in images:
    if int(image['dl_idx']) != int(image['drug_N'][2:]) - 1:
        dl_idx_other[image['drug_N']].append(image)
error_folder_jsons = [j for j in aihub_jsons_1st if j['pill_folder'].endswith('-오류')]
error_folders = sorted({f"{j['combo_folder']}/{j['pill_folder']}" for j in error_folder_jsons})
copy_jsons = [j for j in aihub_jsons_1st if ' copy' in j['path'].name]
copy_kinds = collections.defaultdict(list)  # 파일 이름에 붙은 ' copy', ' copy 2' → 그 JSON
for j in copy_jsons:
    name = unicodedata.normalize('NFC', j['path'].name)
    copy_kinds[name[name.index(' copy'):-len('.json')]].append(j)

broken_note = '; '.join(f'{json_name(j)}의 {line_no}번째 줄 `{line}`에 값이 없어 JSON으로 읽을 수 없다'
                        for j in broken for line_no, line in [broken_line(j['path'])])
dl_idx_note = f'나머지 {sum(len(v) for v in dl_idx_other.values())}개는 ' + '; '.join(
    f"{code}({imgs[0]['dl_name']})의 JSON {len(imgs)}개이다. dl_idx가 모두 {'/'.join(sorted({str(i['dl_idx']) for i in imgs}))}이고, "
    f"dl_mapping_code는 {'/'.join(sorted({i['dl_mapping_code'] for i in imgs}))}이다" for code, imgs in sorted(dl_idx_other.items()))
rows = [
    ['읽을 수 없는 JSON', '0', len(broken),
     f'{broken_note}. JSON 내용으로 세는 아래 행(폴더 이름 = drug_N, 키 구성, id, name, dl_idx)은 이 {len(broken)}개를 뺀 {len(valid)}개로 센다'],
    ['폴더 구조', '`<조합>_json/K-<코드>/<이미지명>.json`', '같음',
     f'예외 2가지: K-코드 뒤에 `-오류`가 붙은 폴더 {len(error_folders)}개 (`-오류` 폴더 JSON 행), '
     f'파일 이름에 ` copy`가 붙은 JSON {len(copy_jsons)}개 (copy 중복 파일 행). `-오류` 폴더 외의 JSON은 K-코드 폴더 이름 = drug_N이다 '
     f"({sum(j['pill_folder'] == j['data']['images'][0]['drug_N'] for j in outside_error)}/{len(outside_error)})"],
    ['JSON 키 구성 (최상위, images, annotations, categories)',
     f"한 가지 {kaggle_structures[kaggle_structure]}/{len(kaggle_json_paths)} (키 {', '.join(str(len(p)) for p in kaggle_structure)}개)",
     f'Kaggle과 같음 {len(valid) - len(key_mismatch)}/{len(valid)}',
     '; '.join(f"다른 {len(key_mismatch)}개: {json_name(j)}의 {key_difference(j['data'])}" for j in key_mismatch)],
    ['JSON 파일 수', len(kaggle_json_paths), len(aihub_jsons_1st), f'1-1의 1차 정제 후 수이다. 읽을 수 없는 JSON {len(broken)}개도 센다'],
    ['image id', f'이미지마다 다름 ({len({r["image"]["id"] for r in train_records.values()})}종)', top_values(aihub_ids['image']), ''],
    ['category_id', f'K-코드 숫자 ({len(train_classes)}종)', top_values(aihub_ids['category']), ''],
    ['annotation id', f'annotation마다 다름 ({len({a["id"] for a in kaggle_anns})}종)', top_values(aihub_ids['annotation']), ''],
    ['categories의 name', '제품명 (dl_name)', top_values(aihub_ids['category_name']), ''],
    ['dl_idx = K-코드', f"{sum(int(a['image']['dl_idx']) == a['category_id'] for a in kaggle_anns)}/{len(kaggle_anns)}",
     f"{sum(int(i['dl_idx']) == int(i['drug_N'][2:]) for i in images)}/{len(valid)}", ''],
    ['dl_idx = K-코드 - 1', '0', f"{len(valid) - sum(len(v) for v in dl_idx_other.values())}/{len(valid)}", dl_idx_note],
    ['-오류 폴더 JSON', '0', len(error_folder_jsons),
     f"폴더 {len(error_folders)}개: {', '.join(f'`{folder}`' for folder in error_folders)}. "
     f"읽을 수 없는 JSON {sum(j['error'] is not None for j in error_folder_jsons)}개도 이 폴더에 있다"],
    ['copy.json, copy 2.json파일', '0', len(copy_jsons),
     ', '.join(f'`<이미지명>{kind}.json` {len(js)}개' for kind, js in sorted(copy_kinds.items()))
     + '. 예: ' + ', '.join(json_name(js[0]) for kind, js in sorted(copy_kinds.items()))],
]
show_table(['항목', 'Kaggle train', 'AI Hub 1차 정제 후', '비고'], rows)

### 0-10. Kaggle 형식으로 바꾸기 (EDA 1-3)

In [ ]:
KAGGLE_ANN_KEYS = kaggle_structure[2]  # Kaggle annotations의 키 순서 (1-2)
# images.id는 이미지 파일명 순서로 1501부터, annotations.id는 같은 순서(이미지 안에서는 JSON 경로 순서)로 5692부터 준다
image_ids = {fn: image_id for image_id, fn in enumerate(sorted({j['file_name'] for j in valid}), start=1501)}
annotation_ids = {j['path']: ann_id for ann_id, j in enumerate(sorted(valid, key=lambda j: (j['file_name'], j['path'])), start=5692)}


def to_kaggle_keys(j):
    """JSON 하나에서 Kaggle과 형식이 다른 키 8개만 바꾼 새 내용을 만든다. 나머지 키는 원본 값 그대로 둔다."""
    image, ann, category = j['data']['images'][0], j['data']['annotations'][0], j['data']['categories'][0]
    code = int(image['drug_N'][2:])  # K-코드 숫자 (K-001900 → 1900)
    new_image = {**image, 'dl_idx': str(code), 'id': image_ids[j['file_name']]}
    new_ann = {**ann, 'iscrowd': 0, 'category_id': code, 'id': annotation_ids[j['path']], 'image_id': new_image['id']}
    new_ann = {**{key: new_ann[key] for key in KAGGLE_ANN_KEYS if key in new_ann}, **new_ann}  # iscrowd가 없던 JSON도 Kaggle 키 순서로 맞춘다
    new_category = {**category, 'id': code, 'name': image['dl_name']}
    return {**j['data'], 'images': [new_image], 'annotations': [new_ann], 'categories': [new_category]}


# 1차 정제 후 JSON 하나가 한 줄 (aihub_jsons_1st와 같은 구조). 읽을 수 없는 JSON은 바꿀 수 없어 data가 None인 채로 둔다
aihub_kaggle_jsons = [{**j, 'data': None if j['data'] is None else to_kaggle_keys(j)} for j in aihub_jsons_1st]
aihub_kaggle = {}  # 이미지 파일명 → 대회 train_records(load_image_records)와 같은 구조. 1-4부터 이미지 단위로 볼 때 쓴다
for j in aihub_kaggle_jsons:
    if j['data'] is not None:
        image = j['data']['images'][0]
        record = aihub_kaggle.setdefault(j['file_name'], {'image': image, 'annotations': []})
        record['annotations'].append({**j['data']['annotations'][0], 'image': image, 'json_path': str(j['path'])})

TARGET_KEYS = [('images', 'id'), ('images', 'dl_idx'), ('annotations', 'id'), ('annotations', 'image_id'),
               ('annotations', 'category_id'), ('annotations', 'iscrowd'), ('categories', 'id'), ('categories', 'name')]


def doc_field(d, part, key):
    """JSON 내용 d의 part(images, annotations, categories) 첫 항목에서 key 값을 꺼낸다. 키가 없으면 '(키 없음)'이다."""
    return d[part][0].get(key, '(키 없음)')


def other_values(d):
    """키 8개를 뺀 나머지 값을 모은다. 바꾸기 전과 바꾼 뒤가 같은지 볼 때 쓴다."""
    values = {('최상위', 'type'): d['type']}
    for part in ['images', 'annotations', 'categories']:
        values.update({(part, key): v for key, v in d[part][0].items() if (part, key) not in TARGET_KEYS})
    return values


def format_checks(docs):
    """Kaggle, 바꾸기 전, 바꾼 뒤에 똑같이 적용하는 검사. 항목 → 표에 적을 값."""
    n = len(docs)
    files = {doc_field(d, 'images', 'file_name') for d in docs}
    ids, ann_ids = [doc_field(d, 'images', 'id') for d in docs], [doc_field(d, 'annotations', 'id') for d in docs]
    codes = [int(doc_field(d, 'images', 'drug_N')[2:]) for d in docs]
    category_ids = [doc_field(d, 'annotations', 'category_id') for d in docs]
    iscrowd = collections.Counter(doc_field(d, 'annotations', 'iscrowd') for d in docs)
    return {
        'JSON 파일 (이미지)': f'{n}개 ({len(files)}장)',
        'images.id': f'{min(ids)}~{max(ids)} ({len(set(ids))}종 / 이미지 {len(files)}장)',
        'images.dl_idx': f"= K-코드 {sum(int(doc_field(d, 'images', 'dl_idx')) == c for d, c in zip(docs, codes))}/{n}",
        'annotations.id': f'{min(ann_ids)}~{max(ann_ids)} ({len(set(ann_ids))}종 / annotation {n}개)',
        'annotations.image_id': f"= images.id {sum(doc_field(d, 'annotations', 'image_id') == doc_field(d, 'images', 'id') for d in docs)}/{n}",
        'annotations.category_id': f'= drug_N 숫자 {sum(a == c for a, c in zip(category_ids, codes))}/{n} ({len(set(category_ids))}종)',
        'annotations.iscrowd': ', '.join(f'{v} ({count})' for v, count in iscrowd.most_common()),
        'categories.id': f"= category_id {sum(doc_field(d, 'categories', 'id') == a for d, a in zip(docs, category_ids))}/{n}",
        'categories.name': f"= dl_name {sum(doc_field(d, 'categories', 'name') == doc_field(d, 'images', 'dl_name') for d in docs)}/{n}",
        'JSON 키 구성': f'Kaggle과 같음 {sum(key_structure(d) == kaggle_structure for d in docs)}/{n}',
    }


kaggle_docs = [json.loads(p.read_text(encoding='utf-8')) for p in kaggle_json_paths]
before_docs = [j['data'] for j in valid]  # 1-2의 읽을 수 있는 JSON
after_docs = [j['data'] for j in aihub_kaggle_jsons if j['data'] is not None]  # before_docs와 같은 순서
checks = [format_checks(docs) for docs in (kaggle_docs, before_docs, after_docs)]
rows = [[item] + [c[item] for c in checks] for item in checks[0]]
rows[0][3] += f'. 읽을 수 없는 JSON {len(broken)}개는 바꾸지 못했다'
rows.append(['그 밖의 키 (bbox, area 등)', '-', '-',
             f'원본과 같음 {sum(other_values(b) == other_values(a) for b, a in zip(before_docs, after_docs))}/{len(after_docs)}'])
show_table(['항목', 'Kaggle train', 'AI Hub 바꾸기 전', 'AI Hub 바꾼 뒤'], rows)

### 0-11. 1절 도우미 (EDA 1-4, 1-5, 1-6의 함수)

In [ ]:
def count_text(counter, total):
    """값별 이미지 수를 많은 순서로 '값 n장 (비율)'로 적는다. 셋이 넘으면 나머지는 합친다."""
    items = counter.most_common()
    text = [f'{v} {n:,}장 ({n / total:.1%})' for v, n in items[:3]]
    if len(items) > 3:
        text.append(f'그 외 {sum(n for _, n in items[3:]):,}장')
    return ', '.join(text)


jsons_of = collections.defaultdict(list)  # 이미지 파일명 → 그 이미지의 JSON (1-3에서 키를 바꾼 것)
for j in aihub_kaggle_jsons:
    jsons_of[j['file_name']].append(j)


def thumbs(prefix):
    """0번에서 계산한 회색 축소 그림을 묶음별로 꺼낸다. _index.png는 뺀다. (파일명 목록, float32 행렬)"""
    idx = [i for i, k in enumerate(stats['keys']) if k.startswith(prefix + '/') and not k.endswith('_index.png')]
    return [stats['keys'][i].split('/', 1)[1] for i in idx], stats['thumb'][idx].reshape(len(idx), -1).astype(np.float32)


def nearest(a, b, groups=None, chunk=2000):
    """a의 축소 그림마다 b에서 RMS 차이가 가장 작은 그림의 번호와 차이를 찾는다. groups를 주면(a와 b가 같을 때) 같은 묶음은 뺀다."""
    b_sq = (b ** 2).sum(axis=1)
    best_idx, best_rms = np.zeros(len(a), int), np.zeros(len(a))
    for start in range(0, len(a), chunk):
        block = a[start:start + chunk]
        d2 = (block ** 2).sum(axis=1)[:, None] + b_sq[None, :] - 2 * block @ b.T  # 제곱 거리 = |a|² + |b|² - 2a·b
        if groups is not None:
            d2[groups[start:start + chunk, None] == groups[None, :]] = np.inf
        best_idx[start:start + chunk] = d2.argmin(axis=1)
        best_rms[start:start + chunk] = np.sqrt(np.maximum(d2.min(axis=1), 0) / a.shape[1])
    return best_idx, best_rms


train_names, train_thumbs = thumbs('train')
test_names, test_thumbs = thumbs('test')
aihub_names, aihub_thumbs = thumbs('aihub')
near = {
    '대회 train': nearest(train_thumbs, aihub_thumbs),
    'test': nearest(test_thumbs, aihub_thumbs),
    'AI Hub 다른 조합 사진': nearest(aihub_thumbs, aihub_thumbs, groups=np.array([combo_name(fn) for fn in aihub_names])),
}
md5_of = collections.defaultdict(list)
for key, digest in zip(stats['keys'], stats['md5']):
    md5_of[digest].append(key)
same_md5 = {group: sum(any(k.startswith('aihub/') for k in keys) and any(k.startswith(group + '/') for k in keys)
                       for keys in md5_of.values()) for group in ['train', 'test']}
aihub_dup_md5 = sum(sum(k.startswith('aihub/') and not k.endswith('_index.png') for k in keys) > 1 for keys in md5_of.values())
missing_train = [fn for fn in train_files if len({a['category_id'] for a in train_records[fn]['annotations']}) < len(pill_codes(fn))]
aihub_photo_names = set(aihub_photos_1st)  # 1차 정제 후 촬영 PNG (1-1)
aihub_combos = {combo_name(fn) for fn in aihub_photo_names}
NEAR_RMS = 2  # 이 값보다 작으면 같은 사진 후보로 센다 (같은 사진에 글자만 적은 _index.png와 90° 사진이 0.6 정도, 1-1)


### 0-12. 라벨 누락 표 (EDA 2-1)

뒤 절이 쓰는 `missing_labels`, `surplus_labels`, `usable_bbox`, `bbox_problem`을 만든다.

In [ ]:
def usable_bbox(bbox):
    """학습에 쓸 수 있는 bbox인지 본다. 값이 4개이고 너비와 높이가 0보다 커야 한다."""
    return isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0


def bbox_problem(j):
    """JSON 하나의 bbox를 쓸 수 없는 원인을 돌려준다. 쓸 수 있으면 None이다. 원인이 둘 이상이면 앞의 것으로 센다."""
    if j['data'] is None:
        return '읽을 수 없는 JSON'
    if j['pill_folder'].endswith('-오류'):
        return '-오류 폴더'
    bbox = j['data']['annotations'][0]['bbox']
    if usable_bbox(bbox):
        return None
    if bbox == []:
        return 'bbox 빈 값'
    if not isinstance(bbox, list) or len(bbox) != 4:
        return '숫자가 붙은 bbox'
    return 'bbox 너비·높이 0 이하'


def json_code(j):
    """JSON의 K-코드 숫자. 읽을 수 없는 JSON은 알약 폴더 이름(`K-016551-오류`)에서 읽는다."""
    return int(j['pill_folder'][2:8]) if j['data'] is None else j['data']['annotations'][0]['category_id']


missing_labels = []  # (이미지, 빠진 K-코드, 원인, JSON 경로, bbox). JSON 없음이면 경로와 bbox가 None이다
extra_labels = []  # (이미지, 파일명에 없는 K-코드, JSON 경로)
surplus_labels = []  # (이미지, K-코드, 남는 box 수, 그 K-코드의 쓸 수 있는 JSON 경로)
for fn in aihub_photos_1st:
    named = collections.Counter(pill_codes(fn))
    jsons_by_code = collections.defaultdict(list)
    for j in jsons_of.get(fn, []):  # jsons_of: 1-5에서 만든 이미지 파일명 → 1-3의 JSON
        jsons_by_code[json_code(j)].append(j)
    for code in sorted(set(named) | set(jsons_by_code)):
        js = jsons_by_code[code]
        if code not in named:
            extra_labels += [(fn, code, j['path']) for j in js]
            continue
        usable = [j for j in js if bbox_problem(j) is None]
        unusable = [j for j in js if bbox_problem(j) is not None]
        for k in range(named[code] - len(usable)):  # 쓸 수 없는 JSON의 원인을 먼저 붙이고, JSON이 모자라면 JSON 없음이다
            if k < len(unusable):
                j = unusable[k]
                bbox = None if j['data'] is None else j['data']['annotations'][0]['bbox']
                missing_labels.append((fn, code, bbox_problem(j), j['path'], bbox))
            else:
                missing_labels.append((fn, code, 'JSON 없음', None, None))
        if len(usable) > named[code]:
            surplus_labels.append((fn, code, len(usable) - named[code], [j['path'] for j in usable]))

copy_pill_folders = {(j['combo_folder'], j['pill_folder']) for j in aihub_kaggle_jsons if ' copy' in j['path'].name}
MISSING_GROUPS = ['JSON 없음: 촬영 PNG만 있는 이미지 (1-1)', 'JSON 없음: copy 파일이 있는 알약 폴더 (1-2)', 'JSON 없음: 그 밖',
                  'bbox 빈 값', '숫자가 붙은 bbox', '-오류 폴더', '읽을 수 없는 JSON', 'bbox 너비·높이 0 이하']


def missing_group(m):
    """라벨 누락 하나가 들어갈 표의 행. JSON 없음은 이유에 따라 셋으로 나눈다."""
    fn, code, cause = m[:3]
    if cause != 'JSON 없음':
        return cause
    if not jsons_of.get(fn):
        return MISSING_GROUPS[0]
    if (f'{combo_name(fn)}_json', f'K-{code:06d}') in copy_pill_folders:
        return MISSING_GROUPS[1]
    return MISSING_GROUPS[2]


def missing_path(m):
    """라벨 누락의 JSON 경로. JSON 없음이면 있어야 할 경로이다."""
    fn, code, _, path, _ = m
    return path or AIHUB_RAW_ANN_DIR / f'{combo_name(fn)}_json' / f'K-{code:06d}' / f"{fn[:-len('.png')]}.json"


def repo_path(path):
    """표에 적을 레포 최상위 기준 경로."""
    return f"`{unicodedata.normalize('NFC', path.relative_to(PROJECT_ROOT).as_posix())}`"


def spread(items, n=3):
    """목록의 처음, 가운데, 끝에서 고르게 n개를 고른다."""
    return [items[i] for i in sorted(set(np.linspace(0, len(items) - 1, min(n, len(items))).round().astype(int)))]


rows = []
for group in MISSING_GROUPS:
    items = [m for m in missing_labels if missing_group(m) == group]
    if items:
        rows.append([group, len(items), len({m[0] for m in items}), len({m[1] for m in items}),
                     '<br>'.join(repo_path(missing_path(m)) for m in spread(items))])
missing_files = sorted({m[0] for m in missing_labels})
per_image = collections.Counter(collections.Counter(m[0] for m in missing_labels).values())
rows.append(['**라벨 누락 합계**', f'**{len(missing_labels)}**', f'**{len(missing_files)}장**', f"**{len({m[1] for m in missing_labels})}**", ''])
rows.append(['이미지당 빠진 알약 수', '', ', '.join(f'{k}개 {v}장' for k, v in sorted(per_image.items())), '', ''])
rows.append(['클래스 불일치: 파일명에 없는 K-코드', len(extra_labels), len({e[0] for e in extra_labels}), len({e[1] for e in extra_labels}),
             '<br>'.join(repo_path(e[2]) for e in spread(extra_labels))])
rows.append(['클래스 불일치: 한 이미지에 같은 K-코드의 box가 파일명보다 많음 (copy 파일이 있는 알약 폴더)', sum(s[2] for s in surplus_labels), len({s[0] for s in surplus_labels}),
             len({s[1] for s in surplus_labels}), '<br>'.join(repo_path(p) for s in spread(surplus_labels, n=2) for p in s[3])])
show_table(['원인', '객체 수', '이미지 수', '클래스 수', '확인할 파일 경로'], rows)

### 0-13. 각인 표시용 annotation과 라벨 누락 글자 (EDA 2-1 그림의 도우미)

In [ ]:
aihub_example = {}  # K-코드 → 바꾼 AI Hub 라벨에서 그 클래스의 annotation 하나 (각인 표시용)
for fn in sorted(aihub_kaggle):
    for ann in aihub_kaggle[fn]['annotations']:
        aihub_example.setdefault(ann['category_id'], ann)


def missing_text(fn):
    """그 이미지에서 빠진 알약의 K-코드와 각인을 적는다. 그림 제목이 옆 그림과 겹치지 않게 두 개씩 줄을 나눈다."""
    labels = [pill_label(aihub_example[m[1]]) if m[1] in aihub_example else f'K-{m[1]:06d}' for m in missing_labels if m[0] == fn]
    return '\n'.join(', '.join(labels[i:i + 2]) for i in range(0, len(labels), 2))


surplus_of = collections.defaultdict(dict)  # 이미지 → {box가 파일명보다 많은 K-코드: (box 수, 파일명에 적힌 수)}
for fn, code, extra, paths in surplus_labels:
    surplus_of[fn][code] = (len(paths), len(paths) - extra)


def json_part(ann):
    """box를 가져온 JSON 이름에서 각도와 copy 부분('75 copy')만 꺼낸다."""
    return Path(ann['json_path']).stem.split('_0_2_0_2_')[1].replace('_000_200', '')


## 2. 라벨 품질과 정제

1-3에서 형식을 바꾼 AI Hub 라벨(`aihub_kaggle`)의 오류를 찾고, 고칠 방법을 정해 정제 라벨을 만든다. 2-1-1, 2-1-2, 2-2-1은 EDA 노트북에 있다. 평가 지표 mAP@[0.75:0.95]는 IoU 0.75 미만인 box를 오답으로 보므로, 라벨 box가 조금만 틀려도 학습과 평가가 모두 흔들린다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 2-1 | 라벨 누락과 클래스 불일치 | 파일명의 알약마다 쓸 수 있는 box가 하나씩 있는지 본다 |
| 2-2 | 좌표 오류와 고치는 방법 | 틀린 좌표를 찾고, 사진에서 알약을 찾아 고친다 |
| 2-3 | 크기·색 대조와 crop 갤러리 | box 안의 알약이 그 클래스가 맞는지 본다 |
| 2-4 | 정제 목록 | 1·2절에서 찾은 오류의 처리를 annotation 한 줄씩 모은다 |
| 2-5 | 정제 목록 적용 | 정제 목록만으로 정제 라벨을 만든다. 3절부터 이 라벨로 본다 |
| 2-6 | 정제 뒤 파일명 | 클래스를 고쳐 파일명과 맞지 않게 된 이미지를 어떻게 할지 정한다 |
| 2-7 | 대회 train 형식으로 만들기 | 정제 라벨을 대회 train과 같은 JSON 구조로 만든다 (메모리) |
| 2-8 | 만든 라벨 다시 점검 | 1·2절의 검사를 다시 해서 문제가 남지 않았는지 본다 |
| 2-9 | 정제 목록 대조 | 메모리의 정제 목록이 레포의 `label_fixes.csv` aihub 줄과 같은지 확인한다 |

### 2-1. 라벨 누락과 클래스 불일치

2-1-1(라벨 누락 표)과 2-1-2(알약 정보)는 EDA 노트북 2-1에 있다. 여기서는 copy 파일을 다시 배정한다.

#### 2-1-3. copy 파일 다시 배정

- copy 파일이 있는 알약 폴더 21개에서 2-1-1 표의 두 행(JSON 없음 23개, 남는 box 23개)이 생긴다.
    - 폴더마다 JSON은 3개인데, 그중 둘 이상이 파일 이름과 JSON 안(`file_name`, `camera_la`)에 같은 각도를 적었다.
    - 1-3은 JSON 파일 이름의 각도대로 사진에 넣었다. 그래서 한 사진에 box가 2~3개 붙고, 다른 각도 사진에는 box가 없다.
- 세 JSON의 box를 70°, 75°, 90° 사진에 모두 그려 보고, 사진마다 맞는 JSON을 하나씩 정해 `COPY_ANGLE_FILES`에 적었다.
    - 70° 사진은 75°·90° 사진과 알약 배치가 다르다. 75°와 90° 사진은 배치가 같고 자리만 조금 다르다.

| 처리 | 뜻 |
|---|---|
| 그대로 | 1-3이 넣은 box가 맞는 JSON 하나뿐이다 |
| 남는 box 지움 | 맞는 JSON의 box와 다른 각도의 box가 함께 붙었다. 다른 각도 box를 지운다 |
| 바꿈 | 1-3이 넣은 box가 다른 각도의 box 하나다. 맞는 box로 바꾼다 |
| 채움 | box가 없다. 맞는 JSON의 box를 넣는다 |

- 옮긴 JSON의 이미지 필드(파일명, 각도, `images.id` 등)는 그 사진의 값으로 바꾸고, 알약 필드와 bbox, area, `annotations.id`는 그대로 둔다.
- 표는 행마다 폴더 하나다. 왼쪽 세 칸은 고치기 전 각 사진의 JSON, 오른쪽 세 칸은 고친 뒤의 JSON과 처리다. box가 없거나 2개 이상인 칸은 굵게 적었다.
- 2-2부터는 다시 배정한 라벨로 본다.

In [ ]:
# 폴더마다 JSON 3개의 box를 세 각도 사진에 모두 그려 보고 정한 배정이다 (아래 그림으로 확인)
COPY_ANGLE_FILES = {  # (조합, 알약 폴더) → 70°, 75°, 90° 사진에 맞는 JSON의 파일 이름 (각도와 copy 부분)
    ('K-001900-003544-004543-016548', 'K-016548'): ('70', '75 copy', '75'),
    ('K-001900-003544-004543-016551', 'K-016551'): ('70', '75 copy', '75'),
    ('K-001900-003544-006563-016551', 'K-001900'): ('70', '75 copy', '75'),
    ('K-001900-003544-016548-021026', 'K-021026'): ('75 copy 2', '75 copy', '75'),
    ('K-001900-003544-016548-027926', 'K-027926'): ('70', '75 copy 2', '75 copy'),
    ('K-001900-003544-016548-029345', 'K-016548'): ('75 copy', '75', '90'),
    ('K-001900-003544-016548-029345', 'K-029345'): ('70', '75 copy', '75'),
    ('K-001900-003544-016548-031705', 'K-001900'): ('70', '75 copy', '75'),
    ('K-001900-003544-016548-033009', 'K-033009'): ('70', '75 copy', '75'),
    ('K-001900-003544-016551-029451', 'K-001900'): ('75', '90 copy', '90'),
    ('K-001900-003544-016551-029451', 'K-029451'): ('70', '90 copy', '90'),
    ('K-001900-003544-016551-031705', 'K-031705'): ('70 copy 2', '70 copy', '70'),
    ('K-001900-004543-016551-019607', 'K-001900'): ('75 copy', '75', '90'),
    ('K-001900-004543-016551-033208', 'K-004543'): ('70', '75 copy', '75'),
    ('K-001900-006563-016548-018110', 'K-018110'): ('70', '75 copy', '75'),
    ('K-001900-006563-016548-019607', 'K-006563'): ('75 copy', '75', '90'),
    ('K-001900-006563-016548-019607', 'K-019607'): ('70', '90 copy', '90'),
    ('K-001900-006563-016548-024850', 'K-024850'): ('70', '90 copy', '90'),
    ('K-001900-010224-016548-029345', 'K-010224'): ('70', '90 copy', '90'),
    ('K-001900-010224-016548-029345', 'K-016548'): ('70', '90 copy', '90'),
    ('K-001900-010224-016548-033009', 'K-033009'): ('70', '75 copy', '75'),
}
ANGLES = ['70', '75', '90']


def raw_json_path(combo, folder, name):
    """'75 copy' 같은 파일 이름 부분으로 알약 폴더 안 원본 JSON의 경로를 만든다."""
    angle, _, copy = name.partition(' ')
    return AIHUB_RAW_ANN_DIR / f'{combo}_json' / folder / f"{combo}_0_2_0_2_{angle}_000_200{' ' + copy if copy else ''}.json"


def copy_action(before_names, name):
    """1-3이 이 사진에 넣은 JSON 이름 목록과 맞는 JSON 이름으로 처리를 정한다."""
    if before_names == [name]:
        return '그대로'
    if not before_names:
        return '채움'
    extra = f'남는 box {len(before_names) - 1}개 지움'
    if name in before_names:
        return extra
    return '바꿈' + (f', {extra}' if len(before_names) > 1 else '')


assert {(f'{combo}_json', folder) for combo, folder in COPY_ANGLE_FILES} == copy_pill_folders, \
    'copy 파일이 있는 알약 폴더(2-1의 copy_pill_folders)와 배정 표의 폴더가 다르다'
copy_fixes = []  # 사진마다 한 줄. 1-3이 넣은 box(before)와 다시 배정한 box(name, box), 처리
copy_json_of = {}  # (이미지, K-코드) → 다시 배정한 원본 JSON 경로 (2-2의 raw_annotation이 읽는다)
for n, ((combo, folder), names) in enumerate(COPY_ANGLE_FILES.items(), 1):
    name_of = {raw_json_path(combo, folder, name): name for name in names}  # 원본 JSON 경로 → 파일 이름 부분
    folder_jsons = {j['path']: j for j in aihub_kaggle_jsons if (j['combo_folder'], j['pill_folder']) == (f'{combo}_json', folder)}
    assert set(name_of) == set(folder_jsons), f'{combo} {folder}의 JSON이 배정 표와 다르다'
    code = int(folder[2:])
    for angle, name in zip(ANGLES, names):
        fn = f'{combo}_0_2_0_2_{angle}_000_200.png'
        record = aihub_kaggle[fn]
        # 1-3은 JSON 파일 이름의 각도대로 넣었다 (jsons_of: 1-5의 이미지 파일명 → 1-3의 JSON). 노트북을 다시 실행해도 같은 값이다
        before = [(name_of[j['path']], j['data']['annotations'][0]['bbox']) for j in jsons_of[fn] if j['path'] in name_of]
        right = folder_jsons[raw_json_path(combo, folder, name)]
        # 파일명, 촬영 각도, images.id 같은 이미지 필드는 이 사진의 값으로 바꾸고, annotation id는 1-3에서 이 JSON에 준 값을 그대로 쓴다
        image = {**right['data']['images'][0], **{key: record['image'][key] for key in IMAGE_FIELDS}}
        ann = {**right['data']['annotations'][0], 'image_id': record['image']['id'], 'image': image, 'json_path': str(right['path'])}
        kept = [a for a in record['annotations'] if Path(a['json_path']) not in name_of]  # 이 폴더의 JSON은 모두 빼고 맞는 하나만 넣는다
        record['annotations'] = sorted(kept + [ann], key=lambda a: a['category_id'])  # 1-3처럼 K-코드 폴더 순서로 둔다
        assert sum(a['category_id'] == code for a in record['annotations']) == pill_codes(fn).count(code), f'{fn} K-{code:06d}의 box 수가 파일명과 다르다'
        copy_json_of[(fn, code)] = right['path']
        copy_fixes.append({'n': n, 'combo': combo, 'folder': folder, 'angle': angle, 'file_name': fn, 'code': code,
                           'before': before, 'name': name, 'box': ann['bbox'],
                           'action': copy_action([b for b, _ in before], name)})
annotation_ids = [a['id'] for record in aihub_kaggle.values() for a in record['annotations']]
assert len(annotation_ids) == len(set(annotation_ids)), '다시 배정한 뒤 annotations.id가 겹친다'

# 2-1 표의 두 행이 모두 풀렸는지 확인하고, 2-2부터는 남은 라벨 문제만 센다
copy_filled = {(f['file_name'], f['code']) for f in copy_fixes if not f['before']}
copy_trimmed = {(f['file_name'], f['code']) for f in copy_fixes if len(f['before']) > 1}
assert {(m[0], m[1]) for m in missing_labels if missing_group(m) == MISSING_GROUPS[1]} <= copy_filled, \
    '2-1의 "JSON 없음: copy 파일이 있는 알약 폴더" 중 채우지 못한 알약이 있다'
assert {(s[0], s[1]) for s in surplus_labels} <= copy_trimmed, '2-1의 "box가 파일명보다 많음" 중 남는 box를 지우지 못한 이미지가 있다'
missing_labels = [m for m in missing_labels if (m[0], m[1]) not in copy_filled]
surplus_labels = [s for s in surplus_labels if (s[0], s[1]) not in copy_trimmed]


def before_text(before):
    """고치기 전 칸: 1-3이 이 사진에 넣은 그 K-코드의 JSON. box가 없거나 많으면 굵게 적는다."""
    names = ', '.join(f'`{name}`' for name, _ in before)
    if len(before) == 1:
        return names
    return f'**box {len(before)}개**: {names}' if before else '**box 없음**'


rows = []
for n, ((combo, folder), names) in enumerate(COPY_ANGLE_FILES.items(), 1):
    fixes = [f for f in copy_fixes if f['n'] == n]
    rows.append([n, f'`{combo}`<br>{folder}', *[before_text(f['before']) for f in fixes],
                 *[f"`{f['name']}` ({f['action']})" for f in fixes]])
by_angle = [[f for f in copy_fixes if f['angle'] == angle] for angle in ANGLES]
rows.append(['**합계**', '**사진에 box 없음**<br>(2-1의 JSON 없음: copy 파일이 있는 알약 폴더)',
             *[f"**{sum(not f['before'] for f in fixes)}개**" for fixes in by_angle], *['**0개**'] * 3])
rows.append(['', '**남는 box**<br>(2-1의 한 이미지에 같은 K-코드의 box가 파일명보다 많음)',
             *[f"**{sum(max(0, len(f['before']) - 1) for f in fixes)}개**" for fixes in by_angle], *['**0개**'] * 3])
action_kinds = [[('남는 box 지움' if f['action'].startswith('남는') else f['action']) for f in fixes] for fixes in by_angle]
rows.append(['', '**처리**', '', '', '',
             *['<br>'.join(f'{action} {count}장' for action, count in collections.Counter(kinds).most_common()) for kinds in action_kinds]])
show_table(['번호', '조합<br>알약 폴더', '고치기 전 70° 사진', '고치기 전 75° 사진', '고치기 전 90° 사진',
            '고친 뒤 70° 사진', '고친 뒤 75° 사진', '고친 뒤 90° 사진'], rows)

- 위 표의 고치기 전과 고친 뒤를 사진에 그려 나란히 놓는다. 행마다 폴더 하나다(번호 = 표의 번호).
    - 왼쪽(붉은 바탕) 세 칸: 고치기 전. 주황 box가 그 K-코드의 box다.
    - 오른쪽(초록 바탕) 세 칸: 고친 뒤. 연두 box가 다시 배정한 box다.
    - [ ] 안은 box를 가져온 JSON 이름이고, 흰 점선은 같은 사진의 다른 알약 box다.
- 고친 뒤 box 하나가 그 알약을 감싸는지 본다.

In [ ]:
def draw_copy_panel(ax, fn, other_boxes, boxes, color):
    """사진 전체에 다른 알약의 box(흰 점선)와 그 K-코드의 box(color, JSON 이름)를 그린다."""
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for code, (x, y, w, h) in other_boxes:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='white', linewidth=1.2, linestyle='--'))
        ax.text(x + 4, y + 4, f'K-{code:06d}', fontsize=6, color='black', va='top', clip_on=True,
                bbox=dict(facecolor='white', alpha=0.7, pad=0.8, linewidth=0))
    for k, (name, (x, y, w, h)) in enumerate(boxes):
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5))
        # box가 여럿이면 이름표가 겹치지 않게 위 왼쪽, 아래 왼쪽, 위 오른쪽 순서로 단다
        tx, ha = (x + w, 'right') if k == 2 else (x, 'left')
        ty, va = (y + h + 6, 'top') if k == 1 else (y - 6, 'bottom')
        ax.text(tx, ty, f'[{name}]', fontsize=8, ha=ha, va=va, clip_on=True, bbox=dict(facecolor=color, alpha=0.85, pad=1, linewidth=0))
    ax.set_xticks([])
    ax.set_yticks([])


n_rows = len(COPY_ANGLE_FILES)
fig = plt.figure(figsize=(20, 4.6 * n_rows + 1), layout='constrained')
before_fig, after_fig = fig.subfigures(1, 2, wspace=0.02)
before_axes, after_axes = before_fig.subplots(n_rows, 3), after_fig.subplots(n_rows, 3)
for fix in copy_fixes:
    row, col = fix['n'] - 1, ANGLES.index(fix['angle'])
    fn, code = fix['file_name'], fix['code']
    # 고치기 전: 1-3이 넣은 그대로 (jsons_of, 1-5). 다른 알약의 box도 1-3 그대로 그린다
    before_others = [(json_code(j), j['data']['annotations'][0]['bbox']) for j in jsons_of[fn]
                     if json_code(j) != code and bbox_problem(j) is None]
    ax = before_axes[row, col]
    draw_copy_panel(ax, fn, before_others, fix['before'], 'orange')
    count = f"box {len(fix['before'])}개" if fix['before'] else 'box 없음'
    ax.set_title(f"{fix['angle']}° 사진: {count}", fontsize=10, color='black' if len(fix['before']) == 1 else 'red')
    # 고친 뒤: 다시 배정한 aihub_kaggle (위 표)
    after_others = [(a['category_id'], a['bbox']) for a in aihub_kaggle[fn]['annotations']
                    if a['category_id'] != code and usable_bbox(a['bbox'])]
    ax = after_axes[row, col]
    draw_copy_panel(ax, fn, after_others, [(fix['name'], fix['box'])], 'lime')
    ax.set_title(f"{fix['angle']}° 사진: box 1개 ({fix['action']})", fontsize=10)
for n, ((combo, folder), _) in enumerate(COPY_ANGLE_FILES.items(), 1):
    for axes in (before_axes, after_axes):
        axes[n - 1, 0].set_ylabel(f'{n}. {folder}\n{combo}', fontsize=9)
before_fig.set_facecolor('#fdecea')
after_fig.set_facecolor('#e8f5e9')
before_fig.suptitle('고치기 전: 1-3의 라벨 (JSON 파일 이름의 각도대로 넣음)\n주황 = 그 K-코드의 box, [ ] = box를 가져온 JSON', fontsize=13)
after_fig.suptitle('고친 뒤: 맞는 각도에 다시 배정한 라벨\n연두 = 그 K-코드의 box, [ ] = box를 가져온 JSON', fontsize=13)
fig.suptitle('copy 파일이 있는 알약 폴더 21개의 box 고치기 전과 고친 뒤 (행 = 위 표의 폴더 번호, 흰 점선 = 다른 알약의 box)', fontsize=15)
save_fig(fig, '2-1_copy_reassign')
plt.show()

#### 2-1-3 결과

| 같은 각도를 가리킨 JSON | 폴더 수 |
|---|---|
| 75° | 14 |
| 90° | 6 |
| 70° | 1 |

- 4번, 12번 폴더는 JSON 3개가 모두 같은 각도를 가리킨다.
- 파일 이름의 각도와 맞는 사진의 각도가 다른 JSON이 있다. copy가 붙은 쪽이 늘 틀린 것도 아니다.
    - 예: 1번 폴더는 `75`가 90° 사진의 box이고, `75 copy`가 75° 사진의 box다.
- 고친 뒤에는 box가 없는 사진(고치기 전 23개)과 남는 box(고치기 전 23개)가 모두 0개다.

그림에서 보이는 것

- 고치기 전 box가 2~3개 붙은 사진에서는 box들이 같은 알약 위에서 조금씩 어긋나 있다. 어긋난 box는 다른 각도 사진의 알약 자리와 맞는다.
- 고친 뒤에는 21개 폴더 모두 세 각도 사진에서 box 하나가 그 알약을 감싼다.

파일 이름의 각도는 믿지 않고 `COPY_ANGLE_FILES`의 배정을 쓴다. 이 배정은 정제 목록의 2-1 줄(45줄)이 된다. (2-4-1)

### 2-2. 좌표 오류와 고치는 방법

| 단계 | 하는 일 |
|---|---|
| 2-2-1 | 좌표 오류를 찾는다. 2-1의 라벨 누락도 함께 센다 |
| 2-2-2 | OpenCV로 사진에서 알약을 찾는 방법을 만들고 검증한다 |
| 2-2-3 | 좌표는 멀쩡해도 알약을 감싸지 않는 box를 찾는다 |
| 2-2-4 | 규칙대로 고치고 처리 결과를 센다 |
| 2-2-5 | 문제 이미지를 모두 그려 눈으로 확인한다 (문제 갤러리) |
| 2-2-6 | `-오류` 폴더의 JSON이 왜 오류인지 본다 |
| 2-2-7 | 고친 뒤 2-1-1 표의 검사를 다시 한다 |
| 2-2-8 | 고친 뒤 2-2-1 표의 검사를 다시 한다 |
| 2-2-9 | 고친 뒤에도 남은 이미지를 본다 |

#### 2-2-1. 좌표 오류 찾기

| 문제 | 뜻 |
|---|---|
| 이미지 밖 | bbox가 이미지(976×1280) 밖으로 나간다 |
| area 불일치 | area가 bbox 너비×높이와 다르다. area 오타일 수도, bbox 오타일 수도 있다 |
| 다른 클래스와 box가 똑같음 | 같은 이미지에서 두 클래스의 box 좌표가 똑같다. 한 알약의 box를 다른 클래스에도 복사한 오류다 |

- 이 세 검사는 쓸 수 있는 bbox에만 한다. copy 파일을 다시 배정한 알약은 2-1-3에서 고친 값을 쓴다.
- 2-1의 라벨 누락도 같은 표에 센다. 고치는 방법(사진에서 찾아 그리기)이 같기 때문이다.
- 아래 그림은 문제마다 예시 이미지다.

In [ ]:
def raw_annotation(fn, code):
    """1-3의 aihub_kaggle에서 그 알약의 원본 (bbox, area)를 꺼낸다. annotation이 없으면 (None, None)이다.

    aihub_kaggle의 bbox와 area는 원본 JSON 값 그대로이고, copy 파일을 다시 배정한 알약(2-1)은 다시 배정한 JSON의 값이다.
    파일명에 같은 K-코드가 두 번 나오는 이미지가 없어서 (이미지, K-코드)마다 annotation은 하나이다.
    """
    anns = [a for a in aihub_kaggle.get(fn, {'annotations': []})['annotations'] if a['category_id'] == code]
    return (anns[0]['bbox'], anns[0]['area']) if anns else (None, None)


problems = collections.defaultdict(list)  # 이미지 → [(K-코드, 문제, 원본 bbox, 원본 area)]
for fn, record in sorted(aihub_kaggle.items()):
    anns = [a for a in record['annotations'] if usable_bbox(a['bbox'])]  # 쓸 수 없는 bbox는 아래에서 2-1의 라벨 누락으로 센다
    same_box = collections.Counter(tuple(a['bbox']) for a in anns)
    for ann in anns:
        x, y, w, h = ann['bbox']
        kinds = []
        if same_box[tuple(ann['bbox'])] > 1:
            kinds.append('다른 클래스와 box가 똑같음')
        if x < 0 or y < 0 or x + w > IMG_W or y + h > IMG_H:
            kinds.append('이미지 밖')
        if ann['area'] != w * h:
            kinds.append('area 불일치')
        if kinds:
            problems[fn].append((ann['category_id'], ' + '.join(kinds), ann['bbox'], ann['area']))
for m in missing_labels:  # 2-1의 라벨 누락 (copy 파일을 다시 배정해 채운 알약은 2-1-3에서 빠졌다)
    fn, code = m[:2]
    problems[fn].append((code, f'라벨 누락: {missing_group(m)}', *raw_annotation(fn, code)))

kind_count = collections.Counter(kind for items in problems.values() for _, kind, _, _ in items)
rows = []
for kind, n in kind_count.most_common():
    files = sorted(fn for fn, items in problems.items() if any(k == kind for _, k, _, _ in items))
    example = next(item for item in problems[files[0]] if item[1] == kind)
    rows.append([kind, n, len(files), f'`{files[0]}` K-{example[0]:06d} bbox {example[2]}, area {example[3]}'])
rows.append(['**합계**', f'**{sum(kind_count.values())}**', f'**{len(problems)}장**', ''])
show_table(['문제', 'annotation 수', '이미지 수', '예시 (이미지, 원본 bbox, 원본 area)'], rows)

In [ ]:
problem_examples = []  # 위 표의 행마다 (문제, 예시 이미지, K-코드, 원본 bbox, 원본 area). 표와 같이 파일명 순서로 첫 이미지이다
for kind, _ in kind_count.most_common():
    files = sorted(fn for fn, items in problems.items() if any(k == kind for _, k, _, _ in items))
    problem_examples.append((kind, files[0], *next((code, bbox, area) for code, k, bbox, area in problems[files[0]] if k == kind)))
n_cols = 4
n_rows = int(np.ceil(len(problem_examples) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 9.5 * n_rows), layout='constrained', squeeze=False)
for ax, (kind, fn, code, bbox, area) in zip(axes.flat, problem_examples):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    anns_of_box = collections.defaultdict(list)  # box → 그 box를 쓰는 annotation. box가 똑같은 클래스는 이름을 한 번에 적는다
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        if usable_bbox(ann['bbox']):  # 쓸 수 없는 bbox(라벨 누락)는 그릴 수 없다
            anns_of_box[tuple(ann['bbox'])].append(ann)
    for (x, y, w, h), anns in anns_of_box.items():
        target = any(a['category_id'] == code for a in anns)  # 예시 알약의 box는 빨간 점선으로 그린다 (이미지 밖으로 나간 부분은 잘려 보인다)
        color = 'red' if target else 'lime'
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5, linestyle='--' if target else '-'))
        ax.text(x, y - 8, '\n'.join(map(pill_label, anns)), fontsize=8, va='bottom', color='white' if target else 'black',
                bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    pill_text = pill_label(aihub_example[code]) if code in aihub_example else f'K-{code:06d}'
    detail = f'{pill_text}: 원본 bbox {bbox}, area {area}'
    if usable_bbox(bbox):  # 좌표 문제는 area, 이미지 크기와 견줄 값을 함께 적는다
        x, y, w, h = bbox
        detail += f'\n너비×높이 {w * h}, 오른쪽 끝(x+너비) {x + w}, 아래 끝(y+높이) {y + h}'
    ax.set_title(f'[{kind}]\n{fn}\n{detail}', fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    if not ax.images:
        ax.axis('off')
fig.suptitle(f'2-2 표의 문제별 예시 (표의 예시 이미지, 빨간 점선 = 예시 알약의 원본 bbox, 연두 = 같은 이미지의 다른 쓸 수 있는 bbox, '
             f'라벨 누락은 그릴 bbox가 없다, 이미지 {IMG_W}×{IMG_H})', fontsize=14)
save_fig(fig, '2-2_problem_examples')
plt.show()

#### 2-2-1 결과

| 문제 | annotation | 이미지 |
|---|---|---|
| area 불일치 | 427 | 414 |
| 다른 클래스와 box가 똑같음 | 134 | 70 |
| 이미지 밖 | 33 | 33 |
| 이미지 밖 + area 불일치 | 27 | 27 |
| 다른 클래스와 box가 똑같음 + area 불일치 | 6 | 6 |
| 라벨 누락 (2-1, copy 재배정 뒤) | 211 | 153 |
| **합계** | **838** | **691** |

그림에서 보이는 것

- area 불일치 예시의 K-002483은 bbox 너비가 210인데 area 10248은 21×488이다. box는 알약을 잘 감싸므로 area만 틀렸다.
- box가 똑같은 예시에서는 K-005002와 K-012778이 ELC 캡슐의 box 하나를 같이 쓰고, K-012778(빨간 YH 알약)에는 box가 없다.
- 이미지 밖 box는 숫자 오타로 보인다.
    - `[77, 158, 3131, 498]`은 너비 313에 1이 더 붙었다.
    - `[644, 851, 190, 498]`은 y 값이 틀려 box가 아래 알약 쪽에 있다.
- 숫자가 붙은 bbox `[121125, 333, 475]`는 `[121, 125, 333, 475]`에서 쉼표가 빠진 값이다.

좌표 오류는 대부분 숫자 하나가 빠지거나 더해지거나, 쉼표가 빠진 오타다. 오타를 고친 후보 중 사진의 알약과 맞는 값으로 고치고, 고칠 수 없으면 사진에서 알약을 찾아 다시 그린다. (2-2-4)

#### 2-2-2. 사진에서 알약 찾기

문제가 있는 box는 사진에서 알약을 직접 찾아 고친다. 이미지가 많아 OpenCV로 찾는다.

| 단계 | 방법 |
|---|---|
| 바닥 추정 | 라벨 box 자리를 뺀 픽셀을 크게 흐려, 위치마다 바닥의 밝기(R+G+B)와 파랑 비율 B/(R+G+B)를 구한다. 사진 위쪽이 더 밝기 때문이다 |
| 알약 픽셀 | 바닥보다 파랑 비율이 0.04 이상 낮거나, 20% 이상 밝거나, 15% 이상 어둡고 더 푸르지 않은 픽셀. 마지막 기준은 검은 캡슐을 잡고, 그림자는 더 푸르러서 빠진다 |
| 알약 덩어리 | 1/4로 줄인 사진에서 알약 픽셀이 붙은 덩어리를 찾는다. 작은 잡음과, 밝기로만 잡히고 사진 테두리와 이어진 덩어리(조명 밖 밝은 모서리)는 뺀다 |
| 알약 범위 | 덩어리 주변을 원본 해상도로 다시 보고, 알약 픽셀이 충분한 행과 열의 범위를 잰다 |
| box | 알약 범위의 네 변에 클래스별 여백(라벨 box 변과 알약 범위 사이 거리)의 중앙값을 더한다 |

- 클래스 기준값: 라벨에 문제가 없는 이미지 3,000장(seed 고정)에서 클래스마다 여백, 알약 범위의 긴 변·짧은 변, 가운데 색(Lab)의 중앙값을 구한다.
    - Lab은 사람 눈이 느끼는 색 차이에 가깝게 만든 색 공간이다. 찾은 알약이 어느 클래스인지 크기와 색으로 정할 때 쓴다.
- 검증: 기준값에 쓰지 않은 다른 이미지 1,000장으로 확인한다.
    - box 하나 지우기: 라벨 box 하나를 지우고 다시 찾아, 크기와 색이 그 클래스에 가장 가까운 덩어리로 만든 box와 라벨 box의 IoU를 본다.
    - box 둘 지우기: 라벨 box 둘을 지우고 다시 찾아, 크기와 색으로 두 클래스를 맞는 알약에 배정하는지 본다.
- 계산 결과는 `runs/eda_raewoo/cache/2-2_pill_finder.json`에 저장한다. 고른 이미지와 box가 같을 때만 저장한 값을 쓴다.
- 둘째 그림은 다시 찾기가 잘 안 되는 클래스(IoU 중앙값 0.85 미만)의 실패 사진이다.

In [ ]:
SCALE = 4  # 알약 덩어리를 찾을 때 사진을 줄이는 배율
BLUE_DROP = 0.04  # 바닥보다 파랑 비율이 이만큼 낮으면 알약 픽셀
BRIGHT_RISE = 0.2  # 바닥보다 이 비율 이상 밝으면 알약 픽셀 (흰 알약)
DARK_DROP, DARK_BLUE = 0.15, 0.01  # 바닥보다 15% 이상 어둡고 파랑 비율이 바닥+0.01보다 낮으면 알약 픽셀 (검은 캡슐)
MIN_SHARE = 0.1  # 알약 픽셀이 가장 많은 행(열)의 이 비율 이상인 행(열)만 알약 범위에 넣는다
REFINE_PAD = 24  # 덩어리 주변을 원본 해상도로 다시 볼 때 넓히는 폭 (픽셀)
MIN_COMPONENT = 300  # 줄인 사진에서 이보다 작은 덩어리(원본 약 4,800픽셀)는 알약으로 보지 않는다
N_REF, N_VAL = 3000, 1000  # 클래스 기준값과 검증에 쓰는 이미지 수


def read_rgb(path):
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)


def box_area_mask(shape, boxes, scale=1, pad=0):
    """xywh box 자리를 True로 칠한 마스크를 만든다."""
    mask = np.zeros(shape, bool)
    for x, y, w, h in boxes:
        mask[max(0, int(y) // scale - pad):int(y + h) // scale + pad, max(0, int(x) // scale - pad):int(x + w) // scale + pad] = True
    return mask


def background(rgb, known_boxes):
    """위치마다 바닥의 밝기(R+G+B)와 파랑 비율을 추정한다. 라벨 box 자리와 바닥과 크게 다른 픽셀은 빼고 흐리게 평균낸다."""
    small = cv2.resize(rgb, (IMG_W // SCALE, IMG_H // SCALE), interpolation=cv2.INTER_AREA).astype(np.float32)
    total = small.sum(axis=2) + 1
    blue = small[..., 2] / total
    known = box_area_mask(total.shape, known_boxes, SCALE, 4)
    floor = (~known) & (np.abs(blue - np.median(blue[~known])) < 0.03) & (np.abs(total / np.median(total[~known]) - 1) < 0.15)
    weight = np.maximum(cv2.GaussianBlur(floor.astype(np.float32), (0, 0), 25), 1e-3)
    bg_total = cv2.GaussianBlur(np.where(floor, total, 0).astype(np.float32), (0, 0), 25) / weight
    bg_blue = cv2.GaussianBlur(np.where(floor, blue, 0).astype(np.float32), (0, 0), 25) / weight
    return cv2.resize(bg_total, (IMG_W, IMG_H)), cv2.resize(bg_blue, (IMG_W, IMG_H))


def pill_pixels(rgb, bg_total, bg_blue, split=False):
    """바닥과 다른 픽셀(알약)을 True로 표시한다. split이면 (색으로 찾은 픽셀, 밝기로만 찾은 픽셀)을 따로 돌려준다."""
    rgb = rgb.astype(np.float32)
    total = rgb.sum(axis=2) + 1
    blue = rgb[..., 2] / total
    colored = (blue < bg_blue - BLUE_DROP) | ((total < bg_total * (1 - DARK_DROP)) & (blue < bg_blue + DARK_BLUE))
    bright = (total > bg_total * (1 + BRIGHT_RISE)) & ~colored
    return (colored, bright) if split else (colored | bright)


def corner_zone(bright):
    """밝기로만 찾은 픽셀 중 사진 테두리와 이어진 덩어리(조명 밖의 밝은 모서리)를 찾는다."""
    _, labels = cv2.connectedComponents(bright.astype(np.uint8), connectivity=8)
    edge = np.unique(np.concatenate([labels[0], labels[-1], labels[:, 0], labels[:, -1]]))
    return np.isin(labels, edge[edge > 0])


def refine_extent(rgb, box, bg_total, bg_blue, known_boxes, corner=None):
    """box 주변을 원본 해상도로 보고 가장 큰 알약 덩어리의 범위 [x0, y0, x1, y1]를 찾는다. 다른 라벨 box 자리와 밝은 모서리는 뺀다."""
    x, y, w, h = box
    x0, y0 = max(0, int(x) - REFINE_PAD), max(0, int(y) - REFINE_PAD)
    x1, y1 = min(IMG_W, int(x + w) + REFINE_PAD), min(IMG_H, int(y + h) + REFINE_PAD)
    if x1 - x0 < 8 or y1 - y0 < 8:
        return None  # box가 이미지 밖에 있다
    mask = pill_pixels(rgb[y0:y1, x0:x1], bg_total[y0:y1, x0:x1], bg_blue[y0:y1, x0:x1])
    if corner is not None:
        mask &= ~corner[y0:y1, x0:x1]
    for kx, ky, kw, kh in known_boxes:
        mask[max(0, int(ky) - y0):max(0, int(ky + kh) - y0), max(0, int(kx) - x0):max(0, int(kx + kw) - x0)] = False
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n, labels, comp_stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return None
    comp = labels == 1 + int(np.argmax(comp_stats[1:, 4]))
    rows = np.nonzero(comp.sum(axis=1) >= MIN_SHARE * comp.sum(axis=1).max())[0]
    cols = np.nonzero(comp.sum(axis=0) >= MIN_SHARE * comp.sum(axis=0).max())[0]
    return [x0 + int(cols[0]), y0 + int(rows[0]), x0 + int(cols[-1]) + 1, y0 + int(rows[-1]) + 1]


def find_pills(rgb, known_boxes):
    """라벨 box 밖에서 알약을 찾아 알약 범위 [x0, y0, x1, y1] 목록을 큰 덩어리부터 돌려준다."""
    bg_total, bg_blue = background(rgb, known_boxes)
    small = cv2.resize(rgb, (IMG_W // SCALE, IMG_H // SCALE), interpolation=cv2.INTER_AREA)
    size = small.shape[1::-1]
    colored, bright = pill_pixels(small, cv2.resize(bg_total, size), cv2.resize(bg_blue, size), split=True)
    corner = cv2.dilate(corner_zone(bright).astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    mask = (colored | bright) & ~corner & ~box_area_mask(colored.shape, known_boxes, SCALE, 4)
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    _, _, comp_stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    corner_full = cv2.resize(corner.astype(np.uint8), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST).astype(bool)
    extents = []
    for x, y, w, h, area in sorted(comp_stats[1:], key=lambda s: -s[4]):
        if area < MIN_COMPONENT:
            continue
        extent = refine_extent(rgb, [x * SCALE, y * SCALE, w * SCALE, h * SCALE], bg_total, bg_blue, known_boxes, corner_full)
        if extent:
            extents.append(extent)
    return extents


def extent_color(rgb, extent):
    """알약 범위 가운데 절반의 평균 색을 Lab으로 구한다."""
    x0, y0, x1, y1 = extent
    patch = rgb[y0 + (y1 - y0) // 4:y1 - (y1 - y0) // 4, x0 + (x1 - x0) // 4:x1 - (x1 - x0) // 4]
    return cv2.cvtColor(patch, cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(float).mean(axis=0)


def appearance_cost(extent, lab, ref):
    """알약 범위의 긴 변·짧은 변과 Lab 색이 클래스 기준값과 얼마나 다른지 잰다. 0이면 같다. (색 차이 30이 크기 e배 차이와 같은 무게)"""
    long_side, short_side = sorted([extent[2] - extent[0], extent[3] - extent[1]], reverse=True)
    return (abs(np.log(long_side / ref['long'])) + abs(np.log(short_side / ref['short']))
            + np.linalg.norm(np.asarray(lab) - np.asarray(ref['lab'])) / 30)


def with_margin(extent, margin):
    """알약 범위에 여백(왼, 위, 오른, 아래)을 더해 xywh box를 만든다. 이미지 밖으로 나가지 않게 자른다."""
    x0, y0, x1, y1 = extent
    x, y = max(0, round(x0 - margin[0])), max(0, round(y0 - margin[1]))
    return [int(x), int(y), int(min(IMG_W, round(x1 + margin[2])) - x), int(min(IMG_H, round(y1 + margin[3])) - y)]


finder_cache = CACHE_DIR / '2-2_pill_finder.json'
clean_pool = sorted(fn for fn in aihub_kaggle if fn not in problems and fn in set(aihub_files))  # 라벨에 문제가 없는 이미지
order = np.random.default_rng(SEED).permutation(len(clean_pool))
ref_files, val_files = [clean_pool[i] for i in order[:N_REF]], [clean_pool[i] for i in order[N_REF:N_REF + N_VAL]]
finder_key = [[fn, [a['bbox'] for a in aihub_kaggle[fn]['annotations']]] for fn in ref_files + val_files]  # 고른 이미지와 그 box
if finder_cache.exists() and json.loads(finder_cache.read_text()).get('key') == finder_key:
    finder = json.loads(finder_cache.read_text())
else:
    measures = collections.defaultdict(list)  # K-코드 → [여백 4개, 긴 변, 짧은 변, Lab 3개]
    for fn in ref_files:
        rgb = read_rgb(AIHUB_IMG_DIR / fn)
        boxes = [a['bbox'] for a in aihub_kaggle[fn]['annotations']]
        bg_total, bg_blue = background(rgb, boxes)
        for k, ann in enumerate(aihub_kaggle[fn]['annotations']):
            extent = refine_extent(rgb, ann['bbox'], bg_total, bg_blue, boxes[:k] + boxes[k + 1:])
            if extent:
                x, y, w, h = ann['bbox']
                measures[ann['category_id']].append([extent[0] - x, extent[1] - y, x + w - extent[2], y + h - extent[3],
                                                     *sorted([extent[2] - extent[0], extent[3] - extent[1]], reverse=True),
                                                     *extent_color(rgb, extent)])
    global_margin = np.median([m[:4] for ms in measures.values() for m in ms], axis=0)
    refs = {str(c): {'margin': (np.median(np.array(ms)[:, :4], axis=0) if len(ms) >= 3 else global_margin).tolist(),
                     'long': float(np.median(np.array(ms)[:, 4])), 'short': float(np.median(np.array(ms)[:, 5])),
                     'lab': np.median(np.array(ms)[:, 6:9], axis=0).tolist(), 'n': len(ms)} for c, ms in measures.items()}
    hide_one, hide_two = [], []  # (K-코드, IoU), (맞게 배정했는지)
    rng = np.random.default_rng(SEED)
    for fn in val_files:
        rgb = read_rgb(AIHUB_IMG_DIR / fn)
        anns = [a for a in aihub_kaggle[fn]['annotations']]
        boxes = [a['bbox'] for a in anns]
        for k, ann in enumerate(anns):
            ref = refs.get(str(ann['category_id']))
            extents = find_pills(rgb, boxes[:k] + boxes[k + 1:]) if ref else []
            if not ref:
                continue
            if not extents:
                hide_one.append((ann['category_id'], 0.0))
                continue
            best = min(extents, key=lambda e: appearance_cost(e, extent_color(rgb, e), ref))
            hide_one.append((ann['category_id'], box_iou(with_margin(best, ref['margin']), ann['bbox'])))
        pair = [k for k in rng.permutation(len(anns))[:2] if str(anns[k]['category_id']) in refs]
        if len(pair) == 2:
            extents = find_pills(rgb, [b for k, b in enumerate(boxes) if k not in pair])
            if len(extents) < 2:
                hide_two += [False, False]
                continue
            cost = np.array([[appearance_cost(e, extent_color(rgb, e), refs[str(anns[k]['category_id'])]) for e in extents] for k in pair])
            for i, j in zip(*scipy.optimize.linear_sum_assignment(cost)):
                ref = refs[str(anns[pair[i]]['category_id'])]
                hide_two.append(box_iou(with_margin(extents[j], ref['margin']), anns[pair[i]]['bbox']) > 0.5)
    finder = {'key': finder_key, 'refs': refs, 'global_margin': global_margin.tolist(),
              'hide_one': hide_one, 'hide_two': [bool(v) for v in hide_two]}
    finder_cache.write_text(json.dumps(finder))
class_refs = {int(c): ref for c, ref in finder['refs'].items()}
hide_one_iou = np.array([v for _, v in finder['hide_one']])

fig, axes = plt.subplots(1, 2, figsize=(20, 5.5), layout='constrained', gridspec_kw={'width_ratios': [1, 2]})
axes[0].hist(hide_one_iou, bins=np.linspace(0, 1, 51), color='tab:blue')
for t in [0.75, 0.95]:
    axes[0].axvline(t, color='gray', linestyle='--', linewidth=1)
axes[0].set_yscale('log')
axes[0].set_xlabel('다시 찾은 box와 라벨 box의 IoU')
axes[0].set_ylabel('box 수 (로그 눈금)')
axes[0].set_title(f'box 하나 지우고 다시 찾기 {len(hide_one_iou):,}개: 중앙값 {np.median(hide_one_iou):.3f}, '
                  f'0.75 미만 {(hide_one_iou < 0.75).mean():.1%}, 0.95 이상 {(hide_one_iou >= 0.95).mean():.1%}', fontsize=11)
by_class = collections.defaultdict(list)
for c, v in finder['hide_one']:
    by_class[c].append(v)
order = sorted(by_class, key=lambda c: np.median(by_class[c]))
axes[1].bar(np.arange(len(order)), [np.median(by_class[c]) for c in order], color='tab:blue')
axes[1].axhline(0.95, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(0.75, color='gray', linestyle='--', linewidth=1)
axes[1].set_xticks(np.arange(len(order)), [f'K-{c:06d}' for c in order], rotation=90, fontsize=6)
axes[1].set_ylim(0.5, 1)
axes[1].set_ylabel('IoU 중앙값')
axes[1].set_title(f'클래스별 IoU 중앙값 ({len(order)}개 클래스, 낮은 순서)', fontsize=11)
fig.suptitle(f'OpenCV로 빠진 box를 찾는 방법의 검증 (이미지 {N_VAL:,}장, 회색 점선 = IoU 0.75와 0.95), '
             f"box 둘 지우고 클래스 배정: {sum(finder['hide_two'])}/{len(finder['hide_two'])} 맞음", fontsize=13)
save_fig(fig, '2-2_finder_validation')
plt.show()

In [ ]:
refind_samples = []  # 위 셀의 box 하나 지우기 검증 box (이미지, annotation 순번, K-코드). hide_one과 같은 순서로 돈다
for fn in val_files:
    for k, ann in enumerate(aihub_kaggle[fn]['annotations']):
        if str(ann['category_id']) in finder['refs']:
            refind_samples.append((fn, k, ann['category_id']))
assert [s[2] for s in refind_samples] == [c for c, _ in finder['hide_one']], '검증 box 순서가 위 셀의 hide_one과 다르다'
refind_samples = [(*s, iou) for s, (_, iou) in zip(refind_samples, finder['hide_one'])]
# IoU 중앙값이 0.85 미만인 클래스(2-2에서 사진으로 찾지 않는 클래스)마다 0.75 미만 사진을 IoU 순서로 고르게 4장 고른다
hard_classes = sorted((c for c in by_class if np.median(by_class[c]) < 0.85), key=lambda c: np.median(by_class[c]))
panels = []
for c in hard_classes:
    fails = sorted((s for s in refind_samples if s[2] == c and s[3] < 0.75), key=lambda s: (s[3], s[0]))
    panels += [fails[i] for i in np.unique(np.linspace(0, len(fails) - 1, min(4, len(fails))).round().astype(int))]
n_rows = int(np.ceil(len(panels) / 4))
fig, axes = plt.subplots(n_rows, 4, figsize=(20, 7.8 * n_rows), layout='constrained', squeeze=False)
for ax, (fn, k, code, iou) in zip(axes.flat, panels):
    rgb = read_rgb(AIHUB_IMG_DIR / fn)
    boxes = [a['bbox'] for a in aihub_kaggle[fn]['annotations']]
    extents = find_pills(rgb, boxes[:k] + boxes[k + 1:])  # 위 셀처럼 k번째 box를 지우고 다시 찾는다
    ax.imshow(rgb)
    for j, (x, y, w, h) in enumerate(boxes):
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime' if j == k else 'white', linewidth=2.5 if j == k else 1))
    for x0, y0, x1, y1 in extents:
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='yellow', linewidth=1.2, linestyle=':'))
    if extents:
        best = min(extents, key=lambda e: appearance_cost(e, extent_color(rgb, e), class_refs[code]))
        found = with_margin(best, class_refs[code]['margin'])
        ax.add_patch(plt.Rectangle(found[:2], found[2], found[3], fill=False, edgecolor='orangered', linewidth=2.5, linestyle='--'))
    assert abs((box_iou(found, boxes[k]) if extents else 0.0) - iou) < 1e-9, f'{fn}: 다시 찾은 IoU가 위 셀과 다르다'
    v = np.array(by_class[code])
    ax.set_title(f"{pill_label(aihub_example[code])} {aihub_example[code]['image']['dl_name']}\n"
                 f'클래스 IoU 중앙값 {np.median(v):.3f}, 0.75 미만 {(v < 0.75).sum()}/{len(v)}개\n{fn}\n'
                 f'이 사진 IoU {iou:.3f}, 찾은 알약 덩어리 후보 {len(extents)}개', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes.flat[len(panels):]:
    ax.axis('off')
fig.suptitle(f'box 하나 지우고 다시 찾기의 IoU 중앙값이 0.85 미만인 클래스 {len(hard_classes)}개의 실패 사진 '
             '(한 줄에 한 클래스, IoU 0.75 미만 사진을 IoU 순서로 고르게 4장)\n'
             '연두 = 지운 라벨 box, 노랑 점선 = 사진에서 찾은 알약 덩어리 후보, 주황 점선 = 크기·색으로 고른 후보에 여백을 더한 box, '
             '흰 선 = 다른 라벨 box (찾을 때 빼는 자리)', fontsize=14)
save_fig(fig, '2-2_finder_failures')
plt.show()

#### 2-2-2 결과

| 검증 | 결과 |
|---|---|
| box 하나 지우기 (3,835개) | IoU 중앙값 0.958, 0.95 이상 60.6%, 0.75 미만 5.0% |
| box 둘 지우기 (1,966개) | 크기·색으로 맞는 알약에 배정 1,816개 (92.4%) |

- IoU 0.75 미만은 대부분 바닥과 색이 같은 파란 알약 4개 클래스에서 나온다.
    - K-010487 50/50개, K-012638 42/42개, K-031885 38/38개, K-041149 21/26개가 0.75 미만이다.
    - 이 151개를 빼면 0.75 미만은 약 1%다.
- 나머지 111개 클래스는 IoU 중앙값이 0.85 이상이다.

그림에서 보이는 것 (실패 사진)

- 파란 캡슐과 파란 알약은 알약 덩어리로 잡히지 않는다.
- 흰색과 파란색 반반인 K-041149는 흰 쪽만 잡힌다.

사진에서 찾은 box는 대부분의 클래스에서 평가 기준인 IoU 0.75를 넘는다. 빠진 box 채우기와 오타 후보 확인에 쓴다. 다만 위 4개 클래스에는 쓰지 않고, 크기·색 배정은 8% 정도 틀리므로 결과를 문제 갤러리(2-2-5)에서 눈으로 확인한다.

#### 2-2-3. 알약을 감싸지 않는 box

- 좌표 값은 멀쩡해도 box가 알약을 감싸지 않는 경우가 있다. (바닥이나 옆 알약을 감쌈, 한쪽으로 밀림) 2-2-1의 검사로는 찾을 수 없어서, 쓸 수 있는 모든 box를 사진과 대조한다.
- box 맞음: 라벨 box 주변에서 알약 범위를 재고, 클래스 여백을 더해 다시 만든 box와 라벨 box의 IoU다.
    - 1이면 라벨 box가 알약을 다른 box들과 같은 여백으로 감싼다. 0이면 box 안에 알약이 없다.
    - 0.75 미만인 box를 "알약을 감싸지 않는 box"로 보고 2-2-4에서 함께 고친다. 0.75는 평가 지표의 가장 낮은 IoU 기준이다.
- 알약 범위를 잘 재지 못하는 클래스는 이 검사에서 뺀다. 이 클래스는 2-3 crop 갤러리로 본다.
    - 2-2-2 검증에서 IoU 중앙값이 0.85 미만인 클래스
    - box 맞음 0.75 미만이 그 클래스 box의 30%를 넘는 클래스 (라벨이 그만큼 틀렸다기보다 재는 방법이 실패했을 가능성이 크다)
- 여기서 잰 알약 범위와 색은 2-3의 크기·색 대조에도 쓴다. 결과는 `runs/eda_raewoo/cache/2-2_box_measures.json`에 저장한다.

In [ ]:
FIT_THRESHOLD = 0.75


def measure_boxes(job):
    """이미지 하나의 box마다 (알약 범위, Lab 색)을 잰다. 알약 범위를 못 찾으면 None이다."""
    fn, boxes = job
    rgb = read_rgb(AIHUB_IMG_DIR / fn)
    bg_total, bg_blue = background(rgb, boxes)
    out = []
    for k, box in enumerate(boxes):
        extent = refine_extent(rgb, box, bg_total, bg_blue, boxes[:k] + boxes[k + 1:])
        out.append([fn, box, extent, extent_color(rgb, extent).tolist() if extent else None])
    return out


def measure_all(records):
    """쓸 수 있는 box마다 알약 범위와 색을 재서 (이미지, box) → (알약 범위, Lab 색)으로 돌려준다.
    같은 box 목록을 이미 잰 이미지는 캐시에서 읽는다."""
    cache_path = CACHE_DIR / '2-2_box_measures.json'
    cached = json.loads(cache_path.read_text()) if cache_path.exists() else {}  # '이미지 box 목록' → box마다 [알약 범위, Lab 색]
    jobs = {}  # 캐시 키 → (이미지, 쓸 수 있는 box 목록)
    for fn, r in sorted(records.items()):
        boxes = [a['bbox'] for a in r['annotations'] if usable_bbox(a['bbox'])]  # 쓸 수 없는 bbox는 2-2 표의 라벨 누락이다
        jobs[f'{fn} {json.dumps(boxes)}'] = (fn, boxes)
    todo = [key for key in jobs if key not in cached]
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        for key, rows in zip(todo, pool.map(measure_boxes, [jobs[key] for key in todo])):
            cached[key] = [[extent, lab] for _, _, extent, lab in rows]
    if todo:
        cache_path.write_text(json.dumps(cached))
    return {(fn, tuple(box)): tuple(cached[key][k]) for key, (fn, boxes) in jobs.items() for k, box in enumerate(boxes)}


box_measures = measure_all({fn: r for fn, r in aihub_kaggle.items() if fn in set(aihub_files)})
no_ref = {'margin': finder['global_margin']}
box_fit = {}  # (이미지, K-코드) → box 맞음 IoU
for fn, record in aihub_kaggle.items():
    if fn not in set(aihub_files):
        continue
    for ann in record['annotations']:
        if not usable_bbox(ann['bbox']):
            continue  # 쓸 수 없는 bbox는 잴 수 없다 (2-2 표의 라벨 누락)
        extent, _ = box_measures[(fn, tuple(ann['bbox']))]
        margin = class_refs.get(ann['category_id'], no_ref)['margin']
        box_fit[(fn, ann['category_id'])] = box_iou(with_margin(extent, margin), ann['bbox']) if extent else 0.0
# 위 검증에서 다시 찾기가 잘 안 되는 클래스(IoU 중앙값 0.85 미만, 예: 바닥과 색이 같은 파란 알약)는 box 맞음을 믿을 수 없어 뺀다
validation_by_class = collections.defaultdict(list)
for c, v in finder['hide_one']:
    validation_by_class[c].append(v)
low_share = collections.defaultdict(list)
for (fn, code), v in box_fit.items():
    low_share[code].append(v < FIT_THRESHOLD)
# 한 클래스의 30% 넘게 box 맞음이 낮으면 라벨 오류가 아니라 알약 범위를 못 재는 클래스로 본다
unreliable_classes = sorted({c for c, v in validation_by_class.items() if np.median(v) < 0.85}
                            | {c for c, v in low_share.items() if len(v) >= 10 and np.mean(v) > 0.3})
already = {(fn, code) for fn, items in problems.items() for code, *_ in items}
low_fit = sorted(key for key, v in box_fit.items() if v < FIT_THRESHOLD and key not in already and key[1] not in unreliable_classes)
for fn, code in low_fit:
    raw_box, raw_area = raw_annotation(fn, code)
    problems[fn].append((code, '알약을 감싸지 않는 box', raw_box, raw_area))
kind_count = collections.Counter(kind for items in problems.values() for _, kind, _, _ in items)

fit_values = np.array(list(box_fit.values()))
fig, axes = plt.subplots(1, 2, figsize=(20, 5.5), layout='constrained', gridspec_kw={'width_ratios': [1, 1.6]})
axes[0].hist(fit_values, bins=np.linspace(0, 1, 51), color='tab:blue')
axes[0].axvline(FIT_THRESHOLD, color='red', linestyle='--', linewidth=1)
axes[0].set_yscale('log')
axes[0].set_xlabel('box 맞음 (다시 만든 box와 라벨 box의 IoU)')
axes[0].set_ylabel('box 수 (로그 눈금)')
axes[0].set_title(f'box {len(fit_values):,}개: 중앙값 {np.median(fit_values):.3f}, {FIT_THRESHOLD} 미만 {(fit_values < FIT_THRESHOLD).sum()}개', fontsize=11)
low_by_class = collections.Counter(code for (fn, code), v in box_fit.items() if v < FIT_THRESHOLD)
n_by_class = collections.Counter(code for fn, code in box_fit)
order = sorted(low_by_class, key=lambda c: -low_by_class[c] / n_by_class[c])[:40]
axes[1].bar([f'K-{c:06d}' for c in order], [low_by_class[c] / n_by_class[c] for c in order], color='tab:orange')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)
axes[1].set_ylabel('box 맞음 0.75 미만 비율')
axes[1].set_title('클래스별 box 맞음 0.75 미만 비율 (높은 40개 클래스)', fontsize=11)
fig.suptitle(f'모든 AI Hub box의 box 맞음 (빨간 점선 = {FIT_THRESHOLD}, 이 아래는 "알약을 감싸지 않는 box"로 고친다), '
             f'믿을 수 없어 뺀 클래스: {", ".join(f"K-{c:06d}" for c in unreliable_classes) or "없음"}', fontsize=13)
save_fig(fig, '2-2_box_fit')
plt.show()

#### 2-2-3 결과

- 쓸 수 있는 box 45,746개의 box 맞음 중앙값은 0.958이다. 0.75 미만은 2,829개(6.2%)다.
- 0.75 미만 비율이 30%를 넘어 검사에서 뺀 클래스는 6개다.

| 클래스 | 0.75 미만 비율 |
|---|---|
| K-005391, K-010487, K-012638, K-031885 | 거의 100% |
| K-041149 | 86% |
| K-027777 | 32% |

- K-027777을 뺀 5개는 바닥과 색이 비슷한 파란 계열 알약이다. (2-2-2 실패 사진)
- 나머지 클래스는 0.75 미만 비율이 5.5%(K-019881) 이하다.
- 이 검사로 새로 찾은 문제 box는 313개다. 좌표 값이 정상 범위라 2-2-1로는 찾지 못한 box다.

#### 2-2-4. 고치는 규칙과 처리 결과

2-2-1의 문제와 "알약을 감싸지 않는 box"를 아래 규칙으로 고친다. 문제 annotation은 모두 라벨 box에서 빼고, 남은 정상 box 밖에서 알약을 찾아 크기와 색으로 짝짓는다. (2-2-2의 box 둘 지우기와 같은 방법)

| 문제 | 짝지을 알약을 찾으면 | 찾지 못하면 |
|---|---|---|
| 원본 bbox가 있음 (이미지 밖, area 불일치, 숫자가 붙은 bbox, 알약을 감싸지 않는 box) | 원본과 짝의 IoU가 0.8 이상이면 bbox는 그대로 두고 area만 다시 계산한다. 아니면 오타 후보 중 짝과 맞는 값으로 고친다("숫자 하나 고침"). 맞는 후보가 없으면 짝의 box로 바꾼다("사진에서 다시 그림") | 쓸 수 있는 원본 bbox는 그대로 둔다. 없으면 너비×높이가 area와 같은 오타 후보가 하나뿐일 때 그 값으로 고친다 |
| 다른 클래스와 box가 똑같음 | 크기와 색으로 그 box의 알약인 클래스를 정한다. 그 클래스는 원본 box, 다른 클래스는 짝의 box를 쓴다 | 다른 클래스가 다른 알약으로 옮겨졌으면 원본 box를 그대로 둔다 |
| 원본 bbox가 없음 (JSON 없음, bbox 빈 값, `-오류` 폴더, 읽을 수 없는 JSON) | 짝의 box를 넣는다 | 고칠 수 없다 |

- 고칠 방법이 하나도 없으면 그 이미지를 학습에서 뺀다. ("찾지 못함")
- 짝짓기 기준
    - 크기·색 차이(`appearance_cost`)가 1.5를 넘는 짝은 만들지 않는다. 정상 라벨 box의 99.5%가 자기 클래스와 1.5 이하다.
    - 2-2-2 검증에서 IoU 중앙값이 0.85 미만인 클래스(파란 알약 등)와, 사진의 두 변에 닿는 덩어리(조명 밖 밝은 모서리)는 짝짓기에 쓰지 않는다.
- 오타 후보는 숫자 하나를 빼거나 넣거나 바꾼 값, 붙은 숫자를 나눈 값이다.
    - 너비×높이가 원본 area와 같은 후보 중 짝과 IoU 0.8 이상인 후보를 먼저 쓴다.
    - 그런 후보가 없으면 모든 후보 중 IoU 0.85 이상인 후보를 쓴다. 후보가 많아 우연히 맞는 값을 피하려고 기준을 높였다.
- 사진으로 확인하지 못한 원본 bbox는, 긴 변이 그 클래스 중앙값과 25% 넘게 다르면 쓰지 않는다. (같은 클래스 정상 box 중 25% 넘게 다른 box는 0.05%)
- `-오류` 폴더의 알약은 사진에 다른 알약이 놓여 있다(2-2-6). 사진으로 확인한 실제 클래스(K-023319 → K-029711, K-016551 → K-016548)의 크기·색으로 짝짓고 그 클래스로 붙인다.
- 마지막으로 문제 갤러리(2-2-5)를 눈으로 보고, 자동 처리가 틀린 box를 `REVIEWED_BOXES`에 적어 고친다. ("눈으로 확인해 고침")
    - 2-2-3이 뺀 클래스도 갤러리에서 틀린 box가 보이면 "눈으로 찾은 틀린 box"로 더해 고친다. (예: 옆 알약을 감싼 K-010487 box)

In [ ]:
def digit_edits(value):
    """숫자 하나에서 숫자 한 개를 빼거나, 넣거나, 바꾼 값을 모두 만든다."""
    text = str(int(value))
    out = set()
    for i in range(len(text) + 1):
        if i < len(text):
            out.add(text[:i] + text[i + 1:])
            out.update(text[:i] + d + text[i + 1:] for d in '0123456789')
        out.update(text[:i] + d + text[i:] for d in '0123456789')
    return {int(t) for t in out if t and not (len(t) > 1 and t[0] == '0')}


def raw_candidates(raw):
    """원본 bbox 값에서 오타를 고친 [x, y, w, h] 후보를 만든다. 이미지 안에 들어가는 후보만 남긴다.

    - 값이 4개인 정수: x, y, w, h 중 하나에서 숫자 한 개를 빼거나, 넣거나, 바꾼 값
    - 값이 3개 이하: 소수점이나 붙은 숫자를 둘로 나눠 값 4개를 만든다. (쉼표가 빠진 것으로 본다. 예: 128.816 → 128, 816)
    """
    tokens = []
    for v in raw:
        tokens += str(v).split('.') if isinstance(v, float) else [str(v)]
    bases = []
    if len(tokens) == 4:
        bases.append([int(t) for t in tokens])
    elif len(tokens) == 3:
        bases += [[int(s) for s in tokens[:i] + [t[:cut], t[cut:]] + tokens[i + 1:]]
                  for i, t in enumerate(tokens) for cut in range(1, len(t))]
    cands = {tuple(b) for b in bases}
    if len(raw) == 4 and all(isinstance(v, int) for v in raw):
        cands |= {tuple(raw[:i] + [v] + raw[i + 1:]) for i in range(4) for v in digit_edits(raw[i])}
    return [list(c) for c in cands if c[2] >= 30 and c[3] >= 30 and c[0] >= 0 and c[1] >= 0
            and c[0] + c[2] <= IMG_W and c[1] + c[3] <= IMG_H]


def choose_box(raw_box, raw_area, found):
    """원본 bbox를 살릴 수 있으면 (box, 처리)를, 없으면 (찾은 box, '사진에서 다시 그림')을 돌려준다."""
    if raw_box and len(raw_box) == 4 and all(isinstance(v, int) for v in raw_box) and raw_box in raw_candidates(raw_box) \
            and box_iou(raw_box, found) >= 0.8:
        return raw_box, '그대로 둠'
    cands = raw_candidates(raw_box) if raw_box else []
    by_area = [c for c in cands if raw_area and c[2] * c[3] == raw_area]
    for pool, threshold in [(by_area, 0.8), (cands, 0.85)]:
        if pool:
            best = max(pool, key=lambda c: box_iou(c, found))
            if box_iou(best, found) >= threshold:
                return best, '숫자 하나 고침'
    return found, '사진에서 다시 그림'


CORNER_GAP = 10  # 알약 범위가 사진의 두 변에서 이 픽셀 안이면 모서리 덩어리로 본다
MAX_COST = 1.5  # 크기·색 차이가 이보다 큰 알약과는 짝짓지 않는다 (정상 라벨 box의 99.5%가 이 안)
LONG_TOLERANCE = 0.25  # 사진으로 확인하지 못한 box는 긴 변이 클래스 중앙값과 이 비율 안일 때만 쓴다
long_values = collections.defaultdict(list)  # K-코드 → 문제가 없는 이미지의 라벨 box 긴 변
for fn in clean_pool:
    for ann in aihub_kaggle[fn]['annotations']:
        long_values[ann['category_id']].append(max(ann['bbox'][2], ann['bbox'][3]))
class_long = {code: float(np.median(v)) for code, v in long_values.items()}  # K-코드 → 긴 변의 중앙값
unfindable_classes = {c for c, v in validation_by_class.items() if np.median(v) < 0.85}  # 위 검증에서 사진으로 다시 찾지 못하는 클래스
# -오류 폴더의 알약은 사진에 다른 알약이 놓였다. 사진으로 확인한 실제 클래스로 짝지어 붙인다 (2-2-6)
ERROR_FOLDER_CLASS = {23319: 29711, 16551: 16548}  # -오류 폴더의 K-코드 → 사진 속 실제 클래스 (K-029711 D|CT, K-016548 GB 주황 원형)
error_folder_labels = {(fn, json_code(j)) for fn, js in jsons_of.items() for j in js if j['pill_folder'].endswith('-오류')}  # (이미지, K-코드)
assert {code for _, code in error_folder_labels} <= set(ERROR_FOLDER_CLASS), '실제 클래스를 정하지 않은 -오류 폴더가 있다'


def fix_candidates(rgb, known_boxes):
    """정상 box 밖에서 알약 범위를 찾고, 사진의 두 변에 닿는 모서리 덩어리(조명 밖의 밝은 모서리)는 뺀다."""
    out = []
    for extent in find_pills(rgb, known_boxes):
        x0, y0, x1, y1 = extent
        touches_x = x0 <= CORNER_GAP or x1 >= IMG_W - CORNER_GAP
        touches_y = y0 <= CORNER_GAP or y1 >= IMG_H - CORNER_GAP
        if not (touches_x and touches_y):
            out.append(extent)
    return out


def usable_raw(raw_box):
    """원본 bbox를 고치지 않고 쓸 수 있는지(이미지 안에 있는 정수 4개) 확인한다."""
    return bool(raw_box) and len(raw_box) == 4 and all(isinstance(v, int) for v in raw_box) and raw_box in raw_candidates(raw_box)


def plausible(box, code):
    """box의 긴 변이 그 클래스 라벨 box 긴 변의 중앙값과 LONG_TOLERANCE 안으로 비슷한지 본다. 중앙값이 없는 클래스는 통과시킨다."""
    return code not in class_long or abs(np.log(max(box[2], box[3]) / class_long[code])) <= np.log(1 + LONG_TOLERANCE)


def solve_image(fn):
    """이미지 하나의 문제 annotation을 고친다. 결과는 문제마다 dict 하나이다."""
    rgb = read_rgb(AIHUB_IMG_DIR / fn)
    items = problems[fn]
    labels = [ERROR_FOLDER_CLASS[code] if (fn, code) in error_folder_labels else code for code, *_ in items]  # 붙일 클래스
    bad = {code for code, *_ in items}
    good_boxes = [a['bbox'] for a in aihub_kaggle.get(fn, {'annotations': []})['annotations'] if a['category_id'] not in bad]
    extents = fix_candidates(rgb, good_boxes)
    labs = [extent_color(rgb, e) for e in extents]
    searchable = [i for i, label in enumerate(labels) if label not in unfindable_classes]  # 사진으로 찾을 수 있는 클래스만 짝짓는다
    appear = np.zeros((len(searchable), len(extents)))  # 크기·색 차이
    cost = np.zeros((len(searchable), len(extents)))  # 짝짓기 비용 = 크기·색 차이 - 원본 bbox 후보와의 IoU
    for row, i in enumerate(searchable):
        code, kind, raw_box, raw_area = items[i]
        ref = class_refs.get(labels[i], no_ref)
        for j, extent in enumerate(extents):
            appear[row, j] = appearance_cost(extent, labs[j], ref) if 'long' in ref else 0
            cost[row, j] = appear[row, j]
            if raw_box and '똑같음' not in kind:  # 원본 bbox가 가리키는 알약이면 짝지을 때 우선한다
                cost[row, j] -= max([box_iou(c, with_margin(extent, ref['margin'])) for c in raw_candidates(raw_box)], default=0)
    cost[appear > MAX_COST] = 1e6  # 크기·색이 너무 다른 짝은 만들지 않는다
    results = []
    pairs = zip(*scipy.optimize.linear_sum_assignment(cost)) if extents and searchable else []
    assigned = {searchable[row]: j for row, j in pairs if appear[row, j] <= MAX_COST}
    owner = {j: labels[i] for i, j in assigned.items()}  # 짝지어진 후보 → 그 K-코드 (갤러리에 근거로 적는다)
    for i, (code, kind, raw_box, raw_area) in enumerate(items):
        label = labels[i]
        result = {'file_name': fn, 'category_id': code, 'label_id': label, 'problem': kind, 'raw_box': raw_box, 'raw_area': raw_area,
                  'n_found': len(extents)}
        if i not in assigned:
            if i in searchable and extents:  # 갤러리에 적을 근거: 크기·색이 가장 비슷한 후보, 그 차이, 그 후보와 짝지어진 클래스
                row = searchable.index(i)
                j = int(np.argmin(appear[row]))
                result.update(nearest=float(appear[row, j]), nearest_owner=owner.get(j),
                              nearest_box=with_margin(extents[j], class_refs.get(label, no_ref)['margin']))
            area_matched = [c for c in raw_candidates(raw_box) if c[2] * c[3] == raw_area and plausible(c, label)] if raw_box and raw_area else []
            if usable_raw(raw_box) and plausible(raw_box, label) and '똑같음' not in kind:
                box, action = raw_box, '그대로 둠 (알약을 찾지 못함)'
            elif len(area_matched) == 1 and '똑같음' not in kind:
                box, action = area_matched[0], '숫자 하나 고침 (area로 확인)'
            else:
                box, action = None, '찾지 못함'
            results.append({**result, 'box': box, 'cost': None, 'action': action})
            continue
        found = with_margin(extents[assigned[i]], class_refs.get(label, no_ref)['margin'])
        if '똑같음' in kind:
            box, action = (raw_box, '그대로 둠') if box_iou(raw_box, found) >= 0.5 else (found, '사진에서 다시 그림')
        else:
            box, action = choose_box(raw_box, raw_area, found)
        row = searchable.index(i)
        results.append({**result, 'box': box, 'found': found, 'action': action, 'cost': float(cost[row, assigned[i]]),
                        'appear': float(appear[row, assigned[i]])})
    # box가 똑같은 쌍에서 다른 클래스가 다른 알약으로 옮겨졌으면, 찾지 못한 쪽이 원본 box의 알약이다
    moved = {tuple(r['raw_box']) for r in results if '똑같음' in r['problem'] and r['action'] == '사진에서 다시 그림'}
    for r in results:
        if ('똑같음' in r['problem'] and r['action'] == '찾지 못함' and tuple(r['raw_box']) in moved and usable_raw(r['raw_box'])
                and plausible(r['raw_box'], r['category_id'])):
            r['box'], r['action'] = r['raw_box'], '그대로 둠 (알약을 찾지 못함)'
    return results


fix_cache = CACHE_DIR / '2-2_box_fixes.json'  # 처리마다 붙일 클래스와 판단 근거(찾은 후보 수, 크기·색 차이)도 저장한다
# 문제 목록, 모든 이미지의 box, 위의 클래스 기준값(finder), -오류 폴더의 실제 클래스가 같을 때만 저장한 결과를 쓴다
problem_key = hashlib.sha1(json.dumps([sorted(problems.items()), [[fn, [a['bbox'] for a in r['annotations']]] for fn, r in sorted(aihub_kaggle.items())],
                                      finder, sorted(ERROR_FOLDER_CLASS.items())]).encode()).hexdigest()
if fix_cache.exists() and json.loads(fix_cache.read_text())['key'] == problem_key:
    box_fixes = json.loads(fix_cache.read_text())['fixes']
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        box_fixes = [r for results in pool.map(solve_image, sorted(problems)) for r in results]
    fix_cache.write_text(json.dumps({'key': problem_key, 'fixes': box_fixes}))

# 아래 문제 갤러리(2-2-5)를 눈으로 확인해, 자동 처리가 틀린 box를 고친다 (주석 #번호 = 갤러리 번호)
# (이미지, K-코드) → (고친 box [x, y, w, h] 또는 '원본'(원본 bbox를 그대로 씀), 고친 이유)
REVIEWED_BOXES = {
    ('K-003832-005391-038162_0_2_0_2_90_000_200.png', 5391): ([134, 846, 196, 195], 'K-038162의 원본 bbox 자리가 이 클래스의 알약(PROSCAR)'),  # #41
    ('K-003832-005391-038162_0_2_0_2_90_000_200.png', 38162): ([625, 790, 239, 268], 'K-005391로 짝지은 알약이 이 클래스의 알약'),  # #41
    ('K-000573-001866-019552-038954_0_2_0_2_70_000_200.png', 38954): ([96, 148, 181, 453], '높이가 길어 아래 끝을 짝지은 알약 box까지 줄임'),  # #298
    ('K-000573-002483-005094-019552_0_2_0_2_90_000_200.png', 5094): ([109, 799, 281, 275], '너비가 모자라 오른쪽 끝을 짝지은 알약 box까지 늘림'),  # #300
    ('K-000573-002483-022627-025438_0_2_0_2_75_000_200.png', 2483): ([627, 92, 226, 469], '너비가 넓어 오른쪽 끝을 짝지은 알약 box까지 줄임'),  # #302
    ('K-000573-003743-005002-019552_0_2_0_2_90_000_200.png', 19552): ([70, 148, 310, 492], '높이가 모자라 아래 끝을 짝지은 알약 box까지 늘림'),  # #304
    ('K-001866-005002-005886-022627_0_2_0_2_75_000_200.png', 5002): ([644, 108, 240, 463], '높이가 길어 위 끝을 짝지은 알약 box까지 줄임'),  # #311
    ('K-001900-004543-016548-024850_0_2_0_2_70_000_200.png', 16548): ('원본', '원본 bbox가 맞음'),  # #316
    ('K-003832-033026-036637_0_2_0_2_90_000_200.png', 33026): ([78, 885, 285, 171], '너비가 모자라 오른쪽 끝을 짝지은 알약 box까지 늘림'),  # #375
    ('K-005391-013900-016232_0_2_0_2_90_000_200.png', 13900): ([656, 744, 151, 274], '너비가 넓어 오른쪽 끝을 짝지은 알약 box까지 줄임'),  # #417
    ('K-005391-016688-041768_0_2_0_2_75_000_200.png', 16688): ([118, 514, 273, 645], '너비가 넓어 오른쪽 끝을 짝지은 알약 box까지 줄임'),  # #419
    ('K-006835-019881-023319-041149_0_2_0_2_90_000_200.png', 19881): ('원본', '원본 bbox가 맞음 (짝지은 알약 box는 알약이 잘림)'),  # #448
    ('K-006835-019881-029711-041149_0_2_0_2_75_000_200.png', 19881): ([611, 717, 163, 249], '왼쪽 끝을 원본 bbox까지 늘림 (짝지은 알약 box는 알약이 잘림)'),  # #454
    ('K-011220-016232-025469-034597_0_2_0_2_70_000_200.png', 25469): ([584, 233, 352, 231], '오른쪽으로 옮김 (짝지은 알약 box는 알약 오른쪽이 잘리고 왼쪽 여백이 큼)'),  # #508
    ('K-016232-025469-031885-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #633
    ('K-016235-020238-025367-027653_0_2_0_2_70_000_200.png', 20238): ([165, 860, 230, 234], '짝지은 알약 box가 맞음'),  # #641
    ('K-016235-025469-031885-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #652
    ('K-003832-016688-024752_0_2_0_2_70_000_200.png', 24752): ('원본', '원본 bbox가 맞음'),  # #743
    ('K-003832-022074-024752_0_2_0_2_70_000_200.png', 24752): ([111, 281, 334, 184], '위 끝을 원본 bbox까지 늘림'),  # #744
    ('K-006835-019881-025200-041149_0_2_0_2_90_000_200.png', 19881): ([646, 798, 157, 246], 'x와 너비를 원본 bbox로 (짝지은 알약 box는 알약이 잘림)'),  # #774
    ('K-006835-019881-029711-041149_0_2_0_2_70_000_200.png', 19881): ('원본', '원본 bbox가 맞음 (고친 너비는 알약이 잘림)'),  # #777
    ('K-006835-019881-029711-041149_0_2_0_2_90_000_200.png', 19881): ([648, 809, 159, 242], 'x와 너비를 원본 bbox로 (짝지은 알약 box는 알약이 잘림)'),  # #778
    ('K-010224-016548-023203-027926_0_2_0_2_90_000_200.png', 16548): ([607, 150, 235, 225], '짝지은 알약 box가 맞음'),  # #800
    ('K-010224-016548-023203-027926_0_2_0_2_90_000_200.png', 27926): ([604, 681, 253, 475], '짝지은 알약 box가 맞음'),  # #800
    ('K-010224-016551-023203-033009_0_2_0_2_70_000_200.png', 23203): ('원본', '원본 bbox가 맞음'),  # #802
    ('K-010487-019881-026993-043233_0_2_0_2_75_000_200.png', 26993): ('원본', '원본 bbox가 맞음 (고친 너비는 알약이 잘림)'),  # #804
    ('K-010487-019881-026993-043233_0_2_0_2_75_000_200.png', 10487): ([555, 0, 240, 427], 'box가 옆 알약(K-026993) 위에 있어 오른쪽의 파란 캡슐로 옮김'),  # #804
    ('K-010487-026455-029711-044266_0_2_0_2_75_000_200.png', 29711): ([35, 0, 414, 468], 'K-010487과 라벨이 바뀜: K-010487의 원본 bbox 자리가 이 클래스의 알약(D|CT)'),  # #808
    ('K-010487-026455-029711-044266_0_2_0_2_75_000_200.png', 10487): ([543, 12, 279, 430], 'K-029711과 라벨이 바뀜: K-029711의 원본 bbox 자리가 이 클래스의 알약(파란 캡슐)'),  # #808
    ('K-011220-016232-022347-025469_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #809
    ('K-011220-016232-025367-025469_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #810
    ('K-011220-022347-025469-029667_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #816
    ('K-011220-022347-025469-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #817
    ('K-011220-022347-025469-036637_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #818
    ('K-011220-025367-025469-029667_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #819
    ('K-011220-025469-028763-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #821
    ('K-011220-025469-029667-034597_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #822
    ('K-011220-025469-030308-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #823
    ('K-011220-025469-031885-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #824
    ('K-012247-016551-024850-029451_0_2_0_2_70_000_200.png', 16551): ([50, 710, 291, 486], '왼쪽 끝을 알약까지 늘림 (짝지은 알약 box는 알약 왼쪽이 잘림)'),  # #827
    ('K-012638-019881-025200-041149_0_2_0_2_90_000_200.png', 19881): ([645, 798, 159, 246], 'x와 너비를 원본 bbox로 (짝지은 알약 box는 알약이 잘림)'),  # #846
    ('K-016232-016235-027653-031885_0_2_0_2_70_000_200.png', 27653): ('원본', '원본 bbox가 맞음'),  # #874
    ('K-016232-022347-025469-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #875
    ('K-016232-025367-025469-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #876
    ('K-016232-025367-025469-038723_0_2_0_2_75_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #877
    ('K-016232-025469-028763-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #878
    ('K-016232-025469-030308-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #879
    ('K-016232-025469-031885-038723_0_2_0_2_75_000_200.png', 25469): ([109, 637, 201, 366], '너비가 모자라 오른쪽 끝을 알약까지 늘림'),  # #880
    ('K-016262-025367-025469-038723_0_2_0_2_70_000_200.png', 25469): ([631, 257, 222, 354], '너비가 모자라 오른쪽 끝을 알약까지 늘림'),  # #894
    ('K-016262-025469-030308-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #895
    ('K-016262-025469-034597-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #896
    ('K-018357-024752-036637_0_2_0_2_70_000_200.png', 24752): ('원본', '원본 bbox가 맞음'),  # #905
}
REVIEW_PROBLEM = '눈으로 찾은 틀린 box'  # box 맞음을 믿을 수 없어 2-2-3이 뺀 클래스라 문제로 잡히지 않았지만, 갤러리에서 틀린 것을 본 box
fix_at = {(f['file_name'], f['category_id']): f for f in box_fixes}  # (이미지, K-코드) → 처리 결과
for (fn, code), (box, reason) in REVIEWED_BOXES.items():
    if (fn, code) not in fix_at:  # 문제로 잡히지 않은 box: 원본 라벨 그대로를 자동 처리로 두고 처리 결과에 더한다
        raw_box, raw_area = raw_annotation(fn, code)
        fix_at[(fn, code)] = {'file_name': fn, 'category_id': code, 'label_id': code, 'problem': REVIEW_PROBLEM, 'raw_box': raw_box,
                              'raw_area': raw_area, 'n_found': None, 'box': raw_box, 'cost': None, 'action': '문제로 잡히지 않음'}
        box_fixes.append(fix_at[(fn, code)])
    fix = fix_at[(fn, code)]
    fix.update(auto_box=fix['box'], auto_action=fix['action'], review=reason,
               box=fix['raw_box'] if box == '원본' else box, action='눈으로 확인해 고침')
    assert usable_raw(fix['box']) and fix['box'] != fix['auto_box'], f'{fn} K-{code:06d}: 눈으로 고친 box가 이미지 밖이거나 자동 처리와 같다'

kinds = [k for k, _ in kind_count.most_common()] + [REVIEW_PROBLEM]
actions = ['그대로 둠', '그대로 둠 (알약을 찾지 못함)', '숫자 하나 고침', '숫자 하나 고침 (area로 확인)', '사진에서 다시 그림', '찾지 못함',
           '눈으로 확인해 고침']
table = collections.Counter((f['problem'], f['action']) for f in box_fixes)
rows = [[kind, *[table[(kind, a)] or '-' for a in actions]] for kind in kinds]
rows.append(['**합계**', *[f'**{sum(table[(k, a)] for k in kinds)}**' for a in actions]])
show_table(['문제', *actions], rows)

#### 2-2-4 결과

| 처리 | annotation |
|---|---|
| 그대로 둠 | 448 |
| 그대로 둠 (알약을 찾지 못함) | 33 |
| 숫자 하나 고침 | 305 |
| 숫자 하나 고침 (area로 확인) | 7 |
| 사진에서 다시 그림 | 287 |
| 눈으로 확인해 고침 | 52 |
| 찾지 못함 | 21 |
| **합계** | **1,153** |

- 793개(69%)는 원본 값을 살렸다. (그대로 둠 481개, 숫자 하나 고침 312개)
- area 불일치 427개 중 337개는 bbox를 그대로 두고 area만 다시 계산한다.
- 알약을 감싸지 않는 box 313개 중 156개는 숫자 하나를 고쳐 맞췄다. 좌표 오타가 box를 다른 자리로 밀어낸 경우다.
- 사진에서 다시 그린 287개 중 174개는 원본 bbox가 없던 라벨 누락이다. (bbox 빈 값 102개, JSON 없음 47개 등)
- 눈으로 확인해 고친 52개는 문제 갤러리(2-2-5)에서 찾았다.

찾지 못한 21개가 있는 이미지는 학습에서 뺀다. box가 없는 알약은 배경으로 학습되기 때문이다.

#### 2-2-5. 문제 갤러리

2-2에서 문제가 있는 이미지를 모두 한 장씩 그려, 문제가 실제로 있는지와 고친 box가 맞는지 눈으로 본다.

| 표시 | 뜻 |
|---|---|
| 빨간 점선 | 원본 bbox. 고친 box와 다르면 오른쪽 아래에 "원본 K-코드"를 적는다 |
| 자홍 점선 | "알약을 감싸지 않는 box"로 판단한 근거. 원본 bbox 주변에서 잰 알약 범위 + 클래스 여백 (2-2-3) |
| 주황 점선 | 사진에서 찾아 짝지은 알약 box (고친 box와 다를 때만) |
| 노랑 점선 | 짝을 찾지 못한 문제에서 크기·색이 가장 비슷한 후보 |
| 하늘 점선 | 눈으로 고친 문제의, 고치기 전 자동 처리 box |
| 파란 실선 | 고친 box. 붙인 클래스의 K-코드와 각인을 적는다 |
| 연두 실선 | 문제가 없는 라벨 box |

- 제목은 번호와 파일명이다. 이미지를 빼는 경우 빨간색으로 쓴다.
- 사진 아래에는 문제마다 "문제로 판단한 값"과 "처리를 정한 근거"를 적는다.
- 이미지는 문제 순서(라벨 누락 → 이미지 밖 → 다른 클래스와 box가 똑같음 → area 불일치 → 알약을 감싸지 않는 box → 눈으로 찾은 틀린 box)로 놓는다.
- 시트는 `runs/eda_raewoo/figures_01/2-2_problem_gallery/`에 8장씩 저장하고, 아래에는 첫 시트만 보인다. 번호로 이미지를 찾는 색인은 같은 폴더의 `index.csv`다.

In [ ]:
PROBLEM_GALLERY_DIR = FIG_DIR / '2-2_problem_gallery'
PROBLEM_ROWS, PROBLEM_COLS = 2, 4  # 시트 한 장에 이미지 8장
PROBLEM_ORDER = ['라벨 누락', '이미지 밖', '다른 클래스와 box가 똑같음', 'area 불일치', '알약을 감싸지 않는 box', REVIEW_PROBLEM]  # 시트에 놓는 순서
TEXT_WIDTH = 92  # 사진 아래 설명의 한 줄 글자 폭 (한글 2칸, 그 밖 1칸)


def problem_rank(problem):
    """문제 이름에 들어 있는 PROBLEM_ORDER 항목 중 가장 앞 순서를 돌려준다."""
    return min(i for i, name in enumerate(PROBLEM_ORDER) if name in problem)


def drawable(box):
    """사진에 그릴 수 있는 xywh box(숫자 4개, 너비와 높이가 양수)인지 확인한다."""
    return bool(box) and len(box) == 4 and all(isinstance(v, (int, float)) for v in box) and box[2] > 0 and box[3] > 0


def problem_reason(fix):
    """문제 annotation을 문제로 판단한 값을 적는다. (2-2-1 표의 검사, 2-2-3의 box 맞음)"""
    fn, code, kind, raw, area = fix['file_name'], fix['category_id'], fix['problem'], fix['raw_box'], fix['raw_area']
    if kind.startswith('라벨 누락'):
        return '원본 bbox 없음' if raw is None else f'원본 bbox {raw}, area {area}'
    if kind == REVIEW_PROBLEM:
        return f'box 맞음 {box_fit[(fn, code)]:.2f}이지만 2-2-3이 box 맞음을 믿을 수 없어 뺀 클래스라 문제로 잡히지 않음, 갤러리에서 틀린 box를 봄'
    x, y, w, h = raw
    reasons = []
    if '똑같음' in kind:
        partners = [f'K-{c:06d}' for c, _, b, _ in problems[fn] if b == raw and c != code]
        reasons.append(f"box {raw}가 {', '.join(partners)}의 box와 똑같음")
    if '이미지 밖' in kind:
        if x < 0 or y < 0:
            reasons.append(f'왼쪽 위 ({x}, {y})가 0보다 작음')
        if x + w > IMG_W:
            reasons.append(f'오른쪽 끝 x+너비 {x + w} > {IMG_W}')
        if y + h > IMG_H:
            reasons.append(f'아래 끝 y+높이 {y + h} > {IMG_H}')
    if 'area 불일치' in kind:
        reasons.append(f'area {area} ≠ 너비×높이 {w}×{h}={w * h}')
    if kind == '알약을 감싸지 않는 box':
        reasons.append(f'box 맞음 {box_fit[(fn, code)]:.2f} < {FIT_THRESHOLD} (원본 bbox와 자홍 점선의 IoU)')
    return ', '.join(reasons)


def changed_values(raw, box):
    """원본 bbox에서 고친 값만 '높이 488→48'처럼 적는다. 값이 4개가 아니면(붙은 숫자를 나눔) 앞뒤 값을 모두 적는다."""
    if len(raw) == 4:
        return ', '.join(f'{name} {a}→{b}' for name, a, b in zip(['x', 'y', '너비', '높이'], raw, box) if a != b)
    return f'{raw}→{box}'


def not_found_reason(fix):
    """사진에서 짝지을 알약을 찾지 못한 이유를 2-2-4의 판단 순서대로 적는다."""
    code = fix['label_id']
    if code in unfindable_classes:
        return f'사진으로 찾을 수 없는 클래스 (위 검증 IoU 중앙값 {np.median(validation_by_class[code]):.2f} < 0.85)'
    if not fix['n_found']:
        return '사진에서 찾은 알약 후보 없음'
    owner = f" (그 후보는 K-{fix['nearest_owner']:06d} 클래스와 짝지어짐)" if fix['nearest_owner'] is not None else ''
    if fix['nearest'] > MAX_COST:
        return f"가장 비슷한 후보(노랑 점선)도 크기·색 차이 {fix['nearest']:.2f} > {MAX_COST}{owner}"
    if owner:
        return f"크기·색이 맞는 후보(노랑 점선, 차이 {fix['nearest']:.2f})를 다른 클래스가 가져감{owner}"
    return f"크기·색이 비슷한 후보(노랑 점선, 차이 {fix['nearest']:.2f})와 짝짓지 못함"


def fallback_failure(fix):
    """짝을 찾지 못했을 때 원본 bbox로도 고치지 못한 이유를 적는다."""
    code, kind, raw, area = fix['label_id'], fix['problem'], fix['raw_box'], fix['raw_area']
    if not raw:
        return '쓸 원본 bbox도 없음'
    if '똑같음' in kind:
        return '원본 box는 같은 box를 쓰는 다른 클래스의 알약'
    reasons = []
    if not usable_raw(raw):
        reasons.append('원본 bbox가 이미지 안의 값 4개가 아님')
    elif not plausible(raw, code):
        reasons.append(f'원본 긴 변이 클래스 중앙값과 25% 넘게 다름 ({max(raw[2], raw[3])} / {class_long[code]:.0f})')
    n_area = sum(1 for c in raw_candidates(raw) if area and c[2] * c[3] == area and plausible(c, code))
    reasons.append(f'너비×높이가 area와 같은 오타 후보 {n_area}개')
    return ', '.join(reasons)


def action_reason(fix):
    """처리 결과를 그렇게 정한 근거를 2-2-4의 규칙 순서대로 적는다."""
    code, kind, raw, area, box, action = (fix[k] for k in ['label_id', 'problem', 'raw_box', 'raw_area', 'box', 'action'])
    if code != fix['category_id']:  # -오류 폴더: 사진 속 실제 클래스로 짝지어 붙였다
        return f'사진 속 실제 클래스(K-{code:06d})로 붙임 (2-2-6); ' + action_reason({**fix, 'category_id': code})
    if action == '눈으로 확인해 고침':  # 갤러리를 눈으로 확인해 고쳤다 (2-2-4의 REVIEWED_BOXES)
        auto = f"{fix['auto_action']} {fix['auto_box']}" if fix['auto_box'] else fix['auto_action']
        return f"{fix['review']} (자동 처리: {auto})"
    if 'found' in fix:  # 사진에서 찾은 알약과 짝지어졌다 (주황 점선 = 짝지은 알약 box)
        found = fix['found']
        match = f"크기·색 차이 {fix['appear']:.2f}" if code in class_refs else '크기·색 기준값이 없어 차이를 0으로 계산'
        if '똑같음' in kind:
            same = box_iou(raw, found)
            if action == '그대로 둠':
                return f'짝지은 알약({match})이 원본 box 자리(IoU {same:.2f} ≥ 0.5) → 원본 box는 이 클래스'
            return f'짝지은 알약({match})이 원본 box 밖(IoU {same:.2f} < 0.5) → 원본 box는 다른 클래스, 짝지은 알약 box로 바꿈'
        if action == '그대로 둠':
            note = ' → area 값만 틀림' if 'area 불일치' in kind else ''
            return f'짝지은 알약({match}) box와 원본 bbox의 IoU {box_iou(raw, found):.2f} ≥ 0.8{note}'
        if action == '숫자 하나 고침':
            area_text = '너비×높이 = area' if box[2] * box[3] == area else '너비×높이 ≠ area'
            return f'{changed_values(raw, box)} ({area_text}), 짝지은 알약({match}) box와 IoU {box_iou(box, found):.2f}'
        if not raw:
            return f'원본 bbox 없음 → 짝지은 알약({match}) box를 넣음'
        cands = raw_candidates(raw)
        if not cands:
            return f'원본 bbox로 만든 이미지 안의 오타 후보 없음 → 짝지은 알약({match}) box로 바꿈'
        best = max(box_iou(c, found) for c in cands)
        return f'원본과 오타 후보 중 짝지은 알약({match}) box와 맞는 값 없음(가장 높은 IoU {best:.2f}) → 짝지은 알약 box로 바꿈'
    why = not_found_reason(fix)
    if action == '그대로 둠 (알약을 찾지 못함)':
        if '똑같음' in kind:
            return f'{why}; 같은 box의 다른 클래스가 사진의 다른 알약으로 옮겨짐 → 원본 box는 이 클래스'
        median = f'{class_long[code]:.0f}' if code in class_long else '없음'
        return f'{why}; 원본 bbox가 이미지 안의 값 4개이고 긴 변이 클래스 중앙값과 25% 안 ({max(raw[2], raw[3])} / {median}) → 그대로 둠'
    if action == '숫자 하나 고침 (area로 확인)':
        return f'{why}; 너비×높이가 area와 같은 오타 후보가 하나뿐 ({changed_values(raw, box)}, area {area})'
    return f'{why}; {fallback_failure(fix)} → 이 이미지를 학습에서 뺌'


def wrap_text(prefix, text, indent):
    """prefix 뒤에 text를 이어 쓰고, 글자 폭(한글 2칸, 그 밖 1칸)이 TEXT_WIDTH를 넘으면 띄어쓰기에서 줄을 나눈다."""
    lines, line = [], prefix
    for word in text.split(' '):
        candidate = line + word if line.endswith(' ') else f'{line} {word}'
        if sum(2 if unicodedata.east_asian_width(ch) in 'WF' else 1 for ch in candidate) > TEXT_WIDTH and line.strip():
            lines.append(line.rstrip())
            line = indent + word
        else:
            line = candidate
    return lines + [line]


def problem_panel(ax, n, fn, fixes):
    """문제 이미지 한 장에 라벨 box, 원본 bbox, 판단 근거 box, 고친 box를 그리고, 사진 아래에 문제마다 근거와 처리를 적는다."""
    # 사진은 절반 크기로 줄여 그리고, extent로 원본 좌표계에 맞춰 box는 원본 좌표 그대로 그린다
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn).convert('RGB').reduce(2), extent=(0, IMG_W, IMG_H, 0))
    bad = {fix['category_id'] for fix in fixes}
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        if ann['category_id'] not in bad:
            x, y, w, h = ann['bbox']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=1.5))
            ax.text(x + 4, y + h - 4, f"K-{ann['category_id']:06d}", fontsize=6, va='bottom',
                    bbox=dict(facecolor='lime', alpha=0.8, pad=1, linewidth=0), clip_on=True)
    lines = []
    for fix in fixes:
        code = fix['category_id']
        if drawable(fix['raw_box']):
            x, y, w, h = fix['raw_box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=2, linestyle='--'))
            # 원본 bbox가 고친 box와 다르고 사진에 걸쳐 있으면, 사진 안에 보이는 오른쪽 아래 모서리에 K-코드를 적는다
            if fix['raw_box'] != fix['box'] and x < IMG_W and y < IMG_H and x + w > 0 and y + h > 0:
                ax.text(min(x + w, IMG_W) - 4, min(y + h, IMG_H) - 4, f'원본 K-{code:06d}', fontsize=6, ha='right', va='bottom',
                        color='white', bbox=dict(facecolor='red', alpha=0.8, pad=1, linewidth=0), clip_on=True)
        if fix['problem'] == '알약을 감싸지 않는 box':  # box 맞음을 잰 box: 원본 bbox 주변에서 잰 알약 범위 + 클래스 여백 (2-2-3)
            extent, _ = box_measures[(fn, tuple(fix['raw_box']))]
            if extent:
                x, y, w, h = with_margin(extent, class_refs.get(code, no_ref)['margin'])
                ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='magenta', linewidth=2, linestyle=':'))
        if fix.get('found') and fix['found'] != fix['box']:
            x, y, w, h = fix['found']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='orange', linewidth=2, linestyle=':'))
        if fix.get('nearest_box'):  # 짝짓지 못한 문제: 크기·색이 가장 비슷한 후보
            x, y, w, h = fix['nearest_box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='yellow', linewidth=2, linestyle=':'))
            ax.text(x + 4, y + h - 4, f"K-{code:06d} 후보 차이 {fix['nearest']:.2f}", fontsize=6, va='bottom',
                    bbox=dict(facecolor='yellow', alpha=0.8, pad=1, linewidth=0), clip_on=True)
        if fix.get('auto_box') and fix['auto_box'] not in (fix['box'], fix['raw_box']):  # 눈으로 고치기 전 자동 처리 box
            x, y, w, h = fix['auto_box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='cyan', linewidth=2, linestyle='--'))
        if fix['box']:
            x, y, w, h = fix['box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='blue', linewidth=2.5))
            label_id = fix['label_id']  # 붙인 클래스 (-오류 폴더만 파일명의 K-코드와 다르다)
            label = pill_label(aihub_example[label_id]) if label_id in aihub_example else f'K-{label_id:06d}'
            ax.text(max(x, 0) + 4, max(y, 0) + 4, label, fontsize=7, va='top', color='white',
                    bbox=dict(facecolor='blue', alpha=0.8, pad=1, linewidth=0))
        lines += wrap_text(f"K-{code:06d} {fix['problem']}: ", problem_reason(fix), '    ')
        lines += wrap_text(f"  → {fix['action']}: ", action_reason(fix), '      ')
    ax.set_xlim(0, IMG_W)
    ax.set_ylim(IMG_H, 0)
    excluded = any(fix['action'] == '찾지 못함' for fix in fixes)
    ax.set_title(f'#{n} {fn}' + (' (이미지 제외)' if excluded else ''), fontsize=8, loc='left', color='red' if excluded else 'black')
    ax.set_xlabel('\n'.join(lines), fontsize=7, loc='left')
    ax.set_xticks([])
    ax.set_yticks([])


fixes_by_file = collections.defaultdict(list)  # 이미지 → 그 이미지의 문제 annotation 처리 결과
for fix in box_fixes:
    fixes_by_file[fix['file_name']].append(fix)
problem_files = sorted(fixes_by_file, key=lambda fn: (min(problem_rank(f['problem']) for f in fixes_by_file[fn]), fn))
per_sheet = PROBLEM_ROWS * PROBLEM_COLS
n_problem_sheets = int(np.ceil(len(problem_files) / per_sheet))
PROBLEM_GALLERY_DIR.mkdir(exist_ok=True)
for old_sheet in PROBLEM_GALLERY_DIR.glob('p*.jpg'):  # 이미지 수가 줄어 남는 옛 시트가 없게 지운다
    old_sheet.unlink()
problem_index = []
for page in range(n_problem_sheets):
    chunk = problem_files[page * per_sheet:(page + 1) * per_sheet]
    sheet_name = f'p{page + 1:03d}.jpg'
    fig, axes = plt.subplots(PROBLEM_ROWS, PROBLEM_COLS, figsize=(20, 17.5), layout='constrained')
    for k, (ax, fn) in enumerate(zip(axes.flat, chunk)):
        n = page * per_sheet + k + 1
        problem_panel(ax, n, fn, fixes_by_file[fn])
        problem_index += [{'sheet': sheet_name, 'number': n, 'file_name': fn, 'category_id': fix['category_id'], 'label_id': fix['label_id'],
                           'problem': fix['problem'], 'action': fix['action'], 'bbox_before': json.dumps(fix['raw_box']),
                           'bbox_after': json.dumps(fix['box']), 'found_box': json.dumps(fix.get('found')),
                           'auto_action': fix.get('auto_action', ''), 'bbox_auto': json.dumps(fix.get('auto_box')) if 'auto_box' in fix else '',
                           'problem_reason': problem_reason(fix), 'action_reason': action_reason(fix)}
                          for fix in fixes_by_file[fn]]
    for ax in axes.flat[len(chunk):]:
        ax.axis('off')
    groups = ', '.join(name for i, name in enumerate(PROBLEM_ORDER)
                       if any(min(problem_rank(f['problem']) for f in fixes_by_file[fn]) == i for fn in chunk))
    fig.suptitle(f'2-2 문제 이미지 {page * per_sheet + 1}~{page * per_sheet + len(chunk)}번째 / {len(problem_files)}장 '
                 f'(시트 {page + 1}/{n_problem_sheets}, {groups})\n'
                 '빨간 점선 = 원본 bbox, 자홍 점선 = box 맞음을 잰 알약 범위 + 여백, 주황 점선 = 짝지은 알약 box, '
                 '노랑 점선 = 짝짓지 못한 문제의 가장 비슷한 후보\n'
                 '하늘 점선 = 눈으로 고치기 전 자동 처리 box, 파란 실선 = 고친 box, 연두 = 문제없는 라벨 box, 빨간 제목 = 이미지 제외\n'
                 '사진 아래: 문제마다 "K-코드 문제: 문제로 판단한 값"과 "→ 처리: 그렇게 정한 근거"', fontsize=14)
    fig.savefig(PROBLEM_GALLERY_DIR / sheet_name, dpi=90, bbox_inches='tight')  # 사진 아래 설명이 긴 시트도 아래가 잘리지 않게 한다
    if page:
        plt.close(fig)
with open(PROBLEM_GALLERY_DIR / 'index.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['sheet', 'number', 'file_name', 'category_id', 'label_id', 'problem', 'action',
                                           'bbox_before', 'bbox_after', 'found_box', 'auto_action', 'bbox_auto',
                                           'problem_reason', 'action_reason'])
    writer.writeheader()
    writer.writerows(problem_index)
plt.show()

#### 2-2-5 결과

- 문제가 있는 이미지는 905장이고, 시트 114장에 나눠 저장했다.

그림에서 보이는 것 (첫 시트, 라벨 누락 8장)

- 고친 파란 box는 모두 빠졌던 알약 하나를 빈틈없이 감싼다.
- 짝지은 크기·색 차이는 대부분 0.09~0.39다. #3은 0.73과 1.31로 기준 1.5 안이지만 큰 편이다.
- 모든 알약이 연두나 파란 box 안에 들어 있다.

시트를 모두 눈으로 보고 찾은 틀린 처리 52개를 `REVIEWED_BOXES`에 적어 고쳤다. (2-2-4 표의 "눈으로 확인해 고침")

#### 2-2-6. `-오류` 폴더

- `-오류` 폴더의 JSON이 왜 오류인지 본다.
    - 표: JSON마다 폴더 이름, drug_N, 제품명, bbox
    - 그림: 이미지마다 고치기 전 라벨(연두)과 2-2-4에서 고친 뒤 라벨(고친 box는 파랑)을 나란히 그린다. 한 줄이 한 조합의 각도별 사진 3장이다.
- 폴더 이름의 알약이 사진에 있는지, 다른 알약이 놓였는지 본다.

In [ ]:
error_jsons = [j['path'] for j in aihub_jsons_1st if j['pill_folder'].endswith('-오류')]  # aihub_jsons_1st는 경로 순서이다
rows = []
for path in error_jsons:
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        rows.append([f'`{path.name}`', unicodedata.normalize('NFC', path.parent.name), '(읽을 수 없음)', '', ''])
        continue
    image = data['images'][0]
    rows.append([f'`{path.name}`', unicodedata.normalize('NFC', path.parent.name), image['drug_N'], image['dl_name'],
                 [a.get('bbox') for a in data['annotations']]])
show_table(['JSON', '폴더', 'drug_N', '제품명', 'bbox'], rows)

In [ ]:
error_files = sorted({path.name.replace('.json', '.png') for path in error_jsons})  # 이름순이라 한 조합의 각도별 사진 3장이 이어진다
n_rows = int(np.ceil(len(error_files) / 3))
fig, axes = plt.subplots(n_rows, 6, figsize=(24, 6.6 * n_rows), layout='constrained', squeeze=False)
for k, fn in enumerate(error_files):  # fixes_by_file은 위 문제 갤러리 셀에서 만든다
    image = Image.open(AIHUB_IMG_DIR / fn)
    fixes = fixes_by_file[fn]
    fixed_codes = {fix['category_id'] for fix in fixes}
    # 고치기 전: 쓸 수 있는 원본 라벨 box (-오류 폴더의 bbox []는 그릴 수 없다). 고친 뒤: 문제없는 box는 그대로, 문제 box는 2-2의 처리 결과
    before = [(ann['category_id'], ann['bbox'], 'lime') for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations'] if usable_bbox(ann['bbox'])]
    after = [(code, box, 'lime') for code, box, _ in before if code not in fixed_codes] + [(fix['label_id'], fix['box'], 'blue') for fix in fixes if fix['box']]
    changes = [f"K-{fix['category_id']:06d}" + (f" → K-{fix['label_id']:06d}" if fix['label_id'] != fix['category_id'] else '') + f": {fix['action']}"
               for fix in fixes]
    excluded = any(fix['action'] == '찾지 못함' for fix in fixes)
    for ax, boxes, title, color in [(axes[k // 3, 2 * (k % 3)], before, f'고치기 전: {fn}\nbbox 없음: {missing_text(fn)}', 'black'),
                                    (axes[k // 3, 2 * (k % 3) + 1], after, '고친 뒤\n' + '\n'.join(changes), 'red' if excluded else 'blue')]:
        ax.imshow(image)
        for code, (x, y, w, h), box_color in boxes:
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=box_color, linewidth=2.5))
            label = pill_label(aihub_example[code]) if code in aihub_example else f'K-{code:06d}'
            text_y, va = (y - 8, 'bottom') if y >= 40 else (y + 8, 'top')  # 사진 위 끝에 붙은 box는 이름을 안쪽에 적어 제목을 가리지 않는다
            ax.text(x, text_y, label, fontsize=7, va=va, color='white' if box_color == 'blue' else 'black',
                    bbox=dict(facecolor=box_color, alpha=0.85, pad=1.5, linewidth=0))
        ax.set_title(title, fontsize=8, color=color)
        ax.set_xticks([])
        ax.set_yticks([])
for ax in axes.flat[2 * len(error_files):]:
    ax.axis('off')
fig.suptitle('-오류 폴더 JSON이 있는 이미지의 고치기 전과 뒤 (한 줄 = 한 조합의 각도별 사진 3장, 두 칸씩 왼쪽 = 고치기 전, 오른쪽 = 고친 뒤)\n'
             '연두 = 원본 라벨 box, 파랑 = 2-2에서 고친 box와 붙인 클래스 (-오류 폴더의 알약은 사진 속 실제 클래스로 붙였다)', fontsize=14)
save_fig(fig, '2-2_error_folder')
plt.show()

#### 2-2-6 결과

- `-오류` 폴더 JSON은 9개다. K-023319 폴더 2개에 6개, K-016551 폴더 1개에 3개가 있다.
- 읽을 수 있는 8개는 bbox가 `[[]]`로 비어 있고, 1개는 읽을 수 없다.
- drug_N과 제품명은 폴더 이름과 맞는다. JSON 값으로는 오류 이유를 알 수 없다.

그림에서 보이는 것

- 9장 모두 폴더 이름의 알약이 사진에 없고, 다른 알약이 box 없이 놓여 있다.

| 폴더 | 사진에 있는 알약 |
|---|---|
| K-023319 (디카맥스디플러스정, 연두 타원형) 6장 | 흰 D\|CT 알약 (K-029711) |
| K-016551 (동아가바펜틴정 800mg, DGTH) 3장 | 노란 GB 알약 (K-016548) |

- AI Hub는 다른 알약이 놓여 찍힌 사진의 라벨을 `-오류` 폴더로 모은 것으로 보인다.
- 고친 뒤에는 파란 box가 실제 알약을 감싸고, 실제 알약의 클래스가 붙는다.
- 둘째 줄 70° 사진은 K-019881 box도 옆 캡슐 위에 있었는데, 숫자 하나를 고쳐 Z 알약으로 옮겼다.

`-오류` 폴더 알약은 사진 속 실제 클래스로 붙인다. 다만 이 세 조합은 모두 뒤에서 뺀다. K-016551 조합 3장은 같은 장면(1-6), K-023319 조합 6장은 같은 알약 구성(2-6)이다.

#### 2-2-7. 고친 뒤 다시 검사: 라벨 누락과 클래스 불일치

- 2-2-4의 처리를 적용한 라벨에 2-1-1과 같은 검사를 다시 한다.
- 처리에 box가 있으면 그 box를 쓴다. "찾지 못함"인 알약은 box가 없어서 라벨 누락으로 남는다.
    - 2-1의 라벨 누락이었던 알약은 원래 원인 행에, 좌표 문제였던 알약은 "2-2에서 고치지 못해 box를 뺀 알약" 행에 센다.
- `-오류` 폴더 알약은 실제 클래스로 붙였으므로 파일명의 K-코드 자리를 채운 것으로 세고, 마지막 행에 따로 적는다.
- 칸은 "객체 수 / 이미지 수"이고, 2-1-1 표 → copy 재배정 뒤(2-1-3) → 2-2 수정 뒤 순서다.
- 마지막 열은 고친 뒤에도 남은 이미지와 문제 갤러리의 시트·번호다.

In [ ]:
def label_issues(labels_of, photos=None):
    """2-1-1 표와 같은 기준으로 이미지마다 파일명의 K-코드와 라벨을 맞대어 본다.

    Args:
        labels_of (dict): 이미지 파일명 → [(K-코드, bbox를 쓸 수 없는 원인)]. 원인이 None이면 쓸 수 있는 box이다.
        photos (list): 검사할 촬영 PNG 이름. 없으면 1차 정제 후 촬영 PNG(1-1)이다.

    Returns:
        tuple: (라벨 누락 [(이미지, K-코드, 원인)], 파일명에 없는 K-코드 [(이미지, K-코드)], 남는 box [(이미지, K-코드, 남는 수)])
    """
    missing, extra, surplus = [], [], []
    for fn in aihub_photos_1st if photos is None else photos:
        named = collections.Counter(pill_codes(fn))
        causes_of = collections.defaultdict(list)  # K-코드 → 그 K-코드의 라벨마다 bbox를 쓸 수 없는 원인
        for code, cause in labels_of.get(fn, []):
            causes_of[code].append(cause)
        for code in sorted(set(named) | set(causes_of)):
            causes = causes_of[code]
            if code not in named:
                extra += [(fn, code)] * len(causes)
                continue
            unusable = [cause for cause in causes if cause is not None]
            n_usable = len(causes) - len(unusable)
            # 쓸 수 없는 라벨의 원인을 먼저 붙이고, 라벨이 모자라면 JSON 없음이다 (2-1-1)
            missing += [(fn, code, unusable[k] if k < len(unusable) else 'JSON 없음') for k in range(named[code] - n_usable)]
            if n_usable > named[code]:
                surplus.append((fn, code, n_usable - named[code]))
    return missing, extra, surplus


UNFIXED = '2-2에서 고치지 못해 box를 뺀 알약'
fix_of = {(f['file_name'], f['category_id']): f for f in box_fixes}  # (이미지, K-코드) → 2-2의 처리
assert len(fix_of) == len(box_fixes), '2-2의 처리에 같은 (이미지, K-코드)가 두 번 있다'
# 2-1-1 표: 1-3이 넣은 그대로의 JSON (1-5의 jsons_of는 2-1-3이 바꾸지 않았다)
labels_21 = {fn: [(json_code(j), bbox_problem(j)) for j in js] for fn, js in jsons_of.items()}
# 2-2 수정 뒤: 처리에 box가 있으면 그 box를 쓰고, 찾지 못한 알약은 원인을 남긴다
labels_22 = collections.defaultdict(list)
for fn, record in aihub_kaggle.items():
    for ann in record['annotations']:
        if (fn, ann['category_id']) not in fix_of:
            assert usable_bbox(ann['bbox']), f"{fn} K-{ann['category_id']:06d}: 쓸 수 없는 bbox가 2-2 표에 없다"
            labels_22[fn].append((ann['category_id'], None))
for (fn, code), fix in fix_of.items():
    if fix['box']:
        labels_22[fn].append((code, None if usable_bbox(fix['box']) else UNFIXED))
    elif fix['problem'].startswith('라벨 누락: '):
        if not fix['problem'].startswith('라벨 누락: JSON 없음'):  # JSON이 없던 알약은 라벨을 두지 않아 다시 JSON 없음으로 센다
            labels_22[fn].append((code, fix['problem'].removeprefix('라벨 누락: ')))
    else:
        labels_22[fn].append((code, UNFIXED))

missing_21, extra_21, surplus_21 = label_issues(labels_21)
assert sorted(m for m in missing_21 if m[:2] not in copy_filled) == sorted(m[:3] for m in missing_labels), \
    '2-1-1 표와 같은 기준으로 다시 센 라벨 누락에서 copy로 채운 알약을 빼면 2-1-3의 missing_labels와 같아야 한다'
states = {'2-1-1 표': (missing_21, extra_21, surplus_21),
          'copy 재배정 뒤 (2-1-3)': (missing_labels, extra_labels, [s[:3] for s in surplus_labels]),
          '2-2 수정 뒤': label_issues(labels_22)}
gallery_of = {row['file_name']: f"{row['sheet']} #{row['number']}" for row in problem_index}  # 이미지 → 2-2 문제 갤러리의 시트와 번호


def count_text(items, weight=lambda item: 1):
    """항목 목록을 '객체 수 / 이미지 수'로 적는다. 없으면 0이다."""
    return f'{sum(map(weight, items))} / {len({item[0] for item in items})}장' if items else '0'


def left_text(items):
    """2-2 수정 뒤에도 남은 이미지를 2-2 문제 갤러리의 시트, 번호와 함께 모두 적는다."""
    return '<br>'.join(f"`{fn}` ({gallery_of.get(fn, '갤러리에 없음')})" for fn in sorted({item[0] for item in items}))


after = states['2-2 수정 뒤']
rows = []
for group in MISSING_GROUPS + [UNFIXED]:
    per_state = [[m for m in missing if missing_group(m) == group] for missing, _, _ in states.values()]
    if any(per_state):
        rows.append([group, *map(count_text, per_state), left_text(per_state[-1])])
rows.append(['**라벨 누락 합계**', *[f'**{count_text(missing)}**' for missing, _, _ in states.values()], ''])
rows.append(['클래스 불일치: 파일명에 없는 K-코드', *[count_text(extra) for _, extra, _ in states.values()], left_text(after[1])])
rows.append(['클래스 불일치: 한 이미지에 같은 K-코드의 box가 파일명보다 많음',
             *[count_text(surplus, weight=lambda s: s[2]) for _, _, surplus in states.values()], left_text(after[2])])
# -오류 폴더: 파일명의 K-코드 자리를 사진 속 실제 클래스로 채웠다 (위 2-2 수정 뒤 열에서는 라벨이 있는 것으로 센다)
relabeled = [(f['file_name'], f['category_id'], f['label_id']) for f in box_fixes if f['box'] and f['label_id'] != f['category_id']]
pairs = collections.Counter((code, label) for _, code, label in relabeled)
rows.append(['참고: `-오류` 폴더 알약을 사진 속 실제 클래스로 붙임', '-', '-', count_text(relabeled),
             '<br>'.join(f'K-{code:06d} → K-{label:06d}: {n}개' for (code, label), n in sorted(pairs.items()))])
show_table(['원인', *states, '2-2 수정 뒤에 남은 이미지 (2-2 문제 갤러리 시트 #번호)'], rows)

#### 2-2-7 결과

| 단계 | 라벨 누락 | 같은 K-코드의 box가 남음 |
|---|---|---|
| 2-1-1 표 | 234개 / 175장 | 23개 / 18장 |
| copy 재배정 뒤 (2-1-3) | 211개 / 153장 | 0 |
| 2-2 수정 뒤 | 21개 / 20장 | 0 |

- 남은 21개는 사진에서 알약을 찾지 못한 경우다. bbox 빈 값 8개(7장), 숫자가 붙은 bbox 1개(1장), 2-2에서 고치지 못해 box를 뺀 알약 12개(12장)다.
- 남은 20장 중 17장의 파일명에 바닥과 색이 비슷한 파란 계열 클래스(K-005391, K-010487, K-012638, K-031885, K-041149)가 있다.
- `-오류` 폴더 알약 9개는 실제 클래스로 붙었다. (K-016551 → K-016548 3개, K-023319 → K-029711 6개)

#### 2-2-8. 고친 뒤 다시 검사: 좌표 오류

- 2-2-4의 처리를 적용한 라벨에 2-2-1과 같은 검사를 다시 한다. area는 고친 box의 너비×높이로 다시 계산한다.
- 2-2-1의 검사를 함수(`table_problems`)로 옮겼다. 고치기 전 라벨에 먼저 적용해 2-2-1 표와 결과가 같은지 assert로 확인한다.
- box를 찾지 못한 좌표 문제는 box를 빼고 "라벨 누락: 2-2에서 고치지 못해 box를 뺀 알약" 행으로 옮긴다.
- "알약을 감싸지 않는 box"(2-2-3)는 2-2-1의 검사가 아니라서 이 표에 없다. 처리 결과는 2-2-4 표에 있다.

In [ ]:
def table_problems(anns_of, missing):
    """2-2-1 표와 같은 검사로 이미지마다 문제가 있는 알약을 찾는다.

    Args:
        anns_of (dict): 이미지 → [(K-코드, bbox, area)]. 쓸 수 있는 bbox만 넣는다.
        missing (list): 라벨 누락 [(이미지, K-코드, 2-1 표의 행 이름)]

    Returns:
        dict: 이미지 → [(K-코드, 문제)]
    """
    found = collections.defaultdict(list)
    for fn, anns in sorted(anns_of.items()):
        same_box = collections.Counter(tuple(bbox) for _, bbox, _ in anns)
        for code, bbox, area in anns:
            x, y, w, h = bbox
            kinds = []
            if same_box[tuple(bbox)] > 1:
                kinds.append('다른 클래스와 box가 똑같음')
            if x < 0 or y < 0 or x + w > IMG_W or y + h > IMG_H:
                kinds.append('이미지 밖')
            if area != w * h:
                kinds.append('area 불일치')
            if kinds:
                found[fn].append((code, ' + '.join(kinds)))
    for fn, code, group in missing:
        found[fn].append((code, f'라벨 누락: {group}'))
    return found


def by_kind(found):
    """이미지 → [(K-코드, 문제)]를 문제 → [(이미지, K-코드)]로 바꾼다."""
    items = collections.defaultdict(list)
    for fn, pairs in sorted(found.items()):
        for code, kind in pairs:
            items[kind].append((fn, code))
    return items


# 2-2-1 표: 2-2-1과 같은 라벨(copy 재배정 뒤)에 함수로 다시 검사해 2-2 표의 problems와 같은지 확인한다
before_22 = table_problems({fn: [(a['category_id'], a['bbox'], a['area']) for a in r['annotations'] if usable_bbox(a['bbox'])]
                            for fn, r in aihub_kaggle.items()},
                           [(m[0], m[1], missing_group(m)) for m in missing_labels])
table_68 = {fn: sorted((code, kind) for code, kind, _, _ in items if kind != '알약을 감싸지 않는 box')  # 2-2-3이 뒤에 더한 문제는 뺀다
            for fn, items in problems.items()}
assert {fn: sorted(items) for fn, items in before_22.items()} == {fn: items for fn, items in table_68.items() if items}, \
    '함수로 다시 한 검사가 2-2-1 표와 다르다'
# 2-2 수정 뒤: 처리에 box가 있으면 그 box를 쓰고 area는 너비×높이로 다시 계산한다. 찾지 못한 알약은 라벨 누락으로 남는다 (위 label_issues와 같은 기준)
anns_22 = collections.defaultdict(list)
missing_22 = []
for fn, record in aihub_kaggle.items():
    anns_22[fn] += [(a['category_id'], a['bbox'], a['area']) for a in record['annotations'] if (fn, a['category_id']) not in fix_of]
for (fn, code), fix in fix_of.items():
    if fix['box'] and usable_bbox(fix['box']):
        anns_22[fn].append((fix['label_id'], fix['box'], fix['box'][2] * fix['box'][3]))
    elif fix['problem'].startswith('라벨 누락: '):  # 2-1의 라벨 누락이었던 알약은 원래 원인의 행에 남는다
        missing_22.append((fn, code, fix['problem'].removeprefix('라벨 누락: ')))
    else:  # 좌표 문제나 알약을 감싸지 않는 box였던 알약은 box를 뺀다
        missing_22.append((fn, code, UNFIXED))
after_22 = table_problems(anns_22, missing_22)
unfixed_from = collections.Counter(fix_of[(fn, code)]['problem'] for fn, code, group in missing_22 if group == UNFIXED)  # 원래 문제 → box를 뺀 알약 수

kinds_before, kinds_after = by_kind(before_22), by_kind(after_22)
count_68 = collections.Counter(kind for pairs in before_22.values() for _, kind in pairs)  # 2-2-1의 kind_count (2-2-3이 다시 만들기 전)
rows = []
for kind in [k for k, _ in count_68.most_common()] + sorted(set(kinds_after) - set(count_68)):
    b, a = kinds_before.get(kind, []), kinds_after.get(kind, [])
    notes = []
    if unfixed_from[kind]:
        notes.append(f'box를 찾지 못한 {unfixed_from[kind]}개는 box를 빼고 "라벨 누락: {UNFIXED}" 행에 센다')
    if kind == f'라벨 누락: {UNFIXED}':
        notes.append('원래 문제: ' + ', '.join(f'{k} {n}개' for k, n in unfixed_from.most_common()))
    if a:
        notes.append(f'예: `{a[0][0]}` K-{a[0][1]:06d} ({gallery_of[a[0][0]]})')
    rows.append([kind, f'{len(b)} → {len(a)}', f'{len({fn for fn, _ in b})} → {len({fn for fn, _ in a})}', '<br>'.join(notes)])
rows.append(['**합계**', f'**{sum(map(len, before_22.values()))} → {sum(map(len, after_22.values()))}**',
             f'**{len(before_22)}장 → {len(after_22)}장**', ''])
show_table(['문제', 'annotation 수 (2-2 표 → 2-2 수정 뒤)', '이미지 수 (2-2 표 → 2-2 수정 뒤)', '2-2 수정 뒤 비고'], rows)

#### 2-2-8 결과

- 2-2-1 표의 문제는 838개(691장)에서 21개(20장)로 줄었다. 97.5%가 풀렸다.
- 좌표 문제 행(area 불일치, box가 똑같음, 이미지 밖, 둘이 겹친 행)은 모두 0이다.
    - area 불일치는 고친 box로 area를 다시 계산하므로 0이 된다. box가 알약에 맞는지는 문제 갤러리(2-2-5)로 확인했다.
    - box를 찾지 못한 12개는 "2-2에서 고치지 못해 box를 뺀 알약" 행으로 옮겼다.
- 라벨 누락 211개 중 202개를 채웠다. bbox 빈 값 8개와 숫자가 붙은 bbox 1개가 남았다.
- 남은 21개는 2-2-7에서 남은 21개(20장)와 같다.

#### 2-2-9. 남은 이미지

- 고친 뒤에도 라벨 문제가 남은 20장을 한 그림에 모아 그린다. 2-5에서 학습 데이터에서 빼는 이미지다.
    - 그리는 방법은 문제 갤러리(2-2-5)와 같다. 제목의 #번호도 그 갤러리의 번호다.
    - 찾지 못한 알약은 파란 box가 없다. 노랑 점선은 크기·색이 가장 비슷한 후보다.
- 빠진 알약이 사진에 정말 있는지, 왜 찾지 못했는지 본다.

In [ ]:
number_of = {row['file_name']: row['number'] for row in problem_index}  # 이미지 → 2-2 문제 갤러리 번호
unfixed_files = sorted(after_22, key=number_of.get)
assert set(unfixed_files) == {f['file_name'] for f in box_fixes if f['action'] == '찾지 못함'}, '2-2 수정 뒤에 남은 이미지가 찾지 못함이 있는 이미지와 다르다'
n_cols = 4
n_rows = int(np.ceil(len(unfixed_files) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 8.75 * n_rows), layout='constrained', squeeze=False)
for ax, fn in zip(axes.flat, unfixed_files):
    problem_panel(ax, number_of[fn], fn, fixes_by_file[fn])
for ax in axes.flat[len(unfixed_files):]:
    ax.axis('off')
fig.suptitle(f'2-2 수정 뒤에도 라벨 문제가 남은 이미지 {len(unfixed_files)}장 (2-5에서 학습 데이터에서 뺌, 제목의 #번호 = 2-2 문제 갤러리 번호)\n'
             '빨간 점선 = 원본 bbox, 자홍 점선 = box 맞음을 잰 알약 범위 + 여백, 주황 점선 = 짝지은 알약 box, '
             '노랑 점선 = 짝짓지 못한 문제의 가장 비슷한 후보\n'
             '하늘 점선 = 눈으로 고치기 전 자동 처리 box, 파란 실선 = 고친 box, 연두 = 문제없는 라벨 box\n'
             '사진 아래: 문제마다 "K-코드 문제: 문제로 판단한 값"과 "→ 처리: 그렇게 정한 근거"', fontsize=14)
save_fig(fig, '2-2_unfixed_gallery')
plt.show()

#### 2-2-9 결과

그림에서 보이는 것

- 20장 모두 box가 없는 알약이 사진에 놓여 있다. `-오류` 폴더처럼 알약이 사진에 없는 경우는 없다.

| 찾지 못한 이유 | 이미지 | 내용 |
|---|---|---|
| 바닥과 색이 비슷한 파란 알약 | 16장 (알약 17개) | K-010487 6개, K-031885 6개, K-041149 3개, K-005391 1개, K-012638 1개. 2-2-2, 2-2-3에서 재는 방법이 실패한 클래스다 |
| 같은 사진의 다른 클래스가 빠진 알약을 차지함 | 4장 | #57, #130, #222는 문제가 없던 연두 box의 클래스가 틀렸다. #243은 기준값이 없는 K-005391이 다른 캡슐과 짝지어졌다 |

- 예: #57은 K-005886(TYLENOL ER)의 연두 box가 K-023223(YH V100) 알약 위에 있고, TYLENOL ER에는 box가 없다.
- 빼는 사진은 파란 클래스마다 그 클래스 촬영 PNG의 1.3% 이하다. (K-031885 459장 중 6장, K-010487 540장 중 6장)

20장은 2-5에서 학습 데이터에서 뺀다. 파란 알약 16장은 손으로 box를 그려야 살릴 수 있고, 나머지 4장은 문제가 없던 라벨의 클래스까지 틀려서 빼는 편이 안전하다.

### 2-3. 크기·색 대조와 crop 갤러리

| 단계 | 하는 일 |
|---|---|
| 2-3-1 | 크기·색으로 클래스를 다시 배정해 라벨과 다른 이미지를 찾고, 시트로 본다 |
| 2-3-2 | 모든 box를 클래스별로 잘라 늘어놓고(crop 갤러리) 눈으로 본다 |

- 2-2의 처리를 적용한 라벨(`fixed_records`)로 본다. 사진에서 알약을 찾지 못한 이미지는 뺀다.

#### 2-3-1. 크기·색 대조

- 이미지마다 box 안의 알약 범위와 색(2-2-3에서 잰 값)이 2-2-2의 클래스 기준값과 가장 잘 맞도록 클래스를 다시 배정한다.
- 다시 배정한 클래스가 라벨과 다르면, 라벨이 서로 바뀌었거나 box가 다른 자리에 있는 후보다.

In [ ]:
def drawn_annotation(fn, code, box):
    """원본 annotation이 없던 알약의 annotation을 만든다. 알약 메타데이터는 같은 클래스의 다른 annotation에서,
    이미지 필드(파일명, 촬영 각도 등)는 그 이미지의 것으로 채운다. (0-6의 대회 train 정제와 같은 방법)"""
    example = aihub_example[code]
    image_fields = (aihub_kaggle[fn]['image'] if fn in aihub_kaggle
                    else {'file_name': fn, 'imgfile': fn, 'camera_la': int(fn.split('_')[5]), 'id': None})
    image = {**example['image'], **{k: image_fields[k] for k in IMAGE_FIELDS if k in image_fields}}
    return {**example, 'id': None, 'image_id': image.get('id'), 'json_path': None, 'bbox': list(box), 'image': image}


def fixed_annotation(fn, ann, fix):
    """2-2의 처리를 annotation에 적용한다. 원본 annotation이 없거나 사진 속 클래스가 다르면(-오류 폴더) 그 클래스의 메타데이터로 만든다."""
    box = list(fix['box'])
    if ann is None or fix['label_id'] != ann['category_id']:
        return drawn_annotation(fn, fix['label_id'], box)
    return {**ann, 'bbox': box, 'area': box[2] * box[3]}


fixed_records = {}  # 2-2 수정 뒤 라벨: 이미지 → {'image', 'annotations'}
fix_lookup = collections.defaultdict(dict)  # 이미지 → K-코드 → 2-2의 처리
for fix in box_fixes:
    fix_lookup[fix['file_name']][fix['category_id']] = fix
unfound_files = {f['file_name'] for f in box_fixes if f['action'] == '찾지 못함'}
for fn in aihub_files:
    if fn in unfound_files:
        continue
    fixes = fix_lookup[fn]
    anns = aihub_kaggle.get(fn, {'annotations': []})['annotations']
    # 1-3의 annotation 순서를 지켜야 2-2에서 잰 알약 범위(캐시)를 다시 쓴다. 원본 annotation이 없던 알약은 끝에 붙인다
    fixed = [fixed_annotation(fn, a, fixes[a['category_id']]) if a['category_id'] in fixes else a for a in anns]
    fixed += [fixed_annotation(fn, None, fix) for code, fix in fixes.items() if code not in {a['category_id'] for a in anns}]
    assert all(usable_bbox(a['bbox']) for a in fixed), f'{fn}: 2-2 수정 뒤에도 쓸 수 없는 bbox가 있다'
    fixed_records[fn] = {'image': fixed[0]['image'], 'annotations': fixed}


fixed_measures = measure_all(fixed_records)  # 2-2에서 잰 값을 다시 쓰고, 고친 box만 새로 잰다


def appearance_check(fn):
    """크기·색으로 다시 배정한 클래스 순서와 (라벨 그대로의 비용, 가장 맞는 배정의 비용)을 구한다. 대조할 수 없으면 None이다."""
    anns = fixed_records[fn]['annotations']
    codes = [a['category_id'] for a in anns]
    measures = [fixed_measures[(fn, tuple(a['bbox']))] for a in anns]
    if len(anns) < 2 or any(c not in class_refs for c in codes) or any(m[0] is None for m in measures):
        return None
    cost = np.array([[appearance_cost(extent, lab, class_refs[c]) for c in codes] for extent, lab in measures])
    _, order = scipy.optimize.linear_sum_assignment(cost)
    return {'order': [int(j) for j in order], 'label_cost': float(np.trace(cost)), 'best_cost': float(cost[np.arange(len(codes)), order].sum())}


appearance = {fn: appearance_check(fn) for fn in sorted(fixed_records)}
checked = {fn: r for fn, r in appearance.items() if r}
mismatched = sorted((fn for fn, r in checked.items() if r['order'] != list(range(len(r['order'])))),
                    key=lambda fn: -(checked[fn]['label_cost'] - checked[fn]['best_cost']))
show_table(['항목', '이미지 수'], [
    ['크기·색 대조를 한 이미지', f'{len(checked):,}장'],
    ['대조하지 못한 이미지 (알약 1개 이하, 기준값이 없는 클래스, 알약 범위를 못 찾은 box)', f'{len(appearance) - len(checked):,}장'],
    ['다시 배정한 클래스가 라벨과 다른 이미지', f'{len(mismatched):,}장 ({len(mismatched) / len(checked):.2%})'],
])

- 라벨과 다른 이미지를 모두 그려 시트로 저장한다. 라벨 그대로의 비용과 가장 맞는 배정의 비용 차이가 큰 순서다.
    - 연두 box가 라벨이다. box 옆에 메타데이터의 K-코드, 각인, 색, 모양을 적는다.
    - 다르게 배정된 box는 빨간 글자로 `before`(라벨)와 `after`(다시 배정한 클래스)를 함께 적는다.
    - K-016548은 각인, 색, 모양 메타데이터가 비어 있어 "메타데이터 없음"으로 적는다.
- 시트는 `runs/eda_raewoo/figures_01/2-3_appearance_sheets/`에 8장씩 저장하고, 아래에는 첫 시트만 보인다.

In [ ]:
def pill_text(ann):
    """K-코드, 각인, 색, 모양을 한 줄로 적는다. (메타데이터 print_front/print_back, color_class1/2, drug_shape)"""
    info = ann['image']
    marks = '/'.join(info[key] for key in ('print_front', 'print_back') if info.get(key))
    colors = '/'.join(info[key] for key in ('color_class1', 'color_class2') if info.get(key))
    if not (marks or colors or info.get('drug_shape')):  # K-016548은 AI Hub와 대회 데이터 모두 메타데이터가 비어 있다
        return f"K-{ann['category_id']:06d}, 메타데이터 없음 ({info.get('dl_name')})"
    return f"K-{ann['category_id']:06d}, {marks or '각인 없음'}, {colors}, {info.get('drug_shape')}"


def place_label(ax, box, text, color, taken, others):
    """글자를 box 위, 아래, 안쪽 위 중 사진 안이면서 다른 글자, 다른 box와 겹치지 않는 첫 자리에 적는다.
    그런 자리가 없으면 글자끼리만 겹치지 않는 자리, 그것도 없으면 첫 자리에 적는다."""
    renderer = ax.figure.canvas.get_renderer()
    frame = ax.get_window_extent(renderer)
    x, y, w, h = box
    near = 'right' if x + w / 2 > IMG_W / 2 else 'left'  # 오른쪽 알약은 글자를 box 오른쪽 끝에 맞춘다
    far = 'left' if near == 'right' else 'right'
    spots = [(row, ha) for row in [(y - 6, 'bottom'), (y + h + 6, 'top'), (y + 6, 'top')] for ha in (near, far)]
    tried = []
    for (text_y, va), ha in spots:
        label = ax.text(x if ha == 'left' else x + w, text_y, text, fontsize=9, va=va, ha=ha, color=color,
                        bbox=dict(facecolor='white', alpha=0.85, pad=1.5, linewidth=0))
        extent = label.get_window_extent(renderer).padded(3)
        inside = frame.x0 <= extent.x0 and extent.x1 <= frame.x1 and frame.y0 <= extent.y0 and extent.y1 <= frame.y1
        free = not any(extent.overlaps(t) for t in taken)
        tried.append((label, extent, inside and free, inside and free and not any(extent.overlaps(o) for o in others)))
        label.set_visible(False)
    label, extent, _, _ = next((t for t in tried if t[3]), None) or next((t for t in tried if t[2]), tried[0])
    for other, *_ in tried:
        if other is not label:
            other.remove()
    label.set_visible(True)
    taken.append(extent)


APPEAR_DIR = FIG_DIR / '2-3_appearance_sheets'
APPEAR_DIR.mkdir(exist_ok=True)
for old in APPEAR_DIR.glob('*.jpg'):
    old.unlink()
for start in range(0, len(mismatched), 8):
    chunk = mismatched[start:start + 8]
    fig, axes = plt.subplots(2, 4, figsize=(24, 17), layout='constrained')
    patches = {}
    for ax, fn in zip(axes.flat, chunk):
        ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
        for i, ann in enumerate(fixed_records[fn]['annotations']):
            x, y, w, h = ann['bbox']
            patches[(fn, i)] = ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
        n = mismatched.index(fn) + 1
        ax.set_title(f"#{n} {fn}\n비용: 라벨 {checked[fn]['label_cost']:.2f} → 다시 배정 {checked[fn]['best_cost']:.2f}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in axes.flat[len(chunk):]:
        ax.axis('off')
    fig.suptitle(f'크기·색으로 다시 배정한 클래스가 라벨과 다른 이미지 {start + 1}~{start + len(chunk)} / {len(mismatched)}장 '
                 '(연두 = 라벨 box, 글자 = K-코드, 각인, 색, 모양, 빨간 글자 = 다르게 배정된 box의 before 라벨 → after 크기·색으로 다시 배정한 클래스)',
                 fontsize=14)
    fig.draw_without_rendering()  # 배치를 먼저 정하고 고정해야 글자 자리를 잴 수 있다
    fig.set_layout_engine('none')
    renderer = fig.canvas.get_renderer()
    for ax, fn in zip(axes.flat, chunk):
        anns = fixed_records[fn]['annotations']
        order = checked[fn]['order']
        taken = []
        for i in sorted(range(len(anns)), key=lambda i: order[i] == i):  # 다르게 배정된 box(긴 글자)부터 자리를 잡는다
            changed = order[i] != i
            text = f"before: {pill_text(anns[i])}\n→ after: {pill_text(anns[order[i]])}" if changed else pill_text(anns[i])
            others = [patches[(fn, k)].get_window_extent(renderer) for k in range(len(anns)) if k != i]
            place_label(ax, anns[i]['bbox'], text, 'red' if changed else 'black', taken, others)
    fig.savefig(APPEAR_DIR / f'2-3_appearance_p{start // 8 + 1:02d}.jpg', dpi=90)
    if start:
        plt.close(fig)
plt.show()

#### 2-3-1 결과

| 항목 | 이미지 |
|---|---|
| 크기·색 대조를 한 이미지 | 10,589장 |
| 대조하지 못한 이미지 (알약 1개 이하, 기준값이 없는 클래스, 범위를 못 잰 box) | 1,400장 |
| 다시 배정한 클래스가 라벨과 다른 이미지 | 70장 (0.66%) |

그림에서 보이는 것 (첫 시트 8장)

- 5장은 같은 사진 안에서 두 알약의 라벨이 서로 바뀌었다.
    - #1, #2: K-019881(Z분할선Z, 하양 장방형)과 K-006835(CKD, 주황 원형)
    - #4: K-029667과 K-030308, #5: K-016262와 K-024752, #6: K-044266(BTS 330)과 K-013004(Igatan F)
- #3, #7, #8(조합 `K-016548-016551-023203-024850`)은 사진에 K-016551(DGTH)이 없고 Airtal 알약(K-004543)이 있다. 크기·색 배정도 틀렸다.
    - 이 조합은 1-6의 같은 장면이라 2-4에서 뺀다.
- 라벨이 서로 바뀐 #1, #2는 비용이 8.0~8.2에서 0.7~1.1로 크게 줄었다.

70장을 모두 시트로 보고, 라벨이 정말 틀린 것만 2-4의 `LABEL_FINDINGS`에 적었다. 크기·색 배정도 틀릴 수 있어서(#3) 결과를 그대로 쓰지 않는다.

#### 2-3-2. crop 갤러리

- 모든 box를 클래스별로 잘라 시트에 늘어놓는다.
    - crop은 box 중심에서 긴 변보다 10% 큰 정사각형을 잘라 96×96으로 줄인 것이다. 노란 선이 라벨 box다.
    - 클래스마다 box 넓이 순서로 놓는다. 크기가 튀는 box(헐거운 box, 옆 알약까지 감싼 box)는 처음이나 끝에 모인다.
    - 대회 train에도 있는 클래스는 첫 시트 첫 줄(빨간 테두리)에 대회 crop을 놓아 같은 알약인지 대조한다.
- 시트 한 장은 10×10 = 100칸이다. `runs/eda_raewoo/figures_01/2-3_gallery/`에 `K-<코드>_p<쪽>.jpg`로 저장하고, 칸 번호로 이미지를 찾는 색인은 같은 폴더의 `index.csv`다.
- 시트는 Claude Code가 모두 보고 1·2차 확인을 했고, 애매한 칸은 사람이 확인했다.

In [ ]:
GALLERY_DIR = FIG_DIR / '2-3_gallery'
GALLERY_DIR.mkdir(exist_ok=True)
THUMB, COLS, ROWS = 96, 10, 10
gallery_items = collections.defaultdict(list)  # K-코드 → [(이미지, bbox)] box 넓이 순서
for fn, record in fixed_records.items():
    for ann in record['annotations']:
        gallery_items[ann['category_id']].append((fn, ann['bbox']))
for code in gallery_items:
    gallery_items[code].sort(key=lambda item: (item[1][2] * item[1][3], item[0]))
gallery_key = hashlib.md5(json.dumps(sorted(gallery_items.items())).encode()).hexdigest()
index_path = GALLERY_DIR / 'index.csv'
font_path = font_manager.findfont(plt.rcParams['font.family'][0])


def thumb_bytes(job):
    """이미지 하나에서 crop을 모두 잘라 JPEG 바이트로 돌려준다. (메모리를 아끼려고 압축해 둔다)"""
    folder, fn, boxes = job
    image = Image.open(folder / fn).convert('RGB')
    out = {}
    for box in boxes:
        x, y, w, h = box
        side = max(w, h) * 1.1  # crop_square와 같은 정사각형 (긴 변 + 10%)
        scale = THUMB / side
        thumb = crop_square(image, box, size=THUMB)
        left, top = THUMB / 2 - w * scale / 2, THUMB / 2 - h * scale / 2  # box는 정사각형 가운데에 온다
        ImageDraw.Draw(thumb).rectangle([left, top, left + w * scale, top + h * scale], outline=(255, 230, 0), width=1)
        buffer = io.BytesIO()
        thumb.save(buffer, format='JPEG', quality=90)
        out[(fn, tuple(box))] = buffer.getvalue()
    return out


def class_header(code, n_items, page, n_pages):
    info = (aihub_example.get(code) or class_example[code])['image']
    marks = '/'.join(m for m in (info.get('print_front'), info.get('print_back')) if m) or '각인 없음'
    return (f"K-{code:06d} {info['dl_name'][:28]} | 각인 {marks} | {info.get('color_class1')} {info.get('drug_shape')} | "
            f"AI Hub {n_items}개 | {page}/{n_pages}쪽")


cached = index_path.exists() and (GALLERY_DIR / 'key.txt').exists() and (GALLERY_DIR / 'key.txt').read_text() == gallery_key
if not cached:
    jobs = collections.defaultdict(list)
    for code, items in gallery_items.items():
        for fn, box in items:
            jobs[(AIHUB_IMG_DIR, fn)].append(box)
    for code in set(gallery_items) & set(train_classes):  # 대회 train crop (정제 라벨)
        for fn, record in clean_records.items():
            for ann in record['annotations']:
                if ann['category_id'] == code:
                    jobs[(TRAIN_IMG_DIR, fn)].append(ann['bbox'])
    thumbs_jpeg = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        for out in pool.map(thumb_bytes, [(folder, fn, boxes) for (folder, fn), boxes in jobs.items()]):
            thumbs_jpeg.update(out)
    font = ImageFont.truetype(font_path, 13)
    small_font = ImageFont.truetype(font_path, 11)
    index_rows = []
    for old in GALLERY_DIR.glob('K-*.jpg'):
        old.unlink()
    for code, items in sorted(gallery_items.items()):
        kaggle = [(fn, a['bbox']) for fn, r in sorted(clean_records.items()) for a in r['annotations'] if a['category_id'] == code][:COLS]
        first_capacity = COLS * (ROWS - 1) if kaggle else COLS * ROWS
        pages = [items[:first_capacity]] + [items[i:i + COLS * ROWS] for i in range(first_capacity, len(items), COLS * ROWS)]
        for p, page_items in enumerate(pages, start=1):
            sheet = Image.new('RGB', (COLS * THUMB, 24 + ROWS * (THUMB + 14)), 'white')
            draw = ImageDraw.Draw(sheet)
            draw.text((4, 4), class_header(code, len(items), p, len(pages)), fill='black', font=font)
            first_row = kaggle if p == 1 else []  # 첫 시트 첫 줄은 대회 train crop
            start_no = 0 if p == 1 else first_capacity + (p - 2) * COLS * ROWS
            for col, (fn, box) in enumerate(first_row):
                x0, y0 = col * THUMB, 24
                sheet.paste(Image.open(io.BytesIO(thumbs_jpeg[(fn, tuple(box))])), (x0, y0))
                draw.rectangle([x0, y0, x0 + THUMB - 1, y0 + THUMB - 1], outline='red', width=3)
                draw.text((x0 + 2, y0 + THUMB), '대회', fill='red', font=small_font)
            for k, (fn, box) in enumerate(page_items):
                row, col = divmod(k + (COLS if first_row else 0), COLS)
                x0, y0 = col * THUMB, 24 + row * (THUMB + 14)
                sheet.paste(Image.open(io.BytesIO(thumbs_jpeg[(fn, tuple(box))])), (x0, y0))
                number = start_no + k + 1
                draw.text((x0 + 2, y0 + THUMB), f'{number}', fill='black', font=small_font)
                index_rows.append({'sheet': f'K-{code:06d}_p{p:02d}.jpg', 'number': number, 'category_id': code,
                                   'file_name': fn, 'bbox': json.dumps(box)})
            sheet.save(GALLERY_DIR / f'K-{code:06d}_p{p:02d}.jpg', quality=85)
    with open(index_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['sheet', 'number', 'category_id', 'file_name', 'bbox'])
        writer.writeheader()
        writer.writerows(index_rows)
    (GALLERY_DIR / 'key.txt').write_text(gallery_key)
with open(index_path, encoding='utf-8') as f:
    gallery_index = list(csv.DictReader(f))
n_sheets = len(list(GALLERY_DIR.glob('K-*.jpg')))
show_table(['항목', '값'], [
    ['crop 수 (AI Hub, 2-2 수정 후)', f"{len(gallery_index):,}개 (이미지 {len(fixed_records):,}장)"],
    ['클래스 수', f'{len(gallery_items)}개 (대회 train crop 줄이 있는 클래스 {len(set(gallery_items) & set(train_classes))}개)'],
    ['시트 수', f'{n_sheets}장 (클래스당 {min(len(v) for v in gallery_items.values()) // 100 + 1}~{max(len(v) for v in gallery_items.values()) // 100 + 1}장)'],
    ['저장 위치', f'`{GALLERY_DIR.relative_to(PROJECT_ROOT)}`'],
])

- 시트 예시로 대회 train에도 있는 K-001900(보령부스파정 5mg)의 첫 시트를 보인다. (1-2-2의 예시 알약)

In [ ]:
example_sheet = Image.open(GALLERY_DIR / 'K-001900_p01.jpg')
fig, ax = plt.subplots(figsize=(14, 14 * example_sheet.height / example_sheet.width), layout='constrained')
ax.imshow(example_sheet)
ax.axis('off')
ax.set_title('crop 갤러리 시트 예시 (첫 줄 빨간 테두리 = 대회 train crop, 아래 = AI Hub crop을 box 넓이 순서로, 칸 아래 숫자 = 색인 번호)', fontsize=12)
save_fig(fig, '2-3_gallery_example')
plt.show()

#### 2-3-2 결과

- crop 45,879개(이미지 11,989장, 116개 클래스)를 시트 512장에 늘어놓았다. 클래스당 1~14장이다.
- 대회 train과 같은 클래스 54개는 첫 줄에 대회 crop이 있다. drug_N으로 매긴 클래스가 대회의 같은 알약인지 바로 대조할 수 있다.

그림에서 보이는 것 (K-001900 첫 시트)

- AI Hub crop 90개는 모두 BSP 각인의 흰 장방형 알약이고, 첫 줄의 대회 crop과 같은 알약이다.
- 노란 box가 알약을 빈틈없이 감싼다. 넓이 순서의 처음과 끝에도 헐거운 box가 보이지 않는다.

시트에서 찾은 잘못된 클래스와 틀린 box를 2-4의 `LABEL_FINDINGS`에 적어 고친다. (클래스 바꿈 47개, box 다시 그림 17개)

### 2-4. 정제 목록

| 단계 | 하는 일 |
|---|---|
| 2-4-1 | 1·2절에서 찾은 오류의 처리를 annotation 한 줄씩 모은다 |
| 2-4-2 | 문제로 제기된 것이 어떻게 해소되었는지 한 표로 본다 |

- 정제 목록은 메모리(`aihub_fix_rows`)에만 두고 파일로 쓰지 않는다.
- 학습 데이터는 레포의 `scripts/prepare_train_data.py`가 만든다.
    - 이 정제 목록은 `data/label_fix/label_fixes.csv`의 aihub 줄로 옮겼다.
    - 그 줄로 만든 학습 데이터의 JSON이 이 노트북의 결과(2-7)와 같은지는 `tests/test_train_dataset.py`가 확인한다. (기준값은 `tests/helpers.py`의 `AIHUB_EXPORT_HASH`, annotation id만 8 차이)

#### 2-4-1. 정제 목록

| 근거 | 내용 |
|---|---|
| 2-1 | copy 파일 다시 배정 (2-1-3). 같은 알약의 2-2 줄보다 앞에 둔다 |
| 2-1, 2-2 | 2-2-4의 처리 결과. 눈으로 고친 box와, `-오류` 폴더 알약을 실제 클래스로 붙인 것도 들어간다 |
| 2-3 | 크기·색 시트와 crop 갤러리에서 눈으로 찾은 오류 (`LABEL_FINDINGS`) |
| 1-6 | 같은 장면 6쌍에서 이름이 틀린 쪽 6장을 뺀다 (`SAME_SCENE_COMBOS`) |
| 2-6 | 고친 클래스 구성이 이름이 맞는 다른 조합과 같은 3개 조합 9장을 뺀다 (`SAME_SET_COMBOS`) |
| 1-4 | "검은색 배경" 1장의 back_color를 연회색 배경으로 고친다 (`META_FINDINGS`) |

- 2-3에서 찾은 오류는 세 종류다.
    - 같은 사진 안에서 두 알약의 라벨이 바뀜: 두 라벨을 맞바꾼다.
    - 사진 속 알약이 파일명에 없는 약: 그 약의 클래스로 바꾼다.
    - box가 틀림(다른 자리, 너무 큼, 덜 감쌈): 사진에서 잰 알약 범위에 클래스 여백을 더해 다시 그리고, 눈으로 확인한 값을 적는다.
- 이미지를 빼는 줄은 사진에 없는 알약의 줄 하나로 적는다.
- 원래 파일명은 PNG, JSON, 정제 목록을 잇는 이름이라 바꾸지 않는다.
- 바뀌는 annotation을 단계마다 한 줄로 적고, 목록 순서대로 적용한다. 같은 annotation을 두 단계에서 고치면 두 줄이다.

| 열 | 뜻 |
|---|---|
| `category_id` | 1-3의 K-코드(파일명의 K-코드). 같은 알약의 줄은 모두 이 값으로 찾는다 |
| `bbox_before`, `bbox_after` | 원본 JSON의 bbox 그대로, 고친 `[x, y, w, h]`. 이미지를 빼는 줄은 `bbox_after`가 비어 있다 |
| `area_before`, `area_after` | 원본 area, 너비×높이로 다시 계산한 area |
| `category_id_after` | 클래스를 바꾸는 줄에만 적는다 |
| `evidence` | 근거가 된 절 번호 |

In [ ]:
# 2-3 crop 갤러리를 눈으로 확인해 찾은 라벨 오류 (주석 = 2-3 갤러리의 시트와 칸 번호)
# (이미지, K-코드) → (문제, 바꿀 K-코드 또는 None, 다시 그린 box [x, y, w, h] 또는 None(box 그대로))
LABEL_FINDINGS = {
    # 같은 사진 안에서 라벨이 서로 바뀜
    ('K-001900-003544-006563-016548_0_2_0_2_90_000_200.png', 3544): ('같은 사진의 K-001900 라벨과 바뀜 (BSP 알약)', 1900, None),  # K-003544_p01 #1
    ('K-001900-003544-006563-016548_0_2_0_2_75_000_200.png', 3544): ('같은 사진의 K-001900 라벨과 바뀜 (BSP 알약)', 1900, None),  # K-003544_p01 #2
    ('K-001900-003544-006563-016548_0_2_0_2_90_000_200.png', 1900): ('같은 사진의 K-003544 라벨과 바뀜 (OG3 알약, 피드백의 짝)', 3544, None),  # K-001900_p02 #162 (피드백 짝)
    ('K-001900-003544-006563-016548_0_2_0_2_75_000_200.png', 1900): ('같은 사진의 K-003544 라벨과 바뀜 (OG3 알약, 피드백의 짝)', 3544, None),  # K-001900_p02 #161 (피드백 짝)
    ('K-006835-019881-020004-023319_0_2_0_2_75_000_200.png', 6835): ('같은 사진의 K-019881 라벨과 바뀜 (Z|Z 알약)', 19881, None),  # K-006835_p01 #1
    ('K-006835-019881-020004-023319_0_2_0_2_70_000_200.png', 6835): ('같은 사진의 K-019881 라벨과 바뀜 (Z|Z 알약)', 19881, None),  # K-006835_p01 #2
    ('K-006835-019881-020004-023319_0_2_0_2_75_000_200.png', 19881): ('같은 사진의 K-006835 라벨과 바뀜 (CKD 알약)', 6835, None),  # K-019881_p04 #359
    ('K-006835-019881-020004-023319_0_2_0_2_70_000_200.png', 19881): ('같은 사진의 K-006835 라벨과 바뀜 (CKD 알약)', 6835, None),  # K-019881_p04 #360
    ('K-013004-026993-038972-044266_0_2_0_2_75_000_200.png', 13004): ('같은 사진의 K-044266 라벨과 바뀜 (BTS 330 캡슐)', 44266, None),  # K-013004_p04 #344
    ('K-013004-026993-038972-044266_0_2_0_2_75_000_200.png', 44266): ('같은 사진의 K-013004 라벨과 바뀜 (이가탄 캡슐)', 13004, None),  # K-044266_p01 #74
    ('K-011220-016262-022347-025469_0_2_0_2_75_000_200.png', 16262): ('같은 사진의 K-022347 라벨과 바뀜 (112 자누비아정)', 22347, None),  # K-016262_p01 #1
    ('K-011220-016262-022347-025469_0_2_0_2_75_000_200.png', 22347): ('같은 사진의 K-016262 라벨과 바뀜 (ZD4522 크레스토정)', 16262, None),  # K-022347_p05 #468
    ('K-003832-016262-024752_0_2_0_2_75_000_200.png', 16262): ('같은 사진의 K-024752 라벨과 바뀜 (FS 알약)', 24752, None),  # K-016262_p09 #795
    ('K-003832-016262-024752_0_2_0_2_75_000_200.png', 24752): ('같은 사진의 K-016262 라벨과 바뀜 (ZD4522 크레스토정)', 16262, None),  # K-024752_p01 #5
    ('K-016206-016262-020014_0_2_0_2_75_000_200.png', 16262): ('같은 사진의 K-020014 라벨과 바뀜 (마도파정)', 20014, None),  # K-016262_p09 #796
    ('K-016206-016262-020014_0_2_0_2_75_000_200.png', 20014): ('같은 사진의 K-016262 라벨과 바뀜 (ZD4522 크레스토정)', 16262, None),  # K-020014_p01 #2
    ('K-016206-020014-020238_0_2_0_2_70_000_200.png', 20014): ('같은 사진의 K-020238 라벨과 바뀜 (75 플라빅스정)', 20238, None),  # K-020014_p01 #1
    ('K-016206-020014-020238_0_2_0_2_70_000_200.png', 20238): ('같은 사진의 K-020014 라벨과 바뀜 (마도파정)', 20014, None),  # K-020238_p07 #620
    ('K-016206-020238-032310_0_2_0_2_70_000_200.png', 20238): ('같은 사진의 K-032310 라벨과 바뀜 (글리아타민 연질캡슐)', 32310, None),  # K-020238_p07 #619
    ('K-016206-020238-032310_0_2_0_2_70_000_200.png', 32310): ('같은 사진의 K-020238 라벨과 바뀜 (75 플라빅스정)', 20238, None),  # K-032310_p01 #1
    ('K-006835-024941-030850-053384_0_2_0_2_70_000_200.png', 24941): ('같은 사진의 K-053384 라벨과 바뀜 (CT/LC 디카테오엘씨정)', 53384, None),  # K-024941_p04 #359
    ('K-006835-024941-030850-053384_0_2_0_2_70_000_200.png', 53384): ('같은 사진의 K-024941 라벨과 바뀜 (TG|L 액티프롤정)', 24941, None),  # K-053384_p01 #1
    ('K-016235-025367-027777-029667_0_2_0_2_90_000_200.png', 27777): ('같은 사진의 K-029667 라벨과 바뀜 (Kowa 203 리바로정)', 29667, None),  # K-027777_p05 #467
    ('K-016235-025367-027777-029667_0_2_0_2_90_000_200.png', 29667): ('같은 사진의 K-027777 라벨과 바뀜 (FMS6 카나브정)', 27777, [143, 811, 180, 211]),  # K-029667_p01 #1
    ('K-003614-027733-029667-030308_0_2_0_2_70_000_200.png', 29667): ('같은 사진의 K-030308 라벨과 바뀜 (D2/850 트라젠타듀오정)', 30308, None),  # K-029667_p07 #611
    ('K-003614-027733-029667-030308_0_2_0_2_70_000_200.png', 30308): ('같은 사진의 K-029667 라벨과 바뀜 (Kowa 203 리바로정)', 29667, None),  # K-030308_p01 #1
    # 사진 속 알약이 파일명에 없는 약 (그 약의 클래스로 바꾼다)
    ('K-016548-016551-023203-024850_0_2_0_2_75_000_200.png', 16548): ('사진 속 알약이 K-004543(Airtal 에어탈정)', 4543, None),  # K-016548_p01 #1
    ('K-016548-016551-023203-024850_0_2_0_2_90_000_200.png', 16548): ('사진 속 알약이 K-004543(Airtal 에어탈정)', 4543, None),  # K-016548_p01 #2
    ('K-016548-016551-023203-024850_0_2_0_2_70_000_200.png', 16548): ('사진 속 알약이 K-004543(Airtal 에어탈정)', 4543, None),  # K-016548_p01 #3
    ('K-016548-016551-023203-024850_0_2_0_2_75_000_200.png', 16551): ('사진 속 알약이 K-016548(GB 가바토파정)', 16548, None),  # K-016551_p01 #1
    ('K-016548-016551-023203-024850_0_2_0_2_90_000_200.png', 16551): ('사진 속 알약이 K-016548(GB 가바토파정)', 16548, None),  # K-016551_p01 #2
    ('K-016548-016551-023203-024850_0_2_0_2_70_000_200.png', 16551): ('사진 속 알약이 K-016548(GB 가바토파정)', 16548, None),  # K-016551_p01 #3
    ('K-011220-016232-027733-031885_0_2_0_2_75_000_200.png', 16232): ('사진 속 알약이 K-029667(Kowa 203 리바로정)', 29667, None),  # K-016232_p09 #802
    ('K-011220-016232-027733-031885_0_2_0_2_90_000_200.png', 16232): ('사진 속 알약이 K-029667(Kowa 203 리바로정)', 29667, None),  # K-016232_p09 #803
    ('K-011220-016232-027733-031885_0_2_0_2_70_000_200.png', 16232): ('사진 속 알약이 K-029667(Kowa 203 리바로정)', 29667, None),  # K-016232_p09 #804
    ('K-016206-018147-022074_0_2_0_2_70_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #173
    ('K-016206-018147-022074_0_2_0_2_90_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #175
    ('K-016206-018147-022074_0_2_0_2_75_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #177
    ('K-015280-018147-022074_0_2_0_2_70_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #172
    ('K-015280-018147-022074_0_2_0_2_90_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #174
    ('K-015280-018147-022074_0_2_0_2_75_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #176 (Claude가 찾음)
    ('K-003614-027777-028763-030308_0_2_0_2_75_000_200.png', 30308): ('사진 속 알약이 K-016262(ZD4522 크레스토정)', 16262, None),  # K-030308_p01 #2
    ('K-003614-027777-028763-030308_0_2_0_2_90_000_200.png', 30308): ('사진 속 알약이 K-016262(ZD4522 크레스토정)', 16262, None),  # K-030308_p01 #3
    ('K-003614-027777-028763-030308_0_2_0_2_70_000_200.png', 30308): ('사진 속 알약이 K-016262(ZD4522 크레스토정)', 16262, None),  # K-030308_p01 #4
    ('K-015280-016688-032310_0_2_0_2_75_000_200.png', 15280): ('사진 속 알약이 K-016206(GX CE2 아보다트연질캡슐)', 16206, None),  # K-015280_p03 #280
    ('K-015280-016688-032310_0_2_0_2_90_000_200.png', 15280): ('사진 속 알약이 K-016206(GX CE2 아보다트연질캡슐)', 16206, None),  # K-015280_p03 #293
    ('K-015280-016688-032310_0_2_0_2_70_000_200.png', 15280): ('사진 속 알약이 K-016206(GX CE2 아보다트연질캡슐)', 16206, None),  # K-015280_p03 #294 (Claude가 찾음)
    # box가 틀림 (사진에서 알약 범위를 재고 클래스 여백을 더해 다시 그린다)
    ('K-003614-027777-029667-030308_0_2_0_2_70_000_200.png', 3614): ('box 너비가 모자람', None, [195, 295, 164, 162]),  # K-003614_p01 #1
    ('K-003614-027777-029667-030308_0_2_0_2_70_000_200.png', 27777): ('box가 바닥을 가리킴 (알약은 오른쪽 위의 FMS6)', None, [632, 289, 175, 201]),  # K-027777_p01 #17
    ('K-006697-006835-026993-043233_0_2_0_2_75_000_200.png', 6697): ('box 너비가 모자람', None, [573, 728, 232, 210]),  # K-006697_p03 #283
    ('K-010221-016551-019607-033009_0_2_0_2_90_000_200.png', 10221): ('box가 왼쪽으로 쏠림', None, [177, 218, 222, 215]),  # K-010221_p02 #105
    ('K-010487-026455-044266-053384_0_2_0_2_75_000_200.png', 10487): ('box가 왼쪽으로 쏠림', None, [532, 33, 265, 417]),  # K-010487_p02 #164
    ('K-010487-020004-026455-026993_0_2_0_2_70_000_200.png', 10487): ('box가 다른 알약을 가리킴 (알약은 왼쪽 아래의 파란 캡슐)', None, [88, 712, 388, 385]),  # K-010487_p06 #532
    ('K-012638-024941-029711-043233_0_2_0_2_75_000_200.png', 12638): ('box가 다른 알약을 가리킴 (알약은 오른쪽 위의 파란 캡슐)', None, [577, 48, 256, 428]),  # K-012638_p03 #210
    ('K-012638-030850-035659-053384_0_2_0_2_70_000_200.png', 12638): ('box가 다른 알약을 가리킴 (알약은 왼쪽 아래의 파란 캡슐)', None, [54, 648, 270, 444]),  # K-012638_p03 #240
    ('K-005000-016262-032310_0_2_0_2_70_000_200.png', 16262): ('box가 왼쪽으로 밀림', None, [653, 213, 239, 229]),  # K-016262_p02 #117
    ('K-001900-004543-010224-016548_0_2_0_2_75_000_200.png', 16548): ('box 너비, 높이가 알약에 맞지 않음', None, [595, 778, 235, 230]),  # K-016548_p14 #1352
    ('K-012420-016548-018110-029451_0_2_0_2_70_000_200.png', 16548): ('box 너비, 높이가 알약에 맞지 않음', None, [169, 784, 232, 231]),  # K-016548_p14 #1353
    ('K-012247-016551-031705-033208_0_2_0_2_90_000_200.png', 16551): ('box 너비가 모자람', None, [636, 120, 285, 476]),  # K-016551_p01 #4
    ('K-000573-022627-025438-038954_0_2_0_2_75_000_200.png', 22627): ('box가 알약보다 너무 큼', None, [588, 79, 309, 470]),  # K-022627_p05 #438
    ('K-013004-024941-026993-041149_0_2_0_2_90_000_200.png', 24941): ('box 너비가 모자라고 오른쪽으로 쏠림', None, [635, 823, 169, 197]),  # K-024941_p01 #1
    ('K-013004-024941-026993-041149_0_2_0_2_90_000_200.png', 41149): ('box가 오른쪽으로 쏠림 (캡슐 왼쪽 끝이 빠지고 K-024941 알약을 덮음, 같은 사진에서 찾음)', None, [9, 651, 564, 406]),  # K-041149_p03 #291 (Claude가 찾음)
    ('K-016232-016235-027653-031885_0_2_0_2_70_000_200.png', 31885): ('box가 알약을 감싸지 못함 (위로 쏠림)', None, [627, 638, 317, 527]),  # K-031885_p02 #117
}
label_at = {(fn, a['category_id']): a for fn, r in fixed_records.items() for a in r['annotations']}  # 2-2 수정 뒤 라벨 (2-3)
for (fn, code), (problem, new_code, box) in LABEL_FINDINGS.items():
    assert (fn, code) in label_at and code in pill_codes(fn), f'{fn} K-{code:06d}: 2-2 수정 뒤 라벨에 없는 알약이다'
    assert new_code is None or new_code in aihub_example, f'K-{new_code:06d}: AI Hub에 메타데이터가 없는 클래스이다'
    assert box is None or usable_bbox(box), f'{fn} K-{code:06d}: 쓸 수 없는 box {box}'
for fn in {fn for fn, _ in LABEL_FINDINGS}:  # 바꾼 뒤에도 한 사진에 같은 클래스가 두 번 나오지 않는다
    codes = [(LABEL_FINDINGS.get((fn, a['category_id'])) or (None, None, None))[1] or a['category_id'] for a in fixed_records[fn]['annotations']]
    assert len(codes) == len(set(codes)), f'{fn}: 바꾼 뒤 같은 클래스가 두 번 있다 {codes}'

# 1-6에서 찾은 같은 장면: 노이즈만 다른 사진이 다른 조합 이름으로 두 번 들어 있다. 파일명이 사진과 맞는 쪽만 남긴다
# 뺄 조합 → 남길 조합. 뺄 쪽은 사진에 K-016551이 없어 2-2(-오류 폴더)와 2-3에서 클래스를 바꿨고, 남길 쪽은 원본 라벨 그대로 맞다
SAME_SCENE_COMBOS = {
    'K-012247-016551-033009-044199': 'K-012247-016548-033009-044199',
    'K-016548-016551-023203-024850': 'K-004543-016548-023203-024850',
}
same_scenes = {fn: fn.replace(drop, keep) for fn in aihub_files for drop, keep in SAME_SCENE_COMBOS.items() if combo_name(fn) == drop}


def classes_after(fn):
    """2-2 수정 뒤 라벨에 2-3을 적용한 이미지의 클래스 목록."""
    return sorted((LABEL_FINDINGS.get((fn, a['category_id'])) or (None, None, None))[1] or a['category_id'] for a in fixed_records[fn]['annotations'])


assert len(same_scenes) == 6 and set(same_scenes) | set(same_scenes.values()) <= set(fixed_records), '같은 장면 사진은 두 조합 쌍에 각도별 3장씩이다'
assert all(classes_after(drop) == classes_after(keep) for drop, keep in same_scenes.items()), '같은 장면인데 고친 클래스가 다르다'

# 2-6에서 찾은 같은 알약 구성: 클래스를 고친 뒤의 알약 4개가 이름이 맞는 다른 조합과 같다. 같은 장면은 아니고 같은 알약을 다시 놓고 찍은 사진이다
# 파일명을 정제 라벨대로 고치면 그 조합과 이름이 겹치므로 이름이 틀린 쪽을 뺀다. 앞의 두 조합은 -오류 폴더(K-023319)가 있는 조합이다
SAME_SET_COMBOS = {  # 뺄 조합 → 남길 조합
    'K-006697-013004-023319-038927': 'K-006697-013004-029711-038927',
    'K-006835-019881-023319-043233': 'K-006835-019881-029711-043233',
    'K-011220-016232-027733-031885': 'K-011220-027733-029667-031885',
}
same_sets = {fn: fn.replace(drop, keep) for fn in aihub_files for drop, keep in SAME_SET_COMBOS.items() if combo_name(fn) == drop}
assert len(same_sets) == 9 and set(same_sets) | set(same_sets.values()) <= set(fixed_records), '같은 알약 구성 사진은 세 조합 쌍에 각도별 3장씩이다'
assert all(classes_after(drop) == classes_after(keep) for drop, keep in same_sets.items()), '같은 알약 구성인데 고친 클래스가 다르다'

# 1-4에서 찾은 메타데이터 오류: 사진은 다른 사진과 같은 연회색 배경인데 JSON의 back_color(와 파일명의 배경 자리 `_0_0_`)가 검은색 배경이다
# 파일명은 PNG, JSON, 정제 목록을 잇는 이름이라 그대로 두고, 정제 라벨의 back_color만 고친다
META_FINDINGS = {'K-012247-016551-024850-029451_0_0_0_2_90_000_200.png': ('back_color', '연회색 배경')}
assert all(fixed_records[fn]['image'][key] != value for fn, (key, value) in META_FINDINGS.items()), '이미 고친 메타데이터이다'


def area_of(box):
    """쓸 수 있는 box의 너비×높이를 구한다. 쓸 수 없는 box는 빈 칸이다."""
    return box[2] * box[3] if usable_bbox(box) else ''


aihub_fix_rows = []  # 정제 목록: 바뀌는 annotation이 한 줄. 파일로 쓰지 않고 2-5가 이 목록을 그대로 적용한다
for f in copy_fixes:  # 2-1의 copy 재배정 (2-5가 목록 순서로 적용하므로 2-2 수정보다 앞에 둔다)
    if f['action'] == '그대로':
        continue
    befores = [box for _, box in f['before']]  # 1-3이 이 사진에 넣은 그 K-코드의 JSON bbox (없거나 여러 개일 수 있다)
    aihub_fix_rows.append({
        'file_name': f['file_name'], 'category_id': f['code'], 'problem': 'JSON 이름의 각도가 틀림',
        'action': f"copy 재배정 ({f['action']}): {f['box']}",
        'bbox_before': json.dumps(befores[0] if len(befores) == 1 else befores) if befores else '', 'bbox_after': json.dumps(f['box']),
        'area_before': (area_of(befores[0]) if len(befores) == 1 else json.dumps([area_of(b) for b in befores])) if befores else '',
        'area_after': area_of(f['box']), 'category_id_after': '', 'evidence': '2-1',
    })
for fix in box_fixes:  # 2-2의 처리
    raw_box, box = fix['raw_box'], fix['box']
    relabel = fix['label_id'] if fix['label_id'] != fix['category_id'] else ''  # -오류 폴더 알약은 사진 속 실제 클래스로 붙였다
    if fix['action'] == '찾지 못함':
        action = '이미지 제외 (사진에서 알약을 찾지 못함)'
    elif fix['action'].startswith('그대로 둠') and not relabel:
        if fix['raw_area'] in (None, box[2] * box[3]):
            continue  # bbox와 area 모두 바뀌지 않음 (box가 똑같은 두 클래스 중 그 box의 알약인 쪽)
        action = 'area를 너비×높이로 다시 계산'
    else:
        action = f"{fix['action']}: {box}"
    aihub_fix_rows.append({
        'file_name': fix['file_name'], 'category_id': fix['category_id'], 'problem': fix['problem'], 'action': action,
        'bbox_before': json.dumps(raw_box) if raw_box is not None else '', 'bbox_after': json.dumps(box) if box else '',
        'area_before': fix['raw_area'] if fix['raw_area'] is not None else '', 'area_after': area_of(box) if box else '',
        'category_id_after': relabel, 'evidence': '2-1, 2-2' if fix['problem'].startswith('라벨 누락') else '2-2',
    })
for (fn, code), (problem, new_code, box) in LABEL_FINDINGS.items():  # 2-3의 눈 확인 (같은 알약의 2-2 수정 다음에 적용한다)
    raw_box, raw_area = raw_annotation(fn, code)
    after = box or label_at[(fn, code)]['bbox']
    action = ', '.join((['클래스 바꿈'] if new_code else []) + (['box 다시 그림'] if box else [])) + f': {after}'
    aihub_fix_rows.append({
        'file_name': fn, 'category_id': code, 'problem': problem, 'action': action,
        'bbox_before': json.dumps(raw_box) if raw_box is not None else '', 'bbox_after': json.dumps(after),
        'area_before': raw_area if raw_area is not None else '', 'area_after': area_of(after),
        'category_id_after': new_code or '', 'evidence': '2-3',
    })
for drops, kind, action, evidence in [(same_scenes, '같은 장면', '이미지 제외 (다른 조합 이름의 같은 장면)', '1-6'),
                                      (same_sets, '같은 알약 구성', '이미지 제외 (같은 알약 구성의 다른 조합)', '2-6')]:
    for fn, keep in drops.items():  # 1-6의 같은 장면과 2-6의 같은 알약 구성 (사진에 없는 알약의 줄로 이미지를 뺀다)
        code = next(c for c in pill_codes(fn) if c not in pill_codes(keep))
        raw_box, raw_area = raw_annotation(fn, code)
        aihub_fix_rows.append({
            'file_name': fn, 'category_id': code, 'problem': f'파일명이 사진과 다름 ({kind}: {keep})', 'action': action,
            'bbox_before': json.dumps(raw_box) if raw_box is not None else '', 'bbox_after': '',
            'area_before': raw_area if raw_area is not None else '', 'area_after': '', 'category_id_after': '', 'evidence': evidence,
        })
for fn, (key, value) in META_FINDINGS.items():  # 1-4의 메타데이터 오류 (알약마다 JSON이 따로 있어 알약마다 적는다)
    for code in pill_codes(fn):
        raw_box, raw_area = raw_annotation(fn, code)
        box = label_at[(fn, code)]['bbox']  # box는 바꾸지 않는다
        aihub_fix_rows.append({
            'file_name': fn, 'category_id': code, 'problem': f"사진은 {value}인데 {key}가 {fixed_records[fn]['image'][key]}", 'action': f'메타데이터 고침: {key}={value}',
            'bbox_before': json.dumps(raw_box) if raw_box is not None else '', 'bbox_after': json.dumps(box),
            'area_before': raw_area if raw_area is not None else '', 'area_after': area_of(box), 'category_id_after': '', 'evidence': '1-4',
        })

action_kind = lambda a: a.split(':')[0]
summary = collections.Counter((row['evidence'], action_kind(row['action'])) for row in aihub_fix_rows)
rows = [[evidence, action, n, len({r['file_name'] for r in aihub_fix_rows if r['evidence'] == evidence and action_kind(r['action']) == action})]
        for (evidence, action), n in sorted(summary.items())]
rows.append(['**합계**', '', f'**{len(aihub_fix_rows)}줄**', f"**{len({r['file_name'] for r in aihub_fix_rows})}장**"])
show_table(['근거', '처리', 'annotation 수', '이미지 수'], rows)

#### 2-4-1 결과

| 처리 | 줄 수 |
|---|---|
| area를 너비×높이로 다시 계산 | 337 |
| 숫자 하나 고침 (area로 확인 7줄 포함) | 312 |
| 사진에서 다시 그림 | 287 |
| 2-3의 클래스 바꿈, box 다시 그림 | 63 |
| 눈으로 확인해 고침 (2-2) | 52 |
| copy 재배정 (2-1) | 45 |
| 이미지 제외 | 36 (35장) |
| 메타데이터 고침 | 4 (1장) |
| **합계** | **1,136 (이미지 961장)** |

- 고치는 이미지 961장은 촬영 PNG 12,009장의 8.0%다.
- 두 단계에서 고친 annotation이 36개라, 고친 annotation은 1,100개다. (2-2와 2-3에서 고친 21개, 고친 뒤 1-6에서 뺀 6개, 2-6에서 뺀 9개)
- 이미지 제외 36줄은 찾지 못함 21줄(20장), 같은 알약 구성 9줄(9장), 같은 장면 6줄(6장)이다.
- 클래스를 바꾸는 줄은 56줄이다. (`-오류` 폴더 9줄, 2-3 47줄)

#### 2-4-2. 문제가 어떻게 풀렸나

1-4, 1-6, 2-1~2-3, 2-6에서 문제로 제기된 annotation이 어떻게 해소되었는지 한 표로 모은다. 행은 문제를 찾은 단계, 열은 결과다.

| 열 | 뜻 |
|---|---|
| 문제로 제기 | 그 단계에서 문제로 잡힌 annotation 수 (1-6과 2-6은 이미지 수) |
| 고칠 값 없음 | 처리해 보니 원본과 같아서 줄이 없는 것. copy 배정이 이미 맞은 JSON, box가 똑같은 쌍 중 그 box의 알약인 쪽, 다시 재 보니 알약을 감싼 box, 크기·색 배정이 틀린 후보, area가 빈 값인 annotation |
| 고침 (자동) | 2-2-4의 규칙으로 고친 것 |
| 고침 (눈으로 확인) | 사진이나 갤러리를 보고 정한 값으로 고친 것 (copy 배정, `REVIEWED_BOXES`, `LABEL_FINDINGS`, `META_FINDINGS`) |
| 이미지 제외 | 같은 장면(1-6)이나 같은 알약 구성(2-6)이라 뺀 이미지 |
| 남은 문제 | 사진에서 알약을 찾지 못해 고치지 못한 것. 그 이미지는 학습에서 뺀다 (2-2-9) |

- 고침, 이미지 제외, 남은 문제를 더하면 정제 목록의 줄 수와 같다. 같은 annotation을 두 단계에서 고쳤으면 두 단계에서 한 번씩 센다.

In [ ]:
FIX_OUTCOMES = ['고칠 값 없음', '고침 (자동)', '고침 (눈으로 확인)', '이미지 제외', '남은 문제']
STAGES = [  # (단계, 찾은 방법)
    ('1-4 메타데이터가 사진과 다름', '파일 (메타데이터 값) → 사진 확인'),
    ('1-6 다른 조합 이름의 같은 장면 (이미지 수)', '사진 계산 (축소 그림 RMS)'),
    ('2-1 copy 파일의 각도', '파일 (copy JSON) → 사진에 그려 확인'),
    ('2-1 라벨 누락 (2-2-1 표의 라벨 누락 행)', '파일 (JSON)'),
    ('2-2 좌표 오류 (2-2-1 표의 좌표 행)', '파일 (JSON)'),
    ('2-2 알약을 감싸지 않는 box (2-2-3)', '사진 계산 (OpenCV 알약 범위)'),
    ('2-2 문제 갤러리에서 새로 찾은 box (2-2-5)', '눈 (2-2 문제 갤러리)'),
    ('2-3 크기·색 대조 (2-3-1)', '사진 계산 (OpenCV 크기·색) → 시트 확인'),
    ('2-3 crop 갤러리', '눈 (2-3 crop 갤러리)'),
    ('2-6 같은 알약 구성의 다른 조합 (이미지 수)', '사진 계산 (정제 라벨의 클래스 → 조합 이름)'),
]


def box_fix_stage(problem):
    """2-2의 문제 이름으로 그 문제를 찾은 단계를 정한다."""
    if problem.startswith('라벨 누락'):
        return STAGES[3][0]
    if problem == '알약을 감싸지 않는 box':
        return STAGES[5][0]
    if problem == REVIEW_PROBLEM:
        return STAGES[6][0]
    return STAGES[4][0]


def box_fix_outcome(fix):
    """2-2의 처리 결과를 표의 열로 나눈다. 정제 목록에 줄을 두는 규칙(2-4-1)과 같다."""
    if fix['action'] == '찾지 못함':
        return '남은 문제'
    if fix['action'] == '눈으로 확인해 고침':
        return '고침 (눈으로 확인)'
    if (fix['action'].startswith('그대로 둠') and fix['label_id'] == fix['category_id']
            and fix['raw_area'] in (None, fix['box'][2] * fix['box'][3])):
        return '고칠 값 없음'
    return '고침 (자동)'


outcomes = {stage: collections.Counter() for stage, _ in STAGES}  # 단계 → 열 → annotation 수
for fn in META_FINDINGS:
    outcomes[STAGES[0][0]]['고침 (눈으로 확인)'] += len(pill_codes(fn))  # 알약마다 JSON이 따로 있어 알약마다 한 줄
outcomes[STAGES[1][0]]['이미지 제외'] += len(same_scenes)
outcomes[STAGES[9][0]]['이미지 제외'] += len(same_sets)
for f in copy_fixes:
    outcomes[STAGES[2][0]]['고칠 값 없음' if f['action'] == '그대로' else '고침 (눈으로 확인)'] += 1
for fix in box_fixes:
    outcomes[box_fix_stage(fix['problem'])][box_fix_outcome(fix)] += 1
# 크기·색 대조: 다시 배정한 클래스가 라벨과 다른 box. 시트에서 확인해 LABEL_FINDINGS에 적은 것만 고쳤다
flagged = {(fn, fixed_records[fn]['annotations'][i]['category_id']) for fn in mismatched for i, j in enumerate(checked[fn]['order']) if i != j}
for key in flagged:
    outcomes[STAGES[7][0]]['고침 (눈으로 확인)' if key in LABEL_FINDINGS else '고칠 값 없음'] += 1
outcomes[STAGES[8][0]]['고침 (눈으로 확인)'] += sum(key not in flagged for key in LABEL_FINDINGS)
total = sum(outcomes.values(), collections.Counter())
assert total['고침 (자동)'] + total['고침 (눈으로 확인)'] + total['이미지 제외'] + total['남은 문제'] == len(aihub_fix_rows), \
    '고침, 이미지 제외, 남은 문제의 합이 정제 목록의 줄 수와 다르다'


def outcome_row(counts):
    """단계 하나의 수: 문제로 제기, 열마다 수, 해결(고침 + 이미지 제외), 남은 문제"""
    solved = counts['고침 (자동)'] + counts['고침 (눈으로 확인)'] + counts['이미지 제외']
    return [sum(counts.values()), *[counts[c] for c in FIX_OUTCOMES[:4]], solved, counts['남은 문제']]


rows = [[stage, how, *[f'{v:,}' if v else '-' for v in outcome_row(outcomes[stage])]] for stage, how in STAGES]
rows.append(['**합계**', '', *[f'**{v:,}**' for v in outcome_row(total)]])
show_table(['단계 (문제를 찾은 곳)', '찾은 방법', '문제로 제기', *FIX_OUTCOMES[:4], '해결 (고침 + 이미지 제외)', '남은 문제'], rows)

#### 2-4-2 결과

| 결과 | 수 | 비율 |
|---|---|---|
| 문제로 제기 | 1,406 | 100% |
| 고칠 값 없음 | 270 | 19.2% |
| 해결 (고침 + 이미지 제외) | 1,115 | 79.3% |
| 남은 문제 | 21 | 1.5% |

- 찾은 방법별로는 파일 검사 905개, 사진 계산 468개, 눈 33개다.
- 해결 1,115개는 자동 규칙 936개, 눈으로 확인 164개, 이미지 제외 15개다.
    - 눈으로 확인한 164개: 2-3 63개, 2-2 문제 갤러리 52개, 2-1 copy 배정 45개, 1-4 메타데이터 4개
- 고칠 값 없음 270개 중 108개는 크기·색 대조 후보다. 후보 70장(140개) 중 16장(32개)만 라벨이 정말 틀렸다.
- 남은 21개(2-1 라벨 누락 9개, 2-2 좌표 오류 12개)는 모두 사진에서 알약을 찾지 못한 경우이고, 이미지 20장에 있다.
- 해결 1,115개와 남은 21개를 더한 1,136개는 정제 목록의 줄 수와 같다.

### 2-5. 정제 목록 적용

- 2-4-1의 정제 목록을 1-3의 바꾼 AI Hub 라벨에 적용해 정제 라벨(`aihub_clean_records`)을 만든다. 3절부터 이 라벨로 본다.
    - 1-3 라벨은 copy 재배정(2-1-3) 전의 라벨이다. copy 재배정도 목록의 2-1 줄로 적용한다.

| 경우 | 적용 |
|---|---|
| 줄이 없는 알약 | 1-3의 annotation을 그대로 쓴다 |
| 줄이 있는 알약 | 목록 순서대로 `bbox_after`로 box를 바꾼다. 1-3의 annotation이 없으면 새로 만든다 (알약 정보는 같은 클래스의 다른 annotation에서) |
| `category_id_after`가 있는 줄 | 클래스와 알약 정보를 그 클래스의 것으로 바꾼다 |
| 메타데이터 고침 줄 | box와 클래스는 그대로 두고 이미지 필드만 고친다 |
| 이미지 제외 줄이 있는 이미지 | 뺀다 |

- 모든 area는 너비×높이로 다시 계산한다.
- 결과가 노트북 안에서 고친 라벨(2-2 수정 뒤 라벨 + 2-3의 `LABEL_FINDINGS`)과 같은지 assert로 확인한다.
- 아래 표는 단계별 라벨의 이미지, 객체, 클래스 수와 남은 라벨 문제다.

In [ ]:
fix_table = aihub_fix_rows  # 2-4-1의 정제 목록
excluded_files = {row['file_name'] for row in fix_table if row['action'].startswith('이미지 제외')}
rows_of = collections.defaultdict(list)  # (이미지, 1-3의 K-코드) → 정제 목록의 줄 (목록 순서)
for row in fix_table:
    rows_of[(row['file_name'], int(row['category_id']))].append(row)
converted = {}  # 이미지 → 1-3이 넣은 annotation (2-1-3이 copy를 다시 배정하기 전). 촬영 PNG만 있는 이미지는 없다
for j in aihub_kaggle_jsons:
    if j['data'] is not None:
        converted.setdefault(j['file_name'], []).append({**j['data']['annotations'][0], 'image': j['data']['images'][0], 'json_path': str(j['path'])})

aihub_clean_records = {}  # 정제 라벨: 이미지 → {'image', 'annotations'}. 3번부터 이 라벨로 본다
for fn in aihub_files:
    if fn in excluded_files:
        continue
    anns = []
    for code in pill_codes(fn):
        found = [a for a in converted.get(fn, []) if a['category_id'] == code]  # copy 파일이 있는 알약 폴더는 0~3개이다 (2-1)
        rows = rows_of[(fn, code)]
        if not rows:
            assert len(found) == 1 and usable_bbox(found[0]['bbox']), f'{fn} K-{code:06d}: 정제 목록에 줄이 없는데 1-3 라벨을 그대로 쓸 수 없다'
            anns.append(dict(found[0]))
            continue
        ann = dict(found[0]) if found else drawn_annotation(fn, code, json.loads(rows[0]['bbox_after']))
        for row in rows:  # 2-1이나 2-2의 줄 다음에 2-3의 줄을 적용한다
            if row['action'].startswith('메타데이터 고침'):  # 1-4: box와 클래스는 그대로 두고 이미지 필드만 고친다
                key, value = row['action'].split(': ', 1)[1].split('=', 1)
                ann['image'] = {**ann['image'], key: value}
                continue
            ann['bbox'] = json.loads(row['bbox_after'])
            if row['category_id_after']:  # 클래스를 바꾸면 알약 메타데이터도 그 클래스의 것으로 바꾼다
                new_code = int(row['category_id_after'])
                ann['category_id'], ann['image'] = new_code, drawn_annotation(fn, new_code, ann['bbox'])['image']
        anns.append(ann)
    for ann in anns:
        ann['area'] = ann['bbox'][2] * ann['bbox'][3]
    aihub_clean_records[fn] = {'image': anns[0]['image'], 'annotations': anns}


def class_boxes(fn, anns, findings=None):
    """이미지의 (클래스, box) 목록. findings를 주면 2-3의 LABEL_FINDINGS를 적용한 값이다."""
    pairs = []
    for a in anns:
        _, new_code, box = (findings or {}).get((fn, a['category_id']), (None, None, None))
        pairs.append((new_code or a['category_id'], tuple(box or a['bbox'])))
    return sorted(pairs)


# 정제 목록만 적용해도 노트북 안에서 고친 라벨(2-2 수정 뒤 라벨 + 2-3)과 같은 라벨이 나오는지 확인한다
assert set(aihub_clean_records) == set(fixed_records) - set(same_scenes) - set(same_sets), \
    '정제 라벨의 이미지가 2-2 수정 뒤 라벨에서 같은 장면(1-6)과 같은 알약 구성(2-6)을 뺀 것과 다르다'
assert all(a['image'][key] == value for fn, (key, value) in META_FINDINGS.items() for a in aihub_clean_records[fn]['annotations']), '메타데이터 고침(1-4)이 정제 라벨에 없다'
assert all(class_boxes(fn, r['annotations']) == class_boxes(fn, fixed_records[fn]['annotations'], LABEL_FINDINGS)
           for fn, r in aihub_clean_records.items()), '정제 목록을 적용한 라벨이 2-2 수정 뒤 라벨에 2-3을 적용한 것과 다르다'


def label_summary(labels):
    """이미지 → annotation 목록에서 이미지, 객체, 클래스 수와 남은 라벨 문제를 센다."""
    objects = [a for anns in labels.values() for a in anns]
    usable = {fn: [a for a in anns if usable_bbox(a['bbox'])] for fn, anns in labels.items()}
    return [f'{len(labels):,}장', f'{len(objects):,}개', f"{len({a['category_id'] for a in objects})}개",
            f"{sum(not usable_bbox(a['bbox']) for a in objects):,}개",
            f"{sum(len(anns) != len(pill_codes(fn)) for fn, anns in usable.items()):,}장",
            f"{sum(any(a['category_id'] not in pill_codes(fn) for a in anns) for fn, anns in labels.items()):,}장"]


show_table(['라벨', '이미지', '객체', '클래스', '쓸 수 없는 bbox', '쓸 수 있는 box 수가 파일명의 알약 수와 다른 이미지', '파일명에 없는 클래스가 있는 이미지'], [
    ['바꾼 AI Hub 라벨 (1-3)', *label_summary(converted)],
    ['copy 재배정 뒤 (2-1-3)', *label_summary({fn: r['annotations'] for fn, r in aihub_kaggle.items()})],
    ['2-2 수정 뒤 (2-3)', *label_summary({fn: r['annotations'] for fn, r in fixed_records.items()})],
    ['**정제 라벨 (2-5, 정제 목록 적용)**', *[f'**{v}**' for v in label_summary({fn: r['annotations'] for fn, r in aihub_clean_records.items()})]],
])

#### 2-5 결과

| 라벨 | 이미지 | 객체 | 쓸 수 없는 bbox | box 수가 파일명과 다른 이미지 | 파일명에 없는 클래스가 있는 이미지 |
|---|---|---|---|---|---|
| 바꾼 AI Hub 라벨 (1-3) | 11,997 | 45,909 | 163 | 179 | 0 |
| copy 재배정 뒤 (2-1-3) | 11,997 | 45,909 | 163 | 141 | 0 |
| 2-2 수정 뒤 | 11,989 | 45,879 | 0 | 0 | 9 |
| **정제 라벨 (2-5)** | **11,974** | **45,819** | **0** | **0** | **12** |

- 정제 라벨은 이미지 11,974장, 객체 45,819개, 클래스 116개다.
- 촬영 PNG 12,009장에서 35장을 뺐다. 알약을 찾지 못한 20장(2-2), 같은 알약 구성 9장(2-6), 같은 장면 6장(1-6)이다.
- 촬영 PNG만 있던 12장은 box를 그려 넣어 들어갔다.
- 파일명에 없는 클래스가 있는 이미지는 12장이다. 모두 2-3에서 사진 속 알약의 클래스로 바꾼 사진이다.
- 정제 목록만 적용한 라벨이 노트북 안에서 고친 라벨과 같고, 메타데이터 고침도 들어갔다. (assert 통과)

이 12장은 파일명의 K-코드로 클래스를 정하면 틀린다. 클래스는 정제 라벨에서 읽는다. 파일명은 2-6에서 다룬다.

### 2-6. 정제 뒤 파일명

| 단계 | 하는 일 |
|---|---|
| 2-6-1 | 파일명과 맞지 않게 된 클래스를 조합마다 보고, 이름을 고치면 다른 조합과 겹치는지 본다 |
| 2-6-2 | 정제 라벨로 파일명을 다시 만들어 바뀌는 이름을 모은다 |

#### 2-6-1. 파일명과 맞지 않는 클래스

- AI Hub 이미지 파일명은 조합 이름(사진에 놓인 알약의 K-코드 목록)으로 시작한다. 2-2와 2-3에서 클래스를 사진에 맞추면서, 파일명의 K-코드와 정제 라벨의 클래스가 다른 이미지가 생겼다.
- 클래스를 바꾼 이미지는 둘로 나뉜다.
    - 파일명 안의 두 알약끼리 라벨을 맞바꾼 이미지: 클래스 목록은 파일명과 같다.
    - 파일명에 없는 알약의 클래스로 바꾼 이미지: 클래스 목록이 파일명과 다르다. 아래 그림은 이 이미지만 그린다.
- 2-4에서 뺀 같은 알약 구성 9장도 넣어 그린다. 이 그림이 뺀 이유다.
- 열마다 조합 하나의 90° 사진이다. 같은 조합의 세 사진은 바뀐 클래스가 같다.
    - 위: 정제 라벨의 box. 파일명과 다른 클래스의 box는 빨간색이다.
    - 아래: 고친 이름의 조합이 이미 있으면 그 조합의 90° 사진과, 두 사진의 축소 그림 RMS 차이 (1-6에서 같은 장면은 0.4)

In [ ]:
def code_list(codes):
    """K-코드 목록을 조합 이름 형식(작은 번호부터)으로 적는다."""
    return 'K-' + '-'.join(f'{c:06d}' for c in sorted(codes))


def thumb_rms(fn_a, fn_b):
    """두 AI Hub 사진의 축소 그림 RMS 차이를 계산한다. (1-6과 같은 값)"""
    a, b = (stats['thumb'][stat_index[f'aihub/{fn}']].astype(np.float32) for fn in (fn_a, fn_b))
    return float(np.sqrt(((a - b) ** 2).mean()))


# 2-4에서 뺀 같은 알약 구성 9장도 넣는다 (뺀 이유를 이 그림으로 보인다). 9장의 라벨은 2-2 수정 뒤 라벨에 2-3을 적용한 값이다
labels_before_set = {fn: r['annotations'] for fn, r in aihub_clean_records.items()}
labels_before_set.update({fn: [{'category_id': c, 'bbox': list(b)} for c, b in class_boxes(fn, fixed_records[fn]['annotations'], LABEL_FINDINGS)]
                          for fn in same_sets})
changed_files = sorted({row['file_name'] for row in fix_table if row['category_id_after'] and row['file_name'] in labels_before_set})  # 클래스를 바꾼 이미지
name_mismatch = sorted(fn for fn, anns in labels_before_set.items() if sorted(a['category_id'] for a in anns) != sorted(pill_codes(fn)))
assert set(name_mismatch) <= set(changed_files), '클래스를 바꾸지 않았는데 파일명과 클래스가 다른 이미지가 있다'
swap_of = {}  # 이미지 → (파일명에만 있는 K-코드, 정제 라벨에만 있는 K-코드)
for fn in name_mismatch:
    classes = {a['category_id'] for a in labels_before_set[fn]}
    (old,), (new,) = set(pill_codes(fn)) - classes, classes - set(pill_codes(fn))  # 모든 이미지에서 알약 하나만 다르다
    swap_of[fn] = (old, new)
mismatch_combos = sorted({combo_name(fn) for fn in name_mismatch})
assert all(len({swap_of[fn] for fn in name_mismatch if combo_name(fn) == c}) == 1 for c in mismatch_combos), '같은 조합의 사진끼리 바뀐 클래스가 다르다'
combos_of_codes = collections.defaultdict(set)  # 파일명의 K-코드 집합 → 그 조합 이름 (정제 라벨에 남은 이미지)
for fn in aihub_clean_records:
    combos_of_codes[frozenset(pill_codes(fn))].add(combo_name(fn))


def photo_90(combo):
    return next(fn for fn in sorted(labels_before_set) if combo_name(fn) == combo and fn.split('_')[5] == '90')  # 파일명의 각도 자리 (1-4에서 메타데이터와 같음)


def draw_labels(ax, fn, highlight=None):
    """정제 라벨의 box를 그린다. highlight=(파일명의 K-코드, 정제 라벨의 클래스)이면 그 클래스의 box를 빨간색으로 그린다."""
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in labels_before_set[fn]:
        x, y, w, h = ann['bbox']
        changed = highlight is not None and ann['category_id'] == highlight[1]
        color = 'red' if changed else 'lime'
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=3 if changed else 1.5))
        text = f'파일명 K-{highlight[0]:06d}\n→ 라벨 K-{highlight[1]:06d}' if changed else f"K-{ann['category_id']:06d}"
        tx, ha = (x + w, 'right') if x + w / 2 > IMG_W / 2 else (x, 'left')  # 오른쪽 알약은 글자를 box 오른쪽 끝에 맞춘다
        ty, va = (y + 10, 'top') if y < 120 else (y - 8, 'bottom')  # 위쪽 끝 알약은 글자를 box 안에 적는다
        ax.text(tx, ty, text, fontsize=8 if changed else 7, ha=ha, va=va, color='white' if changed else 'black',
                bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0), zorder=4 if changed else 3)
    ax.set_xticks([])
    ax.set_yticks([])


fig, axes = plt.subplots(2, len(mismatch_combos), figsize=(4 * len(mismatch_combos), 13.5), layout='constrained')
for col, combo in enumerate(mismatch_combos):
    fn = photo_90(combo)
    old, new = swap_of[fn]
    evidence = next(row['evidence'] for row in rows_of[(fn, old)] if row['category_id_after'])
    draw_labels(axes[0, col], fn, (old, new))
    n_photos = sum(combo_name(f) == combo for f in name_mismatch)
    axes[0, col].set_title(f'파일명: {combo}\n정제 라벨: {code_list(set(pill_codes(fn)) - {old} | {new})}\n'
                           f'90° (같은 조합 {n_photos}장 모두 같음, 근거 {evidence})', fontsize=9)
    renamed_codes = set(pill_codes(fn)) - {old} | {new}
    twins = sorted(combos_of_codes.get(frozenset(renamed_codes), set()))
    ax = axes[1, col]
    if twins:
        twin_fn = photo_90(twins[0])
        draw_labels(ax, twin_fn)
        ax.set_title(f'고친 이름과 같은 조합이 이미 있다 (다른 사진)\n{twins[0]} 90°\n위 사진과 축소 그림 RMS 차이 {thumb_rms(fn, twin_fn):.1f}'
                     f'\n→ 위 조합 {n_photos}장을 2-4에서 뺀다',
                     fontsize=9, color='crimson')
    else:
        ax.text(0.5, 0.5, f'고친 이름\n{code_list(renamed_codes)}\n\n같은 이름의 조합 없음\n(이름을 고쳐도 겹치지 않는다)',
                ha='center', va='center', fontsize=10, transform=ax.transAxes)
        ax.axis('off')
fig.suptitle(f'정제로 생긴 파일명과 클래스의 불일치: 클래스를 바꾼 이미지 {len(changed_files)}장 중 파일명에 없는 클래스가 생긴 {len(name_mismatch)}장 '
             f'({len(mismatch_combos)}개 조합)\n위: 정제 라벨의 90° 사진 (빨간 box = 파일명의 K-코드와 다른 클래스) | '
             f'아래: 파일명을 정제 라벨대로 고친 이름의 조합이 이미 있으면 그 조합의 90° 사진', fontsize=14)
save_fig(fig, '2-6_name_mismatch')
plt.show()

#### 2-6-1 결과

- 클래스를 바꾼 이미지 34장 중 13장은 파일명 안의 두 알약끼리 맞바꿔서, 클래스 목록이 파일명과 같다.
- 21장(7개 조합 × 3각도)은 파일명에 없는 알약의 클래스로 바꿨다. 7개 조합 모두 알약 하나만 다르다.
    - `-오류` 폴더 조합 2개(K-023319 → K-029711)와 2-3에서 고친 조합 5개다.
- 파일명을 정제 라벨대로 고치면 3개 조합(9장)은 이미 있는 다른 조합과 이름이 같아진다.

| 원래 조합 | 고친 이름 (이미 있는 조합) |
|---|---|
| `K-006697-013004-023319-038927` | `K-006697-013004-029711-038927` |
| `K-006835-019881-023319-043233` | `K-006835-019881-029711-043233` |
| `K-011220-016232-027733-031885` | `K-011220-027733-029667-031885` |

그림에서 보이는 것

- 짝 조합은 같은 알약 4개를 다시 놓고 찍은 다른 사진이다. 알약의 위치나 방향이 조금씩 다르고, 축소 그림 RMS 차이는 8.9~21.6으로 같은 장면(0.4)보다 훨씬 크다.
- 나머지 4개 조합(12장)은 고친 이름의 조합이 없어서, 이름을 고쳐도 겹치지 않는다.

이름이 겹치는 3개 조합 9장은 이름이 틀린 쪽이라 2-4에서 뺀다(`SAME_SET_COMBOS`). 짝 조합에 같은 알약 4개가 세 각도로 남아 있어, 클래스마다 3장(K-029711은 6장)만 줄어든다.

#### 2-6-2. 정제 라벨로 다시 만든 이름

- 정제 라벨로 이미지 파일명을 다시 만들어, 원래 이름과 달라지는 이미지를 모은다.
    - 조합 자리: 정제 라벨의 클래스를 작은 번호부터 적는다.
    - 숫자 7자리: 정제 라벨의 이미지 필드(drug_S, back_color, drug_dir, light_color, camera_la, camera_lo, size)를 원래 파일명과 같은 코드로 적는다. 코드 대응표는 원래 파일명과 이미지 필드 값을 짝지어 만든다.
- 클래스와 메타데이터를 고치지 않은 이미지는 다시 만든 이름이 원래 이름과 같아야 한다. 이것으로 규칙을 확인한다. (assert)
- 원래 이름은 원본 PNG·JSON과 정제 목록을 잇는 키라서 바꾸지 않는다. 고친 이름은 2-7에서만 쓴다.

In [ ]:
EXPORT_NAME_FIELDS = ['drug_S', 'back_color', 'drug_dir', 'light_color', 'camera_la', 'camera_lo', 'size']  # 파일명 숫자 7자리의 순서
export_code_pairs = collections.defaultdict(set)  # 이미지 필드 → (값, 파일명의 코드) 짝
for fn, r in aihub_kaggle.items():
    for field, name_code in zip(EXPORT_NAME_FIELDS, fn[:-len('.png')].split('_')[1:]):
        export_code_pairs[field].add((str(r['image'][field]), name_code))
export_code_of = {}  # 이미지 필드 → 값 → 파일명의 코드
for field, pairs in export_code_pairs.items():
    assert len({v for v, _ in pairs}) == len({c for _, c in pairs}) == len(pairs), f'{field}: 값과 파일명 코드가 하나씩 맞지 않는다 {sorted(pairs)}'
    export_code_of[field] = dict(pairs)


def export_name(fn, record):
    """정제 라벨로 이미지 파일명을 다시 만든다. 조합 자리는 정제 클래스, 숫자 자리는 정제 이미지 필드의 코드이다."""
    name_codes = [export_code_of[field][str(record['image'][field])] for field in EXPORT_NAME_FIELDS]
    return '_'.join([code_list({a['category_id'] for a in record['annotations']}), *name_codes]) + '.png'


def rename_reason(fn, new):
    """원래 이름과 고친 이름이 다른 자리를 적는다."""
    reasons = []
    old_codes, new_codes = set(pill_codes(fn)), set(pill_codes(new))
    if old_codes != new_codes:
        (old,), (added,) = old_codes - new_codes, new_codes - old_codes  # 2-6에서 본 대로 알약 하나만 다르다
        reasons.append(f'클래스 K-{old:06d} → K-{added:06d}')
    value_of = {field: {c: v for v, c in m.items()} for field, m in export_code_of.items()}
    for field, a, b in zip(EXPORT_NAME_FIELDS, fn[:-len('.png')].split('_')[1:], new[:-len('.png')].split('_')[1:]):
        if a != b:
            reasons.append(f'{field} {a}({value_of[field][a]}) → {b}({value_of[field][b]})')
    return ', '.join(reasons)


export_names = {fn: export_name(fn, r) for fn, r in aihub_clean_records.items()}  # 원래 이름 → 정제 라벨로 다시 만든 이름
label_fix_rows = collections.defaultdict(list)  # 이미지 → 클래스나 메타데이터를 고친 정제 목록 줄
for row in fix_table:
    if row['category_id_after'] or row['action'].startswith('메타데이터 고침'):
        label_fix_rows[row['file_name']].append(row)
assert all(new == fn for fn, new in export_names.items() if fn not in label_fix_rows), '클래스와 메타데이터를 고치지 않았는데 다시 만든 이름이 원래 이름과 다르다'
assert len(set(export_names.values())) == len(export_names), '다시 만든 이름이 겹친다'
export_renamed = {fn: new for fn, new in export_names.items() if new != fn}  # 이름이 바뀌는 이미지
assert not set(export_renamed.values()) & set(aihub_png), '고친 이름이 이미 있는 PNG 이름과 같다'
export_reasons = {fn: rename_reason(fn, new) for fn, new in export_renamed.items()}
show_table(['원래 이름', '고친 이름', '바뀐 자리', '근거'],
           [[f'`{fn}`', f'`{new}`', export_reasons[fn], ', '.join(sorted({row['evidence'] for row in label_fix_rows[fn]}))]
            for fn, new in sorted(export_renamed.items())])

#### 2-6-2 결과

- 다시 만든 이름이 원래 이름과 다른 이미지는 13장이다.

| 원래 이름 | 고친 이름 | 바뀐 것 | 근거 |
|---|---|---|---|
| `K-003614-027777-028763-030308` 3장 | `K-003614-016262-027777-028763` | K-030308 → K-016262 | 2-3 |
| `K-015280-016688-032310` 3장 | `K-016206-016688-032310` | K-015280 → K-016206 | 2-3 |
| `K-015280-018147-022074` 3장 | `K-015280-018147-033026` | K-022074 → K-033026 | 2-3 |
| `K-016206-018147-022074` 3장 | `K-016206-018147-033026` | K-022074 → K-033026 | 2-3 |
| `K-012247-016551-024850-029451` 90° 1장 | 조합은 그대로 | 배경 자리 0(검은색) → 2(연회색) | 1-4 |

- 나머지 11,961장은 다시 만든 이름이 원래 이름과 같다. 두 알약의 라벨을 맞바꾼 13장도 클래스 목록이 같아 이름이 그대로다.
- 고친 이름은 서로 겹치지 않고, 이미 있는 PNG 이름과도 겹치지 않는다.
- 13장 중 6장(`K-015280-…` 두 조합)은 원본 JSON 없이 촬영 PNG만 있던 이미지다. (1-1-2)

### 2-7. 대회 train 형식으로 만들기

- 정제 라벨(`aihub_clean_records`)을 대회 train과 같은 JSON 구조로 메모리에서 만든다. 파일로 쓰지 않는다.

| 항목 | 만드는 방법 |
|---|---|
| 경로 | `<조합>_json/K-<클래스>/<이미지 이름>.json`. annotation 하나에 JSON 하나 |
| 이미지 이름 | 2-6-2에서 다시 만든 이름 |
| 알약 폴더 | 정제 라벨의 클래스. 그래서 copy 파일 이름, `-오류` 폴더, 클래스를 바꾼 알약의 옛 폴더가 남지 않는다 |
| images | 그 알약의 메타데이터에 이미지 필드(`IMAGE_FIELDS`)를 덮고, file_name·imgfile을 고친 이름으로 바꾼다 |
| id | 1-3과 같이 images는 1501부터, annotations는 5692부터 이름 순서로 매긴다 |

- 이미지 제외 줄이 있는 이미지, 사진 없는 JSON, `_index.png`는 넣지 않는다.
- 만든 JSON을 이미지 단위로 다시 묶어 정제 라벨과 같은지 assert로 확인한다. (클래스와 box, 폴더와 클래스, 파일명의 K-코드와 클래스)

In [ ]:
# 정제 라벨을 대회 train과 같은 JSON 구조로 메모리에서 만든다. 파일로 쓰지 않는다 (학습 데이터는 scripts/prepare_train_data.py가 만든다)
export_image_ids = {new: i for i, new in enumerate(sorted(export_names.values()), start=1501)}
export_ann_id = 5692
export_jsons = {}  # 대회 train_annotations와 같은 구조의 JSON 경로 → 내용
n_new_json = 0  # 원본 JSON이 없던 annotation (그려 넣은 box, 촬영 PNG만 있는 이미지)
for fn, new in sorted(export_names.items(), key=lambda t: t[1]):
    record = aihub_clean_records[fn]
    export_fields = {k: record['image'][k] for k in IMAGE_FIELDS if k in record['image']}
    for ann in sorted(record['annotations'], key=lambda a: a['category_id']):
        cat = ann['category_id']
        x, y, w, h = ann['bbox']
        image = {**ann['image'], **export_fields, 'file_name': new, 'imgfile': new, 'id': export_image_ids[new], 'dl_idx': str(cat)}
        assert image['drug_N'] == f'K-{cat:06d}', f'{fn} K-{cat:06d}: 알약 메타데이터가 다른 클래스의 것이다'
        values = {'area': w * h, 'iscrowd': 0, 'bbox': [x, y, w, h], 'category_id': cat, 'ignore': 0, 'segmentation': [],
                  'id': export_ann_id, 'image_id': export_image_ids[new]}
        assert set(values) == set(KAGGLE_ANN_KEYS), 'Kaggle annotation 키와 다르다'
        data = {'images': [image], 'type': 'instances', 'annotations': [{key: values[key] for key in KAGGLE_ANN_KEYS}],
                'categories': [{'supercategory': 'pill', 'id': cat, 'name': image['dl_name']}]}
        path = f"{combo_name(new)}_json/K-{cat:06d}/{new[:-len('.png')]}.json"
        assert path not in export_jsons, f'{path}: 같은 이미지에 같은 클래스의 JSON이 두 번 생긴다'
        export_jsons[path] = data
        export_ann_id += 1
        n_new_json += ann.get('json_path') is None

# 만든 JSON을 이미지 단위로 다시 묶어 정제 라벨과 같은지 확인한다
export_by_image = collections.defaultdict(list)  # 이미지 이름 → [(JSON 경로, 내용)]
for path, data in export_jsons.items():
    export_by_image[data['images'][0]['file_name']].append((path, data))
assert set(export_by_image) == set(export_names.values()), '만든 JSON의 이미지가 정제 라벨의 이미지와 다르다'
for fn, new in export_names.items():
    anns = [data['annotations'][0] for _, data in export_by_image[new]]
    assert sorted((a['category_id'], tuple(a['bbox'])) for a in anns) == sorted((a['category_id'], tuple(a['bbox'])) for a in aihub_clean_records[fn]['annotations']), \
        f'{new}: 만든 JSON의 클래스와 box가 정제 라벨과 다르다'
    assert sorted(a['category_id'] for a in anns) == sorted(pill_codes(new)), f'{new}: 파일명의 K-코드와 클래스가 다르다'
    for path, data in export_by_image[new]:
        combo_dir, pill_dir = path.split('/')[:2]
        image, cat = data['images'][0], data['annotations'][0]['category_id']
        assert combo_dir == f'{combo_name(new)}_json' and pill_dir == image['drug_N'] == f'K-{cat:06d}', f'{path}: 폴더와 클래스가 다르다'
        assert image['file_name'] == image['imgfile'] == new, f'{path}: JSON 안 이미지 이름이 다르다'

show_table(['항목', '수', '설명'], [
    ['이미지', f'{len(export_names):,}장', '정제 라벨의 이미지. 이름을 고친 이미지는 고친 이름을 쓴다 (2-6-2)'],
    ['JSON', f'{len(export_jsons):,}개', f'annotation 하나에 하나. 원본 JSON이 없던 annotation {n_new_json:,}개는 새로 만들었다'],
    ['조합 폴더', f"{len({combo_name(n) for n in export_names.values()}):,}개", '정제 라벨의 클래스로 만든 조합 이름'],
    ['이름을 고친 이미지', f'{len(export_renamed)}장', '2-6-2 이름 표'],
    ['빠진 촬영 PNG', f'{len(set(aihub_files) - set(aihub_clean_records))}장', '정제 목록에 이미지 제외 줄이 있는 이미지 (2-4-1)'],
    ['빠진 파일', f"{len({j['file_name'] for j in aihub_jsons} - set(aihub_png))}개, {len(aihub_index_files):,}장",
     '사진 없는 JSON의 이미지 이름, `_index.png` (1-1)'],
])

#### 2-7 결과

| 항목 | 수 |
|---|---|
| 이미지 | 11,974장 |
| JSON | 45,819개 |
| 조합 폴더 | 3,997개 |
| 이름을 고친 이미지 | 13장 |

- JSON 수는 정제 라벨의 객체 수(2-5)와 같다.
- 원본 JSON이 없던 annotation 70개는 새로 만들었다. 촬영 PNG만 있던 12장의 42개, copy 재배정으로 채운 23개, 그 밖에 JSON이 없던 5개다.
- 다시 묶은 라벨이 이미지마다 정제 라벨과 같고, 모든 이미지에서 파일명의 K-코드와 클래스 목록이 같다. (assert 통과)

학습 데이터를 만드는 `scripts/prepare_train_data.py`도 같은 구조로 `data/processed/ai_hub/`에 쓴다. (2-4)

### 2-8. 만든 라벨 다시 점검

2-7에서 만든 JSON에 1절과 2절의 검사를 다시 해서, 정제를 마친 라벨에 문제가 남지 않았는지 본다.

| 단계 | 하는 일 |
|---|---|
| 2-8-1 | 형식 검사 (1-1~1-4) |
| 2-8-2 | 라벨 누락, 클래스 불일치, 좌표 오류, OpenCV 검사 (2-1~2-3) |
| 2-8-3 | 남은 문제, 이미지 제외, 같은 장면, 같은 알약 구성, 파일명 (1-6, 2-4~2-6) |

- 검사 함수는 1·2절의 것을 그대로 쓴다.
- 눈으로 찾은 문제(갤러리)는 다시 보지 않는다. 그 결과가 정제 라벨에 들어간 것은 2-5에서 assert로 확인했다.

#### 2-8-1. 형식 검사

- 대회 train, AI Hub 원본(0-6에서 읽은 그대로), AI Hub 정제 라벨(2-7)에 같은 형식 검사를 한다. 항목 끝 괄호는 그 검사를 처음 한 절이다.
- 여기서 더한 검사
    - JSON 안 이미지 이름: file_name과 imgfile이 JSON 파일 이름의 이미지 이름과 같은지
    - 이미지 필드: 같은 이미지의 JSON끼리 이미지 필드(`IMAGE_FIELDS`)가 같은지
    - 저장 형식: JSON 파일의 글자가 `json.dumps(내용, ensure_ascii=False, indent=4)`와 같은지. 파일이 있는 열만 본다

In [ ]:
def read_json_folder(ann_dir):
    """라벨 폴더의 JSON을 0번의 aihub_jsons와 같은 구조(JSON 파일 하나가 한 줄)로 읽는다."""
    rows = []
    for path in sorted(ann_dir.rglob('*.json')):
        try:
            data, error = json.loads(path.read_text(encoding='utf-8')), None
        except json.JSONDecodeError as e:
            data, error = None, str(e)
        rows.append({
            'path': path, 'combo_folder': unicodedata.normalize('NFC', path.parent.parent.name),
            'pill_folder': unicodedata.normalize('NFC', path.parent.name),
            'file_name': unicodedata.normalize('NFC', path.name)[:-len('.json')].split(' copy')[0] + '.png',  # 0번과 같은 규칙
            'data': data, 'error': error,
        })
    return rows


def share_text(n, total):
    """'맞는 수/전체 수'로 적는다."""
    return f'{n:,}/{total:,}'


def recheck_format(jsons, pngs, on_disk=True):
    """1-1, 1-2, 1-3, 1-4의 형식 검사를 JSON 목록과 PNG 이름 목록에 똑같이 적용한다. 항목 → 표에 적을 값.
    on_disk=False(메모리에서 만든 JSON)이면 파일 글자를 보는 저장 형식 검사는 하지 않는다."""
    readable = [j for j in jsons if j['data'] is not None]
    counts = clean_counts(jsons, pngs)  # 1-1
    images_of = collections.defaultdict(list)  # 이미지 이름 → 그 이름을 가리키는 JSON의 images
    for j in readable:
        images_of[j['file_name']].append(j['data']['images'][0])
    n_error = sum(j['pill_folder'].endswith('-오류') for j in jsons)
    n_copy = sum(' copy' in j['path'].name for j in jsons)
    return {
        'JSON 파일 (1-1)': f'{len(jsons):,}개',
        'PNG 파일: 촬영 PNG / `_index.png` (1-1)': '{1} / {2}'.format(*(f'{n:,}장' for n in counts['PNG 파일 (촬영 PNG / _index.png)'])),
        'JSON만 있는 이미지 / 촬영 PNG만 있는 이미지 (1-1)':
            f"{counts['JSON만 있는 이미지 파일명'][0]:,}장 / {counts['촬영 PNG만 있는 이미지 파일명'][0]:,}장",
        '읽을 수 없는 JSON (1-2)': f'{len(jsons) - len(readable)}개',
        '조합 폴더 = `<파일명의 조합>_json` (1-2)': share_text(sum(j['combo_folder'] == f"{combo_name(j['file_name'])}_json" for j in jsons), len(jsons)),
        '알약 폴더 = drug_N (1-2)': share_text(sum(j['data'] is not None and j['pill_folder'] == j['data']['images'][0]['drug_N'] for j in jsons), len(jsons))
            + (f' (`-오류` 폴더 {n_error}개)' if n_error else ''),
        'JSON 파일 이름 = `<이미지 이름>.json` (1-2)':
            share_text(sum(unicodedata.normalize('NFC', j['path'].name) == j['file_name'][:-len('.png')] + '.json' for j in jsons), len(jsons))
            + (f' (copy 파일 {n_copy}개)' if n_copy else ''),
        **{f'{item} (1-3)': value for item, value in format_checks([j['data'] for j in readable]).items() if item != 'JSON 파일 (이미지)'},
        'images.file_name = imgfile = JSON 파일 이름의 이미지':
            share_text(sum(j['data']['images'][0]['file_name'] == j['data']['images'][0].get('imgfile') == j['file_name'] for j in readable), len(readable)),
        '같은 이미지의 JSON끼리 이미지 필드(`IMAGE_FIELDS`)가 같은 이미지':
            share_text(sum(all({k: i.get(k) for k in IMAGE_FIELDS} == {k: imgs[0].get(k) for k in IMAGE_FIELDS} for i in imgs) for imgs in images_of.values()), len(images_of)),
        '파일명 숫자 7자리 = 이미지 필드의 코드 (1-4, 2-6의 대응표)':
            share_text(sum(j['file_name'][:-len('.png')].split('_')[1:] == [export_code_of[f].get(str(j['data']['images'][0][f])) for f in EXPORT_NAME_FIELDS]
                           for j in readable), len(readable)),
        '저장 형식: 4칸 들여쓰기, 파일 끝 줄바꿈 없음':
            share_text(sum(j['path'].read_text(encoding='utf-8') == json.dumps(j['data'], ensure_ascii=False, indent=4) for j in readable), len(readable))
            if on_disk else '- (파일로 쓰지 않음)',
    }


# 2-7에서 만든 JSON을 0번의 aihub_jsons와 같은 구조(JSON 하나가 한 줄)로 펼친다. 아래 두 셀도 이 목록을 쓴다
recheck_jsons = [{'path': Path(path), 'combo_folder': path.split('/')[0], 'pill_folder': path.split('/')[1],
                  'file_name': Path(path).stem.split(' copy')[0] + '.png',  # 0번과 같은 규칙
                  'data': data, 'error': None} for path, data in sorted(export_jsons.items())]
recheck_pngs = sorted(export_names.values())
recheck_columns = {  # 표의 열 → (JSON 목록, PNG 이름 목록, 파일인지)
    'Kaggle train': (read_json_folder(TRAIN_ANN_DIR), train_files, True),
    'AI Hub 원본 (1-1, 1-2)': (aihub_jsons, aihub_png, True),
    'AI Hub 정제 라벨 (2-7)': (recheck_jsons, recheck_pngs, False),
}
format_results = [recheck_format(jsons, pngs, on_disk) for jsons, pngs, on_disk in recheck_columns.values()]
show_table(['항목 (처음 검사한 절)', *recheck_columns], [[item, *[r[item] for r in format_results]] for item in format_results[0]])

#### 2-8-1 결과

- 정제 라벨은 모든 형식 검사에서 대회 train과 같다.
    - JSON 45,819개와 촬영 PNG 11,974장이 짝을 이룬다. JSON만 있거나 PNG만 있는 이미지와 `_index.png`가 없다.
    - 읽을 수 없는 JSON, `-오류` 폴더, copy 파일이 없다.
    - 키 8개가 모두 대회 규칙과 같다. category_id는 116종이다.
    - images.id는 1501~13474, annotations.id는 5692~51510이다. 대회 train(14~1499, 50~5691)과 test(1~1500)의 번호와 겹치지 않는다.
- AI Hub 원본에서 대회와 달랐던 값(id, 클래스 표기, iscrowd 58548과 키 없음)은 모두 바뀌었다.
- 원본 JSON은 저장 형식이 0/45,933이다. 탭으로 들여쓴 파일이라 대회 train(4칸 들여쓰기)과 다르다.
- 같은 이미지의 JSON끼리 이미지 필드가 같고, 파일명 숫자 7자리가 이미지 필드의 코드와 모두 맞는다.
    - "검은색 배경" 1장은 메타데이터와 이름을 함께 연회색(`_0_2_`)으로 고쳤다. (2-6-2)

#### 2-8-2. 1·2절 검사 다시

| 검사 | 처음 한 곳 |
|---|---|
| 라벨 누락, 클래스 불일치 (`label_issues`) | 2-1-1, 2-2-7 |
| 이미지 밖, area 불일치, box가 똑같음 (`table_problems`) | 2-2-1, 2-2-8 |
| box 맞음 0.75 미만 (OpenCV) | 2-2-3 |
| 크기·색으로 다시 배정한 클래스가 라벨과 다름 (OpenCV) | 2-3-1 |

- `1·2절 검사 결과`는 그 검사를 처음 한 라벨의 결과다. 칸은 "객체 수 / 이미지 수"다.
- OpenCV 검사는 원래 이름의 PNG로 잰다. 이름을 고친 13장은 2-6-2의 이름 표로 원래 이름을 찾는다.

In [ ]:
recheck_by_image = collections.defaultdict(list)  # 정제 라벨의 이미지 이름 → 그 이름을 가리키는 JSON (위 셀에서 펼친 것)
for j in recheck_jsons:
    recheck_by_image[j['file_name']].append(j)
recheck_source = {new: fn for fn, new in export_renamed.items()}  # 고친 이름 → 원래 이름 (이름을 고친 13장, 2-6-2)
recheck_anns = {new: [j['data']['annotations'][0] for j in js] for new, js in recheck_by_image.items()}  # 정제 라벨의 이름 → annotation

# 2-1-1 표 (2-2-7의 label_issues): 파일명의 K-코드와 JSON의 클래스, 쓸 수 있는 bbox를 맞대어 본다
recheck_missing, recheck_extra, recheck_surplus = label_issues(
    {new: [(json_code(j), bbox_problem(j)) for j in js] for new, js in recheck_by_image.items()}, photos=recheck_pngs)
# 2-2-1 표 (2-2-8의 table_problems): 이미지 밖, area 불일치, 다른 클래스와 box가 똑같음
recheck_table = by_kind(table_problems({new: [(a['category_id'], a['bbox'], a['area']) for a in anns if usable_bbox(a['bbox'])]
                                         for new, anns in recheck_anns.items()}, []))


def fit_of(fn, ann, measures):
    """2-2-3의 box 맞음: 알약 범위에 클래스 여백을 더해 다시 만든 box와 라벨 box의 IoU. 알약 범위를 못 찾으면 0이다."""
    extent, _ = measures[(fn, tuple(ann['bbox']))]
    return box_iou(with_margin(extent, class_refs.get(ann['category_id'], no_ref)['margin']), ann['bbox']) if extent else 0.0


def appearance_order(fn, anns, measures):
    """2-3-1의 크기·색 대조: 크기와 색이 클래스 기준값과 가장 맞는 클래스 순서. 대조할 수 없으면 None이다."""
    codes = [a['category_id'] for a in anns]
    found = [measures[(fn, tuple(a['bbox']))] for a in anns]
    if len(anns) < 2 or any(c not in class_refs for c in codes) or any(m[0] is None for m in found):
        return None
    cost = np.array([[appearance_cost(extent, lab, class_refs[c]) for c in codes] for extent, lab in found])
    return [int(j) for j in scipy.optimize.linear_sum_assignment(cost)[1]]


# 알약 범위와 색은 원래 이름의 PNG로 잰다 (2-2의 캐시를 다시 쓴다)
recheck_source_anns = {recheck_source.get(new, new): anns for new, anns in recheck_anns.items()}  # 원래 이름 → annotation
recheck_measures = measure_all({fn: {'annotations': anns} for fn, anns in recheck_source_anns.items()})
# 2-2-3과 같은 규칙: 0.75 미만이고, 2-2 표의 문제가 아니고, 알약 범위를 믿을 수 있는 클래스의 box
recheck_low_fit = sorted((fn, a['category_id']) for fn, anns in recheck_source_anns.items() for a in anns
                         if fit_of(fn, a, recheck_measures) < FIT_THRESHOLD and a['category_id'] not in unreliable_classes)
recheck_orders = {fn: appearance_order(fn, anns, recheck_measures) for fn, anns in recheck_source_anns.items()}
recheck_mismatched = sorted(fn for fn, order in recheck_orders.items() if order and order != list(range(len(order))))
recheck_flagged = [(fn, recheck_source_anns[fn][i]['category_id']) for fn in recheck_mismatched for i, j in enumerate(recheck_orders[fn]) if i != j]

fix_by_label = {(f['file_name'], f['label_id']): f for f in box_fixes}  # (이미지, 붙인 클래스) → 2-2의 처리
finding_files = {fn for fn, _ in LABEL_FINDINGS}  # 2-3에서 라벨을 고친 이미지


def objects_text(items):
    """(이미지, ...) 목록을 '객체 수 / 이미지 수'로 적는다."""
    return f'{len(items):,}개 / {len({item[0] for item in items}):,}장'


missing_21, extra_21, surplus_21 = states['2-1-1 표']
before_kinds = by_kind(before_22)
rows = []
for cause in ['JSON 없음', 'bbox 빈 값', '숫자가 붙은 bbox', '-오류 폴더', '읽을 수 없는 JSON', 'bbox 너비·높이 0 이하']:
    before = [m for m in missing_21 if m[2] == cause]
    after = [m for m in recheck_missing if m[2] == cause]
    note = ''
    if cause == 'JSON 없음' and before:
        groups = collections.Counter(missing_group(m) for m in before)
        note = ', '.join(f"{group.removeprefix('JSON 없음: ')} {groups[group]}개" for group in MISSING_GROUPS if group in groups)
    if before or after:
        rows.append([f"라벨 누락: {cause.replace('-오류', '`-오류`')} (2-1)", objects_text(before) + (f' ({note})' if note else ''), objects_text(after), ''])
rows.append(['클래스 불일치: 파일명에 없는 K-코드 (2-1)', objects_text(extra_21), objects_text(recheck_extra), ''])
rows.append(['클래스 불일치: 같은 K-코드의 box가 파일명보다 많음 (2-1)',
             *[f"{sum(s[2] for s in surplus):,}개 / {len({s[0] for s in surplus}):,}장" for surplus in (surplus_21, recheck_surplus)], ''])
for kind in ['이미지 밖', 'area 불일치', '다른 클래스와 box가 똑같음']:
    before = [item for k, items in before_kinds.items() if kind in k.split(' + ') for item in items]
    after = [item for k, items in recheck_table.items() if kind in k.split(' + ') for item in items]
    rows.append([f'좌표 오류: {kind} (2-2)', objects_text(before), objects_text(after), ''])
fit_actions = collections.Counter(fix_by_label[key]['action'] if key in fix_by_label else '2-2에서 다루지 않은 box' for key in recheck_low_fit)
in_gallery = sum(fn in gallery_of for fn, _ in recheck_low_fit)  # gallery_of: 2-2 문제 갤러리에 그린 이미지 (2-2-7)
rows.append([f'알약을 감싸지 않는 box: box 맞음 {FIT_THRESHOLD} 미만 (2-2-3, OpenCV)', objects_text(low_fit), objects_text(recheck_low_fit),
             f'2-2 문제 갤러리에 그린 이미지의 box {in_gallery}개. 2-2의 처리: ' + ', '.join(f'{a} {n}개' for a, n in fit_actions.most_common())])
rows.append(['클래스 불일치: 크기·색으로 다시 배정한 클래스가 라벨과 다름 (2-3-1, OpenCV)',
             f"{len(flagged):,}개 / {len(mismatched):,}장", objects_text(recheck_flagged),
             f'2-3의 {len(mismatched)}장 중 {len(set(recheck_mismatched) & set(mismatched))}장이다. 모두 2-3 시트에서 라벨이 맞다고 본 이미지이다 '
             f'(LABEL_FINDINGS에 없음 {len(set(recheck_mismatched) - finding_files)}장). 2-3에서 라벨을 고친 {len(set(mismatched) & finding_files)}장은 더 잡히지 않는다'])
show_table(['검사 (처음 검사한 절)', '1·2절 검사 결과', '정제 라벨 (2-7)', '정제 라벨에서 잡힌 것'], rows)

#### 2-8-2 결과

| 검사 | 1·2절 검사 결과 | 정제 라벨 |
|---|---|---|
| 라벨 누락 | 234개 / 175장 | 0 |
| 같은 K-코드의 box가 남음 | 23개 / 18장 | 0 |
| 이미지 밖 | 60개 | 0 |
| area 불일치 | 460개 | 0 |
| 다른 클래스와 box가 똑같음 | 140개 / 70장 | 0 |
| box 맞음 0.75 미만 | 313개 / 247장 | 35개 / 35장 |
| 크기·색 배정이 라벨과 다름 | 140개 / 70장 | 108개 / 54장 |

- 파일 검사(2-1, 2-2)로 찾던 문제는 정제 라벨에서 모두 0이다.
- OpenCV 검사에 남는 것은 모두 2절에서 사진을 보고 판단한 것이다.
    - box 맞음 0.75 미만 35개는 모두 문제 갤러리에 그린 이미지의 box다. 29개는 갤러리를 보고 고친 box, 6개는 보고도 그대로 둔 box(찾지 못해 그대로 둠 4개, 그대로 둠 2개)다.
    - 크기·색 54장은 모두 2-3의 70장에 있던 이미지이고, 시트에서 라벨이 맞다고 본 이미지다. 2-3에서 라벨을 고친 16장은 더 잡히지 않는다.
- 2절에서 보지 않은 box나 이미지는 하나도 잡히지 않았다.

#### 2-8-3. 남은 문제와 파일명

| 검사 | 보는 것 |
|---|---|
| 남은 문제, 이미지 제외 | 2-4의 남은 문제가 있는 이미지와 이미지 제외 줄이 있는 이미지가 남았는지 (원래 이름으로 찾음) |
| 같은 장면 | 1-6과 같은 축소 그림 비교로, 다른 조합 사진과 RMS 1 미만인 쌍 |
| 같은 알약 구성 | 조합 폴더 하나에 원래 조합이 둘 이상 들어갔는지 |
| 파일명에 없는 클래스 | 2-5 표와 같은 기준. 정제 라벨은 고친 이름으로 센다 |
| 다시 만든 이름 | 2-7의 JSON에 2-6-2의 이름 규칙을 다시 적용해 파일명과 같은지 |

In [ ]:
SAME_SCENE_RMS = 1  # 이 값보다 작으면 같은 장면으로 센다 (1-6: 같은 장면 0.4, 알약 하나만 바꾼 다른 사진 1.2~2.0)


def close_pairs(names, thumbs_of):
    """1-6처럼 사진마다 다른 조합에서 축소 그림이 가장 가까운 사진을 찾는다. RMS 차이가 NEAR_RMS(1-6) 미만인 사진 쌍 → RMS 차이"""
    idx, rms = nearest(thumbs_of, thumbs_of, groups=np.array([combo_name(fn) for fn in names]))
    return {frozenset((names[i], names[idx[i]])): rms[i] for i in range(len(names)) if rms[i] < NEAR_RMS}


def recheck_record(new):
    """2-7에서 만든 JSON으로 2-6의 export_name이 받는 라벨({'image', 'annotations'})을 만든다."""
    js = recheck_by_image[new]
    return {'image': js[0]['data']['images'][0], 'annotations': [j['data']['annotations'][0] for j in js]}


recheck_names = sorted(recheck_by_image)
recheck_sources = {recheck_source.get(new, new) for new in recheck_names}  # 정제 라벨 이미지의 원래 이름
# 남은 문제 (2-4): 2-2에서 사진으로 알약을 찾지 못한 annotation. 그 이미지는 2-5에서 이미지 제외 줄로 뺐다
unfound = [(f['file_name'], f['category_id']) for f in box_fixes if f['action'] == '찾지 못함']
excluded_reasons = collections.Counter({row['file_name']: row['action'] for row in fix_table if row['action'].startswith('이미지 제외')}.values())
# 같은 장면 (1-6): 1-6과 같은 축소 그림을 원래 이름으로 찾는다
pairs_before = close_pairs(aihub_names, aihub_thumbs)
pairs_after = close_pairs(recheck_names, np.stack([stats['thumb'][stat_index[f'aihub/{recheck_source.get(new, new)}']] for new in recheck_names])
                          .reshape(len(recheck_names), -1).astype(np.float32))
pairs_after_source = {frozenset(recheck_source.get(new, new) for new in pair) for pair in pairs_after}  # 원래 이름의 쌍
# 같은 알약 구성 (2-6): 클래스 목록이 같은 다른 조합. 남았다면 조합 폴더 하나에 원래 조합이 둘 이상 섞인다
combos_before = collections.defaultdict(set)  # 정제 라벨의 클래스 목록 → 원래 조합 (2-6 그림의 라벨, 뺀 9장 포함)
for fn, anns in labels_before_set.items():
    combos_before[code_list({a['category_id'] for a in anns})].add(combo_name(fn))
combos_after = collections.defaultdict(set)  # 2-7의 조합 폴더 → 그 폴더 사진의 원래 조합
for new in recheck_names:
    combos_after[combo_name(new)].add(combo_name(recheck_source.get(new, new)))
same_set_before = {codes: combos for codes, combos in combos_before.items() if len(combos) > 1}
same_set_after = {combo: combos for combo, combos in combos_after.items() if len(combos) > 1}
named_wrong = sorted(new for new in recheck_names if any(a['category_id'] not in pill_codes(new) for a in recheck_anns[new]))
renamed_wrong = sorted(new for new in recheck_names if export_name(new, recheck_record(new)) != new)
clean_named_wrong = sorted(fn for fn, r in aihub_clean_records.items() if any(a['category_id'] not in pill_codes(fn) for a in r['annotations']))

show_table(['검사 (처음 검사한 절)', '1·2절 검사 결과', '정제 라벨 (2-7)', '비고'], [
    ['남은 문제: 2-2에서 사진으로 알약을 찾지 못한 annotation (2-4 정리 표)', objects_text(unfound),
     objects_text([u for u in unfound if u[0] in recheck_sources]), '정제 라벨 이미지의 원래 이름(2-6-2 이름 표)으로 찾았다'],
    ['이미지 제외 줄이 있는 이미지 (2-4)', f'{len(excluded_files)}장', f'{len(excluded_files & recheck_sources)}장',
     ', '.join(f"{reason.removeprefix('이미지 제외 (').removesuffix(')')} {n}장" for reason, n in excluded_reasons.most_common())],
    [f'다른 조합 이름의 같은 장면: 축소 그림 RMS {SAME_SCENE_RMS} 미만인 쌍 (1-6)', f'{sum(v < SAME_SCENE_RMS for v in pairs_before.values())}쌍',
     f'{sum(v < SAME_SCENE_RMS for v in pairs_after.values())}쌍',
     f'가장 가까운 쌍의 RMS {min(pairs_after.values()):.2f}. RMS {SAME_SCENE_RMS}~{NEAR_RMS}인 {len(pairs_after)}쌍 중 {len(pairs_after_source & set(pairs_before))}쌍이 '
     '1-6에도 있던 쌍이다 (알약 하나만 바꿔 찍은 다른 사진이라 1-6에서 그대로 쓰기로 함)'],
    ['같은 알약 구성의 다른 조합: 클래스 목록이 같은 조합 (2-6)', f'{len(same_set_before)}쌍',
     f'{len(same_set_after)}쌍', f'조합 폴더 {len(combos_after):,}개가 모두 원래 조합 하나의 사진이다' if not same_set_after else ''],
    ['파일명에 없는 클래스가 있는 이미지 (2-5)', f'{len(clean_named_wrong)}장', f'{len(named_wrong)}장', '2-5는 원래 이름, 2-7은 고친 이름으로 센다'],
    ['정제 라벨로 다시 만든 이름이 파일명과 다른 이미지 (2-6 이름 표)', f'{len(export_renamed)}장', f'{len(renamed_wrong)}장',
     '2-7에서 만든 JSON(클래스, 이미지 필드)으로 2-6의 `export_name`을 다시 적용했다'],
])

#### 2-8-3 결과

| 검사 | 1·2절 검사 결과 | 정제 라벨 |
|---|---|---|
| 남은 문제 | 21개 / 20장 | 0 |
| 이미지 제외 줄이 있는 이미지 | 35장 | 0 |
| 같은 장면 (RMS 1 미만) | 6쌍 | 0 |
| 같은 알약 구성 | 3쌍 | 0 |
| 파일명에 없는 클래스가 있는 이미지 | 12장 | 0 |
| 다시 만든 이름이 파일명과 다름 | 13장 | 0 |

- 정제 라벨 이미지끼리 가장 가까운 쌍도 RMS 1.25로, 같은 장면(0.4)보다 크다. RMS 1~2인 20쌍은 모두 1-6에도 있던, 알약 하나만 바꾼 다른 사진이다.
- 조합 폴더 3,997개가 모두 원래 조합 하나의 사진이다.
- 정제 라벨에서는 파일명의 K-코드가 클래스와 같다. 조합 단위로 train과 val을 나누면 같은 장면이나 같은 알약 구성이 양쪽에 갈라지지 않는다.

### 2-9. 정제 목록 대조

- 이 노트북의 정제 목록(`aihub_fix_rows`)을 레포의 `data/label_fix/label_fixes.csv` aihub 줄과 (이미지, K-코드)마다 비교한다.
    - 노트북 목록은 같은 annotation을 두 단계에서 고치면 두 줄이고, CSV는 마지막 값 한 줄이다. 그래서 마지막 줄끼리 비교한다.
    - 이미지 제외는 노트북에는 뺀 이유마다 줄이 있고 CSV에는 이미지마다 `exclude` 줄 하나다. 뺀 이미지 집합으로 비교하고, 그 이미지의 다른 줄은 양쪽 모두 뺀다.
    - 메타데이터 고침은 CSV의 `image_fix` 열이다.
- 다르면 어느 줄이 다른지 출력하고 멈춘다. 같으면 CSV가 이 노트북의 결론 그대로라는 뜻이다.

In [ ]:
csv_fixes = read_label_fixes(LABEL_FIXES_CSV, source='aihub')

# 이미지 제외: 노트북은 뺀 이유마다 줄이 있고 CSV는 이미지마다 exclude 줄 하나다. 이미지 집합으로 비교한다
nb_excluded = {row['file_name'] for row in aihub_fix_rows if row['action'].startswith('이미지 제외')}
csv_excluded = {f['file_name'] for f in csv_fixes if f['action'] == 'exclude'}
assert nb_excluded == csv_excluded, f'뺀 이미지가 다르다. 노트북에만 {sorted(nb_excluded - csv_excluded)[:3]}, CSV에만 {sorted(csv_excluded - nb_excluded)[:3]}'

# 나머지 줄: (이미지, 1-3의 K-코드)마다 노트북 목록의 마지막 값을 CSV 형식으로 바꿔 비교한다. 뺀 이미지의 줄은 양쪽 모두 뺀다
nb_by_key = {}
for row in aihub_fix_rows:
    if row['file_name'] in nb_excluded:
        continue
    key = (row['file_name'], int(row['category_id']))
    item = nb_by_key.setdefault(key, {'bbox_after': None, 'area_after': None, 'category_id_after': None, 'image_fix': None})
    if row['action'].startswith('메타데이터 고침'):
        field, value = row['action'].split(': ', 1)[1].split('=', 1)
        item['image_fix'] = {**(item['image_fix'] or {}), field: value}
        continue
    item.update(bbox_after=json.loads(row['bbox_after']), area_after=int(row['area_after']))
    if row['category_id_after']:
        item['category_id_after'] = int(row['category_id_after'])
for (fn, code), item in nb_by_key.items():
    original = next((a for a in converted.get(fn, []) if a['category_id'] == code), None)  # 1-3 라벨의 그 알약 annotation
    item['action'] = 'modify' if original else 'add'  # add: 1-3 라벨에 annotation이 없던 줄
    if item['bbox_after'] is None:  # 메타데이터만 고친 줄: CSV의 modify 줄은 원본 bbox와 너비×높이를 그대로 적는다
        item['bbox_after'], item['area_after'] = list(original['bbox']), original['bbox'][2] * original['bbox'][3]
csv_by_key = {(f['file_name'], f['category_id']): f for f in csv_fixes if f['action'] != 'exclude'}

only_nb, only_csv = sorted(set(nb_by_key) - set(csv_by_key)), sorted(set(csv_by_key) - set(nb_by_key))
FIELDS = ['action', 'bbox_after', 'area_after', 'category_id_after', 'image_fix']
different = [(key, field, nb_by_key[key][field], csv_by_key[key][field]) for key in sorted(set(nb_by_key) & set(csv_by_key)) for field in FIELDS
             if (nb_by_key[key][field] or None) != (csv_by_key[key][field] or None)]
assert not (only_nb or only_csv or different), (f'노트북에만 {len(only_nb)}개 {only_nb[:3]}, CSV에만 {len(only_csv)}개 {only_csv[:3]}, '
                                               f'값이 다른 줄 {len(different)}개 {different[:3]}')
print(f'노트북 정제 목록 {len(aihub_fix_rows):,}줄 → 뺀 이미지 {len(nb_excluded)}장, 고친 annotation {len(nb_by_key):,}개. '
      f'{LABEL_FIXES_CSV.relative_to(PROJECT_ROOT)}의 aihub 줄 {len(csv_fixes):,}개(exclude {len(csv_excluded)}장 + 나머지 {len(csv_by_key):,}개)와 모두 같다.')
